In [ ]:

import pandas as pd
import glob
import os

# Find all uploaded Excel files
files = glob.glob("/content/*.xlsx") + glob.glob("/content/*.xls")

print("Files found:")
for file in files:
    print(os.path.basename(file))

# Read and merge all Excel files
all_data = []

for file in files:
    df = pd.read_excel(file)
    all_data.append(df)

# Combine all rows into one sheet
merged_df = pd.concat(all_data, ignore_index=True)

# Save the merged Excel file
output_file = "/content/Merged.xlsx"
merged_df.to_excel(output_file, index=False)

print("\nMerge completed successfully!")
print("Total rows:", len(merged_df))
print("Total columns:", len(merged_df.columns))
print("Output file:", output_file)

# Download the merged file
from google.colab import files as colab_files
colab_files.download(output_file)


Files found:
MedicalEstablishmentData_Lucknow_2026-27 (5).xlsx
MedicalEstablishmentData_Lucknow_2026-27 (6).xlsx
MedicalEstablishmentData_Lucknow_2026-27 (4).xlsx
MedicalEstablishmentData_Lucknow_2026-27 (2).xlsx
MedicalEstablishmentData_Lucknow_2026-27.xlsx
MedicalEstablishmentData_Lucknow_2026-27 (3).xlsx
MedicalEstablishmentData_Lucknow_2026-27 (1).xlsx
MedicalEstablishmentData_Lucknow_2025-26.xlsx

Merge completed successfully!
Total rows: 3339
Total columns: 9
Output file: /content/Merged.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
from google.colab import files

# Load the merged Excel file
df = pd.read_excel("/content/Merged.xlsx")

# Filter rows where Establishment type is Hospital
filtered_df = df[
    df["Establishment Type"].astype(str).str.strip().str.lower() == "hospital"
]

# Save the filtered data
output_file = "/content/Hospitals_Only.xlsx"
filtered_df.to_excel(output_file, index=False)

# Display results
print("Original rows:", len(df))
print("Hospital rows:", len(filtered_df))

# Download the filtered file
files.download(output_file)

Original rows: 3339
Hospital rows: 1277


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install pandas openpyxl requests

In [ ]:

import pandas as pd
import requests
import time
from getpass import getpass
from google.colab import files

# Enter your Google Maps Geocoding API key
API_KEY = getpass("Enter your Google Maps API key: ")

# Load hospital data
df = pd.read_excel("/content/Hospitals_Only.xlsx")

# Inspect column names
print(df.columns.tolist())


Enter your Google Maps API key: ··········
['S.No.', 'District', 'Registration No.', 'Establishment Name', 'Applicant Name', 'No. of Beds', 'Establishment Type', 'Actions', 'Application Number']


In [ ]:

import pandas as pd
import requests
import time
from google.colab import files

# Load hospital data
df = pd.read_excel("/content/Hospitals_Only.xlsx")

# Google Maps Geocoding API
url = "https://maps.googleapis.com/maps/api/geocode/json"

def get_coordinates(row):
    name = str(row["Establishment Name"]).strip()
    district = str(row["District"]).strip()

    if not name or name.lower() == "nan":
        return pd.Series([None, None, None, "Missing hospital name"])

    # Search using hospital name and district
    query = f"{name}, {district}, Uttar Pradesh, India"

    params = {
        "address": query,
        "key": API_KEY,
        "region": "in"
    }

    try:
        response = requests.get(url, params=params, timeout=20)
        response.raise_for_status()
        result = response.json()

        if result["status"] == "OK" and result["results"]:
            match = result["results"][0]
            location = match["geometry"]["location"]

            return pd.Series([
                location["lat"],
                location["lng"],
                match.get("formatted_address", ""),
                "Found - verify"
            ])

        return pd.Series([
            None,
            None,
            "",
            result.get("status", "NOT_FOUND")
        ])

    except Exception as e:
        return pd.Series([None, None, "", f"Error: {str(e)[:100]}"])


# Get coordinates for each hospital
results = []

for i, row in df.iterrows():
    results.append(get_coordinates(row))

    if (i + 1) % 25 == 0:
        print(f"Processed {i + 1}/{len(df)} hospitals")

    time.sleep(0.1)

# Append new columns
geo_df = pd.DataFrame(
    results,
    columns=[
        "Latitude",
        "Longitude",
        "Geocoded Address",
        "Geocoding Status"
    ],
    index=df.index
)

df = pd.concat([df, geo_df], axis=1)

# Save output
output_path = "/content/Hospitals_With_Coordinates.xlsx"
df.to_excel(output_path, index=False)

# Summary
print("\nProcessing completed!")
print("Total hospitals:", len(df))
print("Coordinates found:", df["Latitude"].notna().sum())
print("Coordinates missing:", df["Latitude"].isna().sum())

# Download the Excel file
files.download(output_path)


Processed 25/1277 hospitals
Processed 50/1277 hospitals
Processed 75/1277 hospitals
Processed 100/1277 hospitals
Processed 125/1277 hospitals
Processed 150/1277 hospitals
Processed 175/1277 hospitals
Processed 200/1277 hospitals
Processed 225/1277 hospitals
Processed 250/1277 hospitals
Processed 275/1277 hospitals
Processed 300/1277 hospitals
Processed 325/1277 hospitals
Processed 350/1277 hospitals
Processed 375/1277 hospitals
Processed 400/1277 hospitals
Processed 425/1277 hospitals
Processed 450/1277 hospitals
Processed 475/1277 hospitals
Processed 500/1277 hospitals
Processed 525/1277 hospitals
Processed 550/1277 hospitals
Processed 575/1277 hospitals
Processed 600/1277 hospitals
Processed 625/1277 hospitals
Processed 650/1277 hospitals
Processed 675/1277 hospitals
Processed 700/1277 hospitals
Processed 725/1277 hospitals
Processed 750/1277 hospitals
Processed 775/1277 hospitals
Processed 800/1277 hospitals
Processed 825/1277 hospitals
Processed 850/1277 hospitals
Processed 875/127

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import re
import unicodedata
from rapidfuzz import fuzz, process
from google.colab import files

# Find the updated CSV already uploaded to Colab
csv_candidates = glob.glob(
    "/content/*lucknow_public_hospitals_beds_updated*.csv"
)

if not csv_candidates:
    print("Please upload your updated public-hospital CSV.")
    uploaded = files.upload()
    csv_candidates = [
        f"/content/{name}" for name in uploaded
        if name.lower().endswith(".csv")
    ]

if not csv_candidates:
    raise FileNotFoundError("No public-hospital CSV was found.")

# Select the most recently modified matching CSV
csv_path = max(csv_candidates, key=os.path.getmtime)

# Find the Excel workbook, if already uploaded
xlsx_candidates = glob.glob(
    "/content/*Hospitals_With_Coordinates (2)*.xlsx"
)

if not xlsx_candidates:
    print("Please upload Hospitals_With_Coordinates.xlsx")
    uploaded = files.upload()
    xlsx_candidates = [
        f"/content/{name}" for name in uploaded
        if name.lower().endswith((".xlsx", ".xls"))
    ]

if not xlsx_candidates:
    raise FileNotFoundError("No hospital Excel workbook was found.")

xlsx_path = max(xlsx_candidates, key=os.path.getmtime)

# Read the files
public_df = pd.read_csv(csv_path)
hospital_df = pd.read_excel(xlsx_path)

print("Public CSV:", csv_path, public_df.shape)
print("Hospital workbook:", xlsx_path, hospital_df.shape)

print("\nCSV columns:")
print(public_df.columns.tolist())

print("\nExcel columns:")
print(hospital_df.columns.tolist())

Public CSV: /content/lucknow_public_hospitals_beds_updated (1).csv (19, 13)
Hospital workbook: /content/Hospitals_With_Coordinates (2).xlsx (1277, 13)

CSV columns:
['S.No.', 'District', 'Registration No.', 'Establishment Name', 'Applicant Name', 'No. of Beds', 'Establishment Type', 'Actions', 'Application Number', 'Latitude', 'Longitude', 'Geocoded Address', 'Geocoding Status']

Excel columns:
['S.No.', 'District', 'Registration No.', 'Establishment Name', 'Applicant Name', 'No. of Beds', 'Establishment Type', 'Actions', 'Application Number', 'Latitude', 'Longitude', 'Geocoded Address', 'Geocoding Status']


In [ ]:
def standardize_columns(df):
    df = df.copy()

    new_columns = []
    for col in df.columns:
        col = unicodedata.normalize("NFKD", str(col))
        col = col.strip().lower()
        col = re.sub(r"[^a-z0-9]+", "_", col)
        col = col.strip("_")
        new_columns.append(col)

    # Ensure standardized column names are unique
    counts = {}
    unique_columns = []

    for col in new_columns:
        counts[col] = counts.get(col, 0) + 1
        unique_columns.append(
            col if counts[col] == 1
            else f"{col}_{counts[col]}"
        )

    df.columns = unique_columns
    return df


public_df = standardize_columns(public_df)
hospital_df = standardize_columns(hospital_df)

print("CSV:", public_df.columns.tolist())
print("Excel:", hospital_df.columns.tolist())

CSV: ['s_no', 'district', 'registration_no', 'establishment_name', 'applicant_name', 'no_of_beds', 'establishment_type', 'actions', 'application_number', 'latitude', 'longitude', 'geocoded_address', 'geocoding_status']
Excel: ['s_no', 'district', 'registration_no', 'establishment_name', 'applicant_name', 'no_of_beds', 'establishment_type', 'actions', 'application_number', 'latitude', 'longitude', 'geocoded_address', 'geocoding_status']


In [ ]:
COLUMN_ALIASES = {
    "facility_name": [
        "establishment_name", "hospital_name", "facility_name",
        "name_of_hospital", "name", "hospital", "institution_name"
    ],
    "facility_type": [
        "establishment_type", "facility_type", "hospital_type",
        "type_of_hospital", "type"
    ],
    "beds": [
        "no_of_beds", "number_of_beds", "beds", "bed_count",
        "total_beds", "no_of_bed", "bed_strength"
    ],
    "latitude": [
        "latitude", "lat", "hospital_latitude", "location_latitude"
    ],
    "longitude": [
        "longitude", "lon", "lng", "long",
        "hospital_longitude", "location_longitude"
    ],
    "address": [
        "address", "hospital_address", "full_address",
        "location", "locality", "village", "town"
    ],
    "registration_no": [
        "registration_no", "registration_number",
        "reg_no", "registration_id", "license_no"
    ],
    "district": [
        "district", "district_name"
    ],
    "coordinate_status": [
        "geocoding_status", "coordinate_status",
        "location_status", "geocode_status"
    ]
}


def find_column(df, field):
    # Exact alias match first
    for alias in COLUMN_ALIASES[field]:
        if alias in df.columns:
            return alias

    # Conservative partial match for common column variations
    for col in df.columns:
        for alias in COLUMN_ALIASES[field]:
            if alias in col or col in alias:
                if len(col) >= 4:
                    return col

    return None


def prepare_source(df, source_name):
    df = df.copy()

    detected = {
        field: find_column(df, field)
        for field in COLUMN_ALIASES
    }

    if not detected["facility_name"]:
        raise ValueError(
            f"Could not identify facility name in {source_name}. "
            f"Available columns: {df.columns.tolist()}"
        )

    if not detected["beds"]:
        raise ValueError(
            f"Could not identify bed-count column in {source_name}. "
            "Inspect the column names and add the correct alias."
        )

    if not detected["latitude"] or not detected["longitude"]:
        raise ValueError(
            f"Could not identify both coordinates in {source_name}. "
            f"Available columns: {df.columns.tolist()}"
        )

    out = pd.DataFrame(index=df.index)

    for field, source_col in detected.items():
        if source_col is not None:
            out[field] = df[source_col]
        else:
            out[field] = pd.NA

    out["source"] = source_name

    # Preserve the source registration number as text
    out["registration_no"] = (
        out["registration_no"].astype("string").str.strip()
        .replace("", pd.NA)
    )

    # Clean facility names
    out["facility_name"] = (
        out["facility_name"].astype("string").str.strip()
    )

    # Convert numeric fields safely
    for col in ["beds", "latitude", "longitude"]:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    # Infer ownership only when supported by explicit source information.
    # Do not classify all records in the public list as public without
    # checking that the CSV truly contains public facilities.
    out["ownership"] = "unknown"

    name = source_name.lower()
    if "public" in name:
        out["ownership"] = "public"

    # Keep only the required columns
    keep = [
        "facility_name", "facility_type", "ownership", "beds",
        "latitude", "longitude", "address", "registration_no",
        "district", "source", "coordinate_status"
    ]

    return out[keep], detected


public_clean, public_mapping = prepare_source(
    public_df, "public_hospitals_csv"
)

workbook_clean, workbook_mapping = prepare_source(
    hospital_df, "hospital_workbook"
)

print("CSV field mapping:", public_mapping)
print("Workbook field mapping:", workbook_mapping)

print("\nPrepared shapes:")
print(public_clean.shape, workbook_clean.shape)

CSV field mapping: {'facility_name': 'establishment_name', 'facility_type': 'establishment_type', 'beds': 'no_of_beds', 'latitude': 'latitude', 'longitude': 'longitude', 'address': 'geocoded_address', 'registration_no': 'registration_no', 'district': 'district', 'coordinate_status': 'geocoding_status'}
Workbook field mapping: {'facility_name': 'establishment_name', 'facility_type': 'establishment_type', 'beds': 'no_of_beds', 'latitude': 'latitude', 'longitude': 'longitude', 'address': 'geocoded_address', 'registration_no': 'registration_no', 'district': 'district', 'coordinate_status': 'geocoding_status'}

Prepared shapes:
(19, 11) (1277, 11)


In [ ]:
def normalize_name(value):
    if pd.isna(value):
        return ""

    value = unicodedata.normalize("NFKD", str(value))
    value = value.lower().strip()
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def clean_source(df):
    df = df.copy()

    df["facility_name"] = df["facility_name"].astype("string").str.strip()
    df["facility_type"] = df["facility_type"].astype("string").str.strip()
    df["address"] = df["address"].astype("string").str.strip()

    # Treat blank strings as missing
    for col in ["facility_type", "address", "registration_no",
                "district", "coordinate_status"]:
        df[col] = df[col].replace(r"^\s*$", pd.NA, regex=True)

    # Remove rows without a facility name
    df = df[
        df["facility_name"].notna()
        & (df["facility_name"].str.strip() != "")
    ].copy()

    # Coordinate validity: global range checks only.
    # A separate geographic boundary test will be needed for Lucknow.
    df["coordinate_valid"] = (
        df["latitude"].between(-90, 90)
        & df["longitude"].between(-180, 180)
    )

    # A valid-looking pair must contain both coordinates.
    df.loc[~df["coordinate_valid"], ["latitude", "longitude"]] = np.nan

    # Bed counts must be non-negative integers.
    invalid_beds = (
        df["beds"].notna()
        & (
            (df["beds"] < 0)
            | (df["beds"] % 1 != 0)
        )
    )

    df.loc[invalid_beds, "beds"] = np.nan

    # A zero bed count is retained for investigation, not automatically
    # interpreted as a missing value.
    df["bed_status"] = np.select(
        [
            df["beds"].isna(),
            df["beds"].eq(0)
        ],
        [
            "unknown_or_invalid",
            "zero_requires_verification"
        ],
        default="reported"
    )

    df["normalized_name"] = df["facility_name"].map(normalize_name)

    return df


public_clean = clean_source(public_clean)
workbook_clean = clean_source(workbook_clean)

public_before = len(public_clean)
workbook_before = len(workbook_clean)

# Remove exact duplicates within each source
public_clean = public_clean.drop_duplicates().copy()
workbook_clean = workbook_clean.drop_duplicates().copy()

print("Public source rows after exact deduplication:", len(public_clean))
print("Public exact duplicates removed:", public_before - len(public_clean))

print("Workbook rows after exact deduplication:", len(workbook_clean))
print("Workbook exact duplicates removed:", workbook_before - len(workbook_clean))

print("\nBed status — public CSV:")
print(public_clean["bed_status"].value_counts(dropna=False))

print("\nBed status — workbook:")
print(workbook_clean["bed_status"].value_counts(dropna=False))

Public source rows after exact deduplication: 19
Public exact duplicates removed: 0
Workbook rows after exact deduplication: 267
Workbook exact duplicates removed: 1010

Bed status — public CSV:
bed_status
reported    19
Name: count, dtype: int64

Bed status — workbook:
bed_status
reported                      259
zero_requires_verification      8
Name: count, dtype: int64


In [ ]:
from rapidfuzz import fuzz, process
import pandas as pd
import re

# Generic words that contribute little to hospital identity.
# We retain distinctive names, acronyms, and meaningful terms.
GENERIC_WORDS = {
    "hospital", "hospitals", "medical", "institute", "institutes",
    "institution", "institutions", "health", "care", "centre",
    "center", "clinic", "clinics", "nursing", "home", "homes",
    "limited", "ltd", "private", "pvt", "trust", "the"
}

def distinctive_name(name):
    name = normalize_name(name)
    tokens = [
        token for token in name.split()
        if token not in GENERIC_WORDS
    ]
    return " ".join(tokens)


# Work with distinct names, not every row
public_names = (
    public_clean[["facility_name", "normalized_name"]]
    .drop_duplicates(subset=["normalized_name"])
    .copy()
)

workbook_names = (
    workbook_clean[["facility_name", "normalized_name"]]
    .drop_duplicates(subset=["normalized_name"])
    .copy()
)

public_names["match_name"] = public_names["facility_name"].map(
    distinctive_name
)
workbook_names["match_name"] = workbook_names["facility_name"].map(
    distinctive_name
)

# Exclude empty names after generic-word removal
workbook_lookup = workbook_names[
    workbook_names["match_name"].str.len() >= 4
].copy()

candidate_rows = []

for _, pub in public_names.iterrows():
    query = pub["match_name"]

    if len(query) < 4:
        continue

    candidates = process.extract(
        query,
        workbook_lookup["match_name"].tolist(),
        scorer=fuzz.ratio,
        limit=5
    )

    for candidate, score, idx in candidates:
        # A high score only identifies a candidate for human review.
        if score >= 75:
            wb = workbook_lookup.iloc[idx]

            candidate_rows.append({
                "public_name": pub["facility_name"],
                "workbook_name": wb["facility_name"],
                "public_match_name": query,
                "workbook_match_name": candidate,
                "similarity_score": round(score, 2),
                "decision": "REVIEW - NOT AUTO-MERGED"
            })

improved_match_review = pd.DataFrame(candidate_rows)

if not improved_match_review.empty:
    improved_match_review = (
        improved_match_review
        .sort_values(
            ["similarity_score", "public_name"],
            ascending=[False, True]
        )
        .reset_index(drop=True)
    )

print("Candidate pairs for manual review:", len(improved_match_review))

display(improved_match_review.head(50))

Candidate pairs for manual review: 0


""


In [ ]:
master = pd.concat(
    [public_clean, workbook_clean],
    ignore_index=True
)

# Create a deterministic record key from normalized name, coordinates,
# and registration number. This is a record identifier, not proof that
# each key represents a unique physical facility.
master["record_key"] = (
    master["normalized_name"].fillna("")
    + "|"
    + master["registration_no"].fillna("").str.lower()
    + "|"
    + master["latitude"].round(5).astype("string").fillna("")
    + "|"
    + master["longitude"].round(5).astype("string").fillna("")
)

# Do not merge records with different registration numbers merely
# because their names are similar. Exact duplicate source records have
# already been removed.
master = master.drop_duplicates(
    subset=[
        "normalized_name", "registration_no",
        "latitude", "longitude", "beds", "facility_type"
    ]
).copy()

# Flag coordinates for verification
master["coordinate_status"] = (
    master["coordinate_status"].astype("string").fillna("unverified")
)

master.loc[
    ~master["coordinate_valid"],
    "coordinate_status"
] = "invalid_or_missing"

master.loc[
    master["coordinate_valid"]
    & master["coordinate_status"].str.contains(
        "verify|approx|unverified", case=False, na=False
    ),
    "coordinate_status"
] = "needs_verification"

master.loc[
    master["coordinate_valid"]
    & master["coordinate_status"].eq("unverified"),
    "coordinate_status"
] = "needs_verification"

# Keep only the required master columns
master_columns = [
    "record_key",
    "facility_name",
    "facility_type",
    "ownership",
    "beds",
    "latitude",
    "longitude",
    "address",
    "registration_no",
    "district",
    "source",
    "coordinate_status",
    "bed_status"
]

master = master[master_columns].reset_index(drop=True)

print("Master dataset shape:", master.shape)
print("Unique record keys:", master["record_key"].nunique())

display(master.head(10))

Master dataset shape: (286, 13)
Unique record keys: 284


,record_key,facility_name,facility_type,ownership,beds,latitude,longitude,address,registration_no,district,source,coordinate_status,bed_status
0,dr shyama prasad mukherjee civil hospital||26....,Dr. Shyama Prasad Mukherjee (Civil) Hospital,Government Hospital,public,401.0,26.844668,80.948981,"Hazratganj, Lucknow, Uttar Pradesh, India",<NA>,Lucknow,public_hospitals_csv,Coordinates found in public geospatial directo...,reported
1,king george s medical university kgmu hospital...,King George’s Medical University (KGMU) Hospital,Government Teaching Hospital,public,3500.0,26.861000,80.929000,"Chowk, Lucknow, Uttar Pradesh, India",<NA>,Lucknow,public_hospitals_csv,needs_verification,reported
2,dr ram manohar lohia institute of medical scie...,Dr. Ram Manohar Lohia Institute of Medical Sci...,Government Teaching Hospital,public,1285.0,26.870000,81.005000,"Gomti Nagar, Lucknow, Uttar Pradesh, India",<NA>,Lucknow,public_hospitals_csv,needs_verification,reported
3,sanjay gandhi postgraduate institute of medica...,Sanjay Gandhi Postgraduate Institute of Medica...,Government Teaching Hospital,public,2272.0,26.746420,80.936060,"Raebareli Road, Lucknow, Uttar Pradesh, India",<NA>,Lucknow,public_hospitals_csv,needs_verification,reported
4,balrampur hospital||26.85729|80.92594,Balrampur Hospital,Government Hospital,public,776.0,26.857292,80.925939,"Kaiserbagh, Lucknow, Uttar Pradesh, India",<NA>,Lucknow,public_hospitals_csv,Coordinates found in public geospatial directo...,reported
5,rani laxmi bai combined hospital||26.83134|80....,Rani Laxmi Bai Combined Hospital,Government Hospital,public,100.0,26.831341,80.891741,"Rajajipuram, Lucknow, Uttar Pradesh, India",<NA>,Lucknow,public_hospitals_csv,Coordinates found in public geospatial directo...,reported
6,lok bandhu shri raj narayan combined hospital|...,Lok Bandhu Shri Raj Narayan Combined Hospital,Government Hospital,public,318.0,26.793499,80.901125,"Kanpur Road, Lucknow, Uttar Pradesh, India",<NA>,Lucknow,public_hospitals_csv,Coordinates found in public geospatial directo...,reported
7,avanti bai mahila chikitsalaya dufferin hospit...,Avanti Bai Mahila Chikitsalaya (Dufferin Hospi...,Government Women Hospital,public,326.0,26.851000,80.936000,"Lucknow, Uttar Pradesh, India",<NA>,Lucknow,public_hospitals_csv,needs_verification,reported
8,chc malihabad||26.9232|80.71005,CHC Malihabad,Community Health Centre,public,30.0,26.923200,80.710050,"Malihabad, Lucknow, Uttar Pradesh, India",<NA>,Lucknow,public_hospitals_csv,needs_verification,reported
9,chc chinhat||26.88146|81.00265,CHC Chinhat,Community Health Centre,public,50.0,26.881460,81.002650,"Chinhat, Lucknow, Uttar Pradesh, India",<NA>,Lucknow,public_hospitals_csv,needs_verification,reported


In [ ]:
audit_report = pd.DataFrame([
    {
        "metric": "Public CSV rows originally loaded",
        "value": len(public_df)
    },
    {
        "metric": "Public CSV rows after exact deduplication",
        "value": len(public_clean)
    },
    {
        "metric": "Workbook rows originally loaded",
        "value": len(hospital_df)
    },
    {
        "metric": "Workbook rows after exact deduplication",
        "value": len(workbook_clean)
    },
    {
        "metric": "Master records",
        "value": len(master)
    },
    {
        "metric": "Master records missing bed counts",
        "value": master["beds"].isna().sum()
    },
    {
        "metric": "Master records with zero beds",
        "value": master["beds"].eq(0).sum()
    },
    {
        "metric": "Master records missing coordinates",
        "value": (
            master["latitude"].isna() | master["longitude"].isna()
        ).sum()
    },
    {
        "metric": "Master records needing coordinate verification",
        "value": master["coordinate_status"].eq(
            "needs_verification"
        ).sum()
    },
    {
        "metric": "Exact normalized names shared across sources",
        "value": len(exact_name_matches)
    },
    {
        "metric": "Potential fuzzy matches requiring review",
        "value": len(match_review)
    }
])

display(audit_report)

,metric,value
0,Public CSV rows originally loaded,19
1,Public CSV rows after exact deduplication,19
2,Workbook rows originally loaded,1277
3,Workbook rows after exact deduplication,267
4,Master records,286
5,Master records missing bed counts,0
6,Master records with zero beds,8
7,Master records missing coordinates,0
8,Master records needing coordinate verification,282
9,Exact normalized names shared across sources,0


In [ ]:
output_csv = "/content/lucknow_hospital_master.csv"
output_excel = "/content/lucknow_hospital_audit.xlsx"

master.to_csv(output_csv, index=False)

with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:
    master.to_excel(writer, sheet_name="Master", index=False)
    audit_report.to_excel(writer, sheet_name="Audit_Report", index=False)

    if match_review.empty:
        pd.DataFrame(
            columns=[
                "public_name", "workbook_candidate",
                "similarity_score", "public_beds",
                "public_latitude", "public_longitude"
            ]
        ).to_excel(writer, sheet_name="Match_Review", index=False)
    else:
        match_review.to_excel(
            writer, sheet_name="Match_Review", index=False
        )

print("Saved:", output_csv)
print("Saved:", output_excel)

files.download(output_csv)
files.download(output_excel)

Saved: /content/lucknow_hospital_master.csv
Saved: /content/lucknow_hospital_audit.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install openpyxl xlrd pdfplumber pandas

from pathlib import Path

RAW_DIR = Path("/content/drive/MyDrive/Lucknow_Census_Raw")
OUT_DIR = Path("/content/drive/MyDrive/lucknow_census_processed")

OUT_DIR.mkdir(parents=True, exist_ok=True)

required_files = [
    "DH_2011_DCHB_Village_Release_0900.xlsx",
    "DH_2011_DCHB_Town_Release_0900.xlsx",
    "DH_2011_0926-Lucknow.zip",
    "DH_2011_0926_PART_A_DCHB_LUCKNOW.pdf",
    "DH_2011_0926_PART_B_DCHB_LUCKNOW.pdf",
]

for filename in required_files:
    path = RAW_DIR / filename
    print(
        f"{'FOUND' if path.exists() else 'MISSING'}: "
        f"{filename}"
    )

print("\nOutput folder:", OUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FOUND: DH_2011_DCHB_Village_Release_0900.xlsx
FOUND: DH_2011_DCHB_Town_Release_0900.xlsx
FOUND: DH_2011_0926-Lucknow.zip
FOUND: DH_2011_0926_PART_A_DCHB_LUCKNOW.pdf
FOUND: DH_2011_0926_PART_B_DCHB_LUCKNOW.pdf

Output folder: /content/drive/MyDrive/lucknow_census_processed


In [ ]:
import pandas as pd
import json

village_file = RAW_DIR / required_files[0]
town_file = RAW_DIR / required_files[1]

inventory = []

for filepath in [village_file, town_file]:
    excel = pd.ExcelFile(filepath, engine="openpyxl")

    for sheet in excel.sheet_names:
        # Read only the header row, not the entire dataset.
        header = pd.read_excel(
            filepath,
            sheet_name=sheet,
            header=0,
            nrows=0,
            engine="openpyxl"
        )

        inventory.append({
            "file": filepath.name,
            "sheet": sheet,
            "number_of_columns": len(header.columns),
            "columns": json.dumps(
                [str(c) for c in header.columns],
                ensure_ascii=False
            )
        })

inventory_df = pd.DataFrame(inventory)

inventory_df.to_csv(
    OUT_DIR / "raw_inventory.csv",
    index=False
)

for _, row in inventory_df.iterrows():
    print(f"\nFILE: {row['file']}")
    print(f"SHEET: {row['sheet']}")
    print(f"COLUMNS: {row['number_of_columns']}")

    columns = json.loads(row["columns"])
    for i, column in enumerate(columns):
        print(f"{i}: {column}")

print("\nSaved raw_inventory.csv")


FILE: DH_2011_DCHB_Village_Release_0900.xlsx
SHEET: Village_Data_0900
COLUMNS: 396
0: State Code
1: State Name
2: District Code
3: District Name
4: Sub District Code
5: Sub District Name
6: Village Code
7: Village Name
8: CD Block Code
9: CD Block Name
10: Gram Panchayat Code
11: Gram Panchayat Name
12: Reference Year
13: Sub District Head Quarter (Name)
14: Sub District Head Quarter (Distance in km)
15: District Head Quarter  (Name)
16: District Head Quarter  (Distance in km)
17: Nearest Statutory Town  (Name)
18: Nearest Statutory Town (Distance in km)
19: Within the State/UT (Name)
20: Within the State/UT (Distance in km)
21: Outside the State/UT, if nearest one is not within the State/UT (Name)
22: Outside the State/UT distance, if nearest one is not within the State/UT (Distance in km)
23: Total Geographical Area (in Hectares)
24: Total   Households 
25: Total Population of Village
26: Total Male Population of Village
27: Total Female Population of Village
28: Total Scheduled Cas

In [ ]:
import csv
from openpyxl import load_workbook

village_output = OUT_DIR / "village_data_lucknow.csv"
village_selected_output = (
    OUT_DIR / "village_data_lucknow_selected.csv"
)

selected_keywords = [
    "state code",
    "state name",
    "district code",
    "district name",
    "sub district code",
    "sub district name",
    "village code",
    "village name",
    "cd block code",
    "cd block name",
    "gram panchayat code",
    "gram panchayat name",
    "reference year",
    "total geographical area",
    "total households",
    "total population",
    "male population",
    "female population",
    "scheduled castes population",
    "scheduled tribes population",
    "community health centre",
    "primary health centre",
    "primary heallth sub centre",
    "primary health sub centre",
    "hospital allopathic",
    "hospital alternative medicine",
    "dispensary",
    "maternity and child welfare",
    "family welfare centre",
    "mobile health clinic",
    "medicine shop",
    "medical practitioner",
    "public bus service",
    "private bus service",
    "railway station",
    "major district road",
    "other district road",
    "black topped",
    "all weather road",
    "drinking water",
    "toilet",
    "electricity",
    "nearest statutory town",
]

def cell_value(cell, header):
    """Preserve numeric IDs as strings where possible."""
    value = cell.value

    if value is None:
        return ""

    if "code" in str(header).lower() and isinstance(value, (int, float)):
        if isinstance(value, float) and value.is_integer():
            value = int(value)

        fmt = cell.number_format or ""
        if isinstance(value, int) and fmt and set(fmt) == {"0"}:
            return f"{value:0{len(fmt)}d}"

        return str(value)

    return value

wb = load_workbook(
    village_file,
    read_only=True,
    data_only=True
)

ws = wb["Village_Data_0900"]

header_cells = next(
    ws.iter_rows(min_row=1, max_row=1)
)
headers = [str(cell.value or f"column_{i}")
           for i, cell in enumerate(header_cells)]

selected_indices = [
    i for i, name in enumerate(headers)
    if any(key in name.lower() for key in selected_keywords)
]

selected_headers = [headers[i] for i in selected_indices]

matches = 0
identifier_mismatches = 0

with open(village_output, "w", newline="", encoding="utf-8-sig") as f_all, \
     open(village_selected_output, "w", newline="", encoding="utf-8-sig") as f_sel:

    writer_all = csv.writer(f_all)
    writer_sel = csv.writer(f_sel)

    writer_all.writerow(headers)
    writer_sel.writerow(selected_headers)

    for row in ws.iter_rows(min_row=2):
        district_code = str(row[2].value or "").strip().lstrip("0")
        district_name = str(row[3].value or "").strip().casefold()

        code_match = district_code == "157"
        name_match = district_name == "lucknow"

        # Count possible code/name disagreements for the audit.
        if code_match != name_match:
            identifier_mismatches += 1

        # Require both identifiers to match.
        if not (code_match and name_match):
            continue

        values = [
            cell_value(cell, headers[i])
            for i, cell in enumerate(row)
        ]

        writer_all.writerow(values)
        writer_sel.writerow([values[i] for i in selected_indices])

        matches += 1

wb.close()

report = [
    "Lucknow Village Directory extraction",
    f"Source: {village_file.name}",
    "Filter: District Code = 157 AND District Name = Lucknow",
    f"Matching rows: {matches}",
    f"District code/name disagreement rows: {identifier_mismatches}",
    f"Full output: {village_output}",
    f"Selected output: {village_selected_output}",
]

with open(
    OUT_DIR / "village_extraction_report.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write("\n".join(report))

print("\n".join(report))

Lucknow Village Directory extraction
Source: DH_2011_DCHB_Village_Release_0900.xlsx
Filter: District Code = 157 AND District Name = Lucknow
Matching rows: 807
District code/name disagreement rows: 0
Full output: /content/drive/MyDrive/lucknow_census_processed/village_data_lucknow.csv
Selected output: /content/drive/MyDrive/lucknow_census_processed/village_data_lucknow_selected.csv


In [ ]:
from openpyxl import load_workbook
import csv

def export_filtered_sheet(
    excel_path,
    sheet_name,
    output_path,
    district_code=157,
    district_name="Lucknow"
):
    wb = load_workbook(
        excel_path,
        read_only=True,
        data_only=True
    )

    ws = wb[sheet_name]
    rows = ws.iter_rows()

    header_cells = next(rows)
    headers = [
        str(cell.value or f"column_{i}")
        for i, cell in enumerate(header_cells)
    ]

    # Find the identifier columns by their names.
    lower_headers = [h.lower().strip() for h in headers]

    code_idx = next(
        i for i, h in enumerate(lower_headers)
        if h == "district code"
    )
    name_idx = next(
        i for i, h in enumerate(lower_headers)
        if h == "district name"
    )

    count = 0
    mismatches = 0

    with open(
        output_path, "w", newline="", encoding="utf-8-sig"
    ) as f:
        writer = csv.writer(f)
        writer.writerow(headers)

        for row in rows:
            code = str(row[code_idx].value or "").strip().lstrip("0")
            name = str(row[name_idx].value or "").strip().casefold()

            code_match = code == str(district_code)
            name_match = name == district_name.casefold()

            if code_match != name_match:
                mismatches += 1

            if not (code_match and name_match):
                continue

            writer.writerow([
                cell_value(cell, headers[i])
                for i, cell in enumerate(row)
            ])
            count += 1

    wb.close()

    print(f"{sheet_name}: {count} matching records")
    print(f"Identifier disagreements: {mismatches}")
    print(f"Saved: {output_path}")

    return count


export_filtered_sheet(
    town_file,
    "Town_0900",
    OUT_DIR / "town_data_lucknow.csv"
)

export_filtered_sheet(
    town_file,
    "Slum_0900",
    OUT_DIR / "slum_data_lucknow.csv"
)

Town_0900: 12 matching records
Identifier disagreements: 0
Saved: /content/drive/MyDrive/lucknow_census_processed/town_data_lucknow.csv
Slum_0900: 3 matching records
Identifier disagreements: 0
Saved: /content/drive/MyDrive/lucknow_census_processed/slum_data_lucknow.csv


3

In [ ]:
wb = load_workbook(
    village_file,
    read_only=True,
    data_only=True
)

ws = wb["Hamlet_Data_0900"]

with open(
    OUT_DIR / "hamlet_data_raw.csv",
    "w",
    newline="",
    encoding="utf-8-sig"
) as f:
    writer = csv.writer(f)

    for row in ws.iter_rows():
        writer.writerow([cell.value for cell in row])

wb.close()

print("Saved:", OUT_DIR / "hamlet_data_raw.csv")

Saved: /content/drive/MyDrive/lucknow_census_processed/hamlet_data_raw.csv


In [ ]:
from zipfile import ZipFile
from pathlib import Path
import pandas as pd

zip_file = RAW_DIR / "DH_2011_0926-Lucknow.zip"

extract_dir = OUT_DIR / "lucknow_zip_extracted"
csv_dir = OUT_DIR / "census_appendices_raw_csv"

extract_dir.mkdir(parents=True, exist_ok=True)
csv_dir.mkdir(parents=True, exist_ok=True)

with ZipFile(zip_file, "r") as z:
    z.extractall(extract_dir)

workbooks = sorted(extract_dir.rglob("*.xls"))

print("Excel workbooks extracted:", len(workbooks))

archive_inventory = []

for filepath in workbooks:
    try:
        excel = pd.ExcelFile(filepath, engine="xlrd")

        for sheet in excel.sheet_names:
            # Read as raw cells. Do not assume a header row.
            df = pd.read_excel(
                filepath,
                sheet_name=sheet,
                header=None,
                dtype=str,
                engine="xlrd"
            )

            safe_name = (
                filepath.stem + "__" +
                sheet.replace("/", "_").replace("\\", "_")
                + ".csv"
            )

            df.to_csv(
                csv_dir / safe_name,
                index=False,
                header=False,
                encoding="utf-8-sig"
            )

            archive_inventory.append({
                "workbook": filepath.name,
                "sheet": sheet,
                "rows": len(df),
                "columns": len(df.columns),
                "raw_csv": safe_name,
                "status": "OK"
            })

    except Exception as e:
        archive_inventory.append({
            "workbook": filepath.name,
            "sheet": "",
            "rows": None,
            "columns": None,
            "raw_csv": "",
            "status": f"ERROR: {str(e)}"
        })

inventory_zip_df = pd.DataFrame(archive_inventory)

inventory_zip_df.to_csv(
    OUT_DIR / "zip_workbook_inventory.csv",
    index=False
)

print("\nWorkbook inventory:")
print(inventory_zip_df.to_string(index=False))

print("\nSaved ZIP inventory and raw worksheet CSVs.")

Excel workbooks extracted: 62
*** formula/tFunc unknown FuncID:186
*** formula/tFunc unknown FuncID:186
*** formula/tFunc unknown FuncID:186
*** formula/tFunc unknown FuncID:186
*** formula/tFunc unknown FuncID:186
*** formula/tFunc unknown FuncID:186
*** formula/tFunc unknown FuncID:186
*** formula/tFunc unknown FuncID:186
*** formula/tFunc unknown FuncID:186
*** formula/tFunc unknown FuncID:186
*** formula/tFunc unknown FuncID:186
*** formula/tFunc unknown FuncID:186

Workbook inventory:
                   workbook   sheet  rows  columns                              raw_csv  status
       Appendix_IA_0926.xls  Sheet1  15.0      8.0         Appendix_IA_0926__Sheet1.csv      OK
       Appendix_IB_0926.xls  Sheet1  15.0      7.0         Appendix_IB_0926__Sheet1.csv      OK
       Appendix_IC_0926.xls  Sheet1  15.0      7.0         Appendix_IC_0926__Sheet1.csv      OK
       Appendix_II_0926.xls  Sheet1  49.0     12.0         Appendix_II_0926__Sheet1.csv      OK
        Appendix_I_0926.x

In [ ]:
import pdfplumber

pdf_files = [
    RAW_DIR / "DH_2011_0926_PART_A_DCHB_LUCKNOW.pdf",
    RAW_DIR / "DH_2011_0926_PART_B_DCHB_LUCKNOW.pdf",
]

for pdf_path in pdf_files:
    output_txt = OUT_DIR / (pdf_path.stem + "_extracted.txt")

    with pdfplumber.open(pdf_path) as pdf:
        with open(output_txt, "w", encoding="utf-8") as f:
            for page_no, page in enumerate(pdf.pages, start=1):
                f.write(f"\n\n===== PAGE {page_no} =====\n\n")
                f.write(page.extract_text() or "[No text extracted]")

    print("Extracted:", output_txt)

Extracted: /content/drive/MyDrive/lucknow_census_processed/DH_2011_0926_PART_A_DCHB_LUCKNOW_extracted.txt
Extracted: /content/drive/MyDrive/lucknow_census_processed/DH_2011_0926_PART_B_DCHB_LUCKNOW_extracted.txt


In [ ]:
import pandas as pd

def validate_csv(path, key_columns=None, chunksize=20000):
    path = Path(path)

    if not path.exists():
        print(f"MISSING: {path.name}")
        return

    total_rows = 0
    seen_keys = set()
    duplicate_rows = 0
    district_codes = set()
    district_names = set()
    sample_missing = None

    for chunk in pd.read_csv(
        path,
        dtype=str,
        chunksize=chunksize,
        encoding="utf-8-sig"
    ):
        chunk.columns = [str(c).strip() for c in chunk.columns]
        total_rows += len(chunk)

        if "District Code" in chunk:
            district_codes.update(
                chunk["District Code"].dropna().unique()
            )

        if "District Name" in chunk:
            district_names.update(
                chunk["District Name"].dropna().unique()
            )

        if key_columns and all(c in chunk.columns for c in key_columns):
            keys = chunk[key_columns].fillna("").astype(str).agg(
                "|".join, axis=1
            )

            for key in keys:
                if key in seen_keys:
                    duplicate_rows += 1
                else:
                    seen_keys.add(key)

        if sample_missing is None:
            sample_missing = chunk.isna().mean()

    print(f"\nFILE: {path.name}")
    print("Rows:", total_rows)
    print("District codes:", sorted(district_codes))
    print("District names:", sorted(district_names))
    print("Duplicate key occurrences:", duplicate_rows)

    if sample_missing is not None:
        print("Missingness in first chunk:")
        print(sample_missing.sort_values(ascending=False).head(10))


validate_csv(
    OUT_DIR / "village_data_lucknow.csv",
    ["Village Code"]
)

validate_csv(
    OUT_DIR / "town_data_lucknow.csv",
    ["Town Code"]
)

validate_csv(
    OUT_DIR / "slum_data_lucknow.csv",
    ["Town Code", "Slum Name"]
)


FILE: village_data_lucknow.csv
Rows: 807
District codes: ['157']
District names: ['Lucknow']
Duplicate key occurrences: 0
Missingness in first chunk:
Outside the State/UT distance, if nearest one is not within the State/UT (Distance in km)    0.995043
Spring Functioning All round the year (Status A(1)/NA(2))                                    0.992565
Spring Functioning in Summer months (April-September) (Status A(1)/NA(2))                    0.992565
Others Functioning in Summer months (April-September) (Status A(1)/NA(2))                    0.986369
Others Functioning All round the year (Status A(1)/NA(2))                                    0.983891
Outside the State/UT, if nearest one is not within the State/UT (Name)                       0.981413
Handicrafts  Commodities (Third)                                                             0.980173
Handicrafts Commodities (Second)                                                             0.978934
Nearest Facility Status (Govt(1)/

In [ ]:
import pandas as pd
import re

selected_csv = OUT_DIR / "village_data_lucknow_selected.csv"

df = pd.read_csv(
    selected_csv,
    dtype=str,
    encoding="utf-8-sig"
)

# Remove accidental whitespace from column names only.
df.columns = [str(c).strip() for c in df.columns]

# Identify population and household columns from the actual headers.
population_col = next(
    (c for c in df.columns if c.lower() == "total population of village"),
    None
)

households_col = next(
    (c for c in df.columns if c.lower() == "total   households"),
    None
)

block_col = next(
    (c for c in df.columns if c.lower() == "cd block name"),
    None
)

if population_col is None or block_col is None:
    raise ValueError(
        "Could not locate the expected population/CD block columns. "
        "Inspect village_data_lucknow_selected.csv before proceeding."
    )

df[population_col] = pd.to_numeric(
    df[population_col], errors="coerce"
)

if households_col is not None:
    df[households_col] = pd.to_numeric(
        df[households_col], errors="coerce"
    )

aggregation = {
    "village_records": (population_col, "size"),
    "population_2011": (population_col, "sum"),
}

if households_col is not None:
    aggregation["households_2011"] = (households_col, "sum")

block_summary = (
    df.groupby(block_col, dropna=False)
      .agg(**aggregation)
      .reset_index()
      .rename(columns={block_col: "CD Block Name"})
)

block_summary.to_csv(
    OUT_DIR / "village_population_by_cd_block.csv",
    index=False
)

# Build a data dictionary from the actual column names.
def classify_column(name):
    s = name.lower()

    if any(x in s for x in ["district", "sub district", "village code",
                             "village name", "cd block", "gram panchayat",
                             "state code", "town code"]):
        return "Administrative identifiers"

    if any(x in s for x in ["population", "household", "sex ratio",
                             "literacy", "worker"]):
        return "Demographics and households"

    if any(x in s for x in ["health", "hospital", "medical", "dispensary",
                             "maternity", "doctor", "para medical",
                             "medicine shop", "family welfare"]):
        return "Healthcare"

    if any(x in s for x in ["road", "bus", "railway", "transport",
                             "taxi", "nearest statutory town"]):
        return "Transport and access"

    if any(x in s for x in ["water", "toilet", "drainage",
                             "electricity", "sanitation"]):
        return "Utilities and sanitation"

    if any(x in s for x in ["area", "forest", "irrigation", "land"]):
        return "Geography and land use"

    if "reference year" in s:
        return "Reference metadata"

    return "Other amenities or metadata"


dictionary = pd.DataFrame({
    "column_name": df.columns,
    "category": [classify_column(c) for c in df.columns],
    "source": ["Village_Data_0900"] * len(df.columns)
})

dictionary.to_csv(
    OUT_DIR / "data_dictionary.csv",
    index=False
)

print("\nCD block population summary:")
print(block_summary.to_string(index=False))

print("\nData dictionary:")
print(dictionary.groupby("category").size().sort_values(
    ascending=False
).to_string())

print("\nSaved:")
print(OUT_DIR / "village_population_by_cd_block.csv")
print(OUT_DIR / "data_dictionary.csv")


CD block population summary:
  CD Block Name  village_records  population_2011
Bakshi-Ka-Talab              161           239938
        Chinhat               57           137251
      Gosaiganj              116           198184
         Kakori               83           152277
            Mal               87           170962
      Malihabad              100           179673
   Mohanlalganj              113           248512
  Sarojaninagar               90           224045

Data dictionary:
category
Healthcare                     38
Administrative identifiers     13
Demographics and households     9
Transport and access            7
Utilities and sanitation        2
Other amenities or metadata     2
Geography and land use          1
Reference metadata              1

Saved:
/content/drive/MyDrive/lucknow_census_processed/village_population_by_cd_block.csv
/content/drive/MyDrive/lucknow_census_processed/data_dictionary.csv


In [ ]:
!apt-get -qq update
!apt-get -qq install -y libreoffice-calc
!pip -q install xlrd openpyxl

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Preconfiguring packages ...
Selecting previously unselected package poppler-data.
(Reading database ... 122807 files and directories currently installed.)
Preparing to unpack .../00-poppler-data_0.4.12-1_all.deb ...
Unpacking poppler-data (0.4.12-1) ...
Selecting previously unselected package libreoffice-style-colibre.
Preparing to unpack .../01-libreoffice-style-colibre_4%3a24.2.7-0ubuntu0.24.04.7_all.deb ...
Unpacking libreoffice-style-colibre (4:24.2.7-0ubuntu0.24.04.7) ...
Selecting previously unselected package libreoffice-uiconfig-common.
Preparing to unpack .../02-libreoffice-uiconfig-common_4%3a24.2.7-0ubuntu0.24.04.7_all.deb ...
Unpacking libreoffice-uiconfig-common (4:24.2.7-0ubuntu0.24.04.7) ...
Selecting previously unselected package libuno-sal3t64

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd
import json

# Existing folders from our previous notebook cells
OUT_DIR = Path("/content/drive/MyDrive/lucknow_census_processed")

EXTRACTED_DIR = (
    OUT_DIR
    / "lucknow_zip_extracted"
    / "0926-Lucknow"
)

CSV_DIR = OUT_DIR / "census_appendices_raw_csv"
CONVERTED_DIR = OUT_DIR / "repaired_excel_files"

CSV_DIR.mkdir(parents=True, exist_ok=True)
CONVERTED_DIR.mkdir(parents=True, exist_ok=True)

tables = ["T42", "T43"]
results = []

for table in tables:
    source_file = EXTRACTED_DIR / f"{table}_0926.xls"

    if not source_file.exists():
        print(f"MISSING: {source_file}")
        results.append({
            "table": table,
            "status": "SOURCE FILE MISSING"
        })
        continue

    print(f"\nProcessing {table}...")

    # Convert the old .xls workbook to .xlsx
    command = [
        "libreoffice",
        "--headless",
        "-env:UserInstallation=file:///tmp/lo_profile_" + table,
        "--convert-to", "xlsx",
        "--outdir", str(CONVERTED_DIR),
        str(source_file)
    ]

    process = subprocess.run(
        command,
        capture_output=True,
        text=True,
        timeout=120
    )

    converted_file = CONVERTED_DIR / f"{table}_0926.xlsx"

    if process.returncode != 0 or not converted_file.exists():
        print("Conversion failed.")
        print("STDOUT:", process.stdout)
        print("STDERR:", process.stderr)

        results.append({
            "table": table,
            "status": "CONVERSION FAILED"
        })
        continue

    # Read every worksheet without assuming the first row is a header.
    try:
        workbook = pd.ExcelFile(
            converted_file,
            engine="openpyxl"
        )

        for sheet_name in workbook.sheet_names:
            df = pd.read_excel(
                workbook,
                sheet_name=sheet_name,
                header=None,
                dtype=object
            )

            # Match the existing raw-export naming convention.
            safe_sheet = (
                "".join(
                    c if c.isalnum() or c in "-_" else "_"
                    for c in sheet_name
                ).strip("_")
                or "Sheet"
            )

            output_file = (
                CSV_DIR
                / f"{table}_0926__{safe_sheet}.csv"
            )

            df.to_csv(
                output_file,
                index=False,
                header=False,
                encoding="utf-8-sig"
            )

            print(
                f"EXPORTED: {output_file.name} | "
                f"{df.shape[0]} rows x {df.shape[1]} columns"
            )

            results.append({
                "table": table,
                "sheet": sheet_name,
                "rows": df.shape[0],
                "columns": df.shape[1],
                "csv_file": output_file.name,
                "status": "OK"
            })

    except Exception as e:
        print(f"Reading/export failed for {table}: {e}")

        results.append({
            "table": table,
            "status": f"READ/EXPORT FAILED: {e}"
        })

# Save a repair log
repair_report = pd.DataFrame(results)
repair_report.to_csv(
    OUT_DIR / "t42_t43_repair_report.csv",
    index=False
)

display(repair_report)


Processing T42...
EXPORTED: T42_0926__Sheet1.csv | 19 rows x 7 columns

Processing T43...
EXPORTED: T43_0926__Sheet1.csv | 16 rows x 3 columns


,table,sheet,rows,columns,csv_file,status
0,T42,Sheet1,19,7,T42_0926__Sheet1.csv,OK
1,T43,Sheet1,16,3,T43_0926__Sheet1.csv,OK


In [ ]:
for table in ["T42", "T43"]:
    files = sorted(CSV_DIR.glob(f"{table}_0926__*.csv"))

    print(f"\n{table}: {len(files)} CSV file(s)")

    for file in files:
        df = pd.read_csv(
            file,
            header=None,
            dtype=str,
            encoding="utf-8-sig"
        )

        print(f"\nFile: {file.name}")
        print(f"Shape: {df.shape}")
        display(df.head(8))


T42: 1 CSV file(s)

File: T42_0926__Sheet1.csv
Shape: (19, 7)


,0,1,2,3,4,5,6
0,"Table 42: Schools/ colleges per 10,000 populat...",NaN,NaN,NaN,NaN,NaN,NaN
1,Sr. No.,Name of the town,Type of educational institution (Approx. numbers),NaN,NaN,NaN,NaN
2,NaN,NaN,Primary,Middle,Secondary / matriculation,Senior secondary,College*
3,1,2,3,4,5,6,7
4,1,800946-Malihabad (NP),9,4,1,1,0
5,2,800947-Mahona (NP),7,4,0,0,0
6,3,800948-Itaunja (NP),11,7,3,3,0
7,4,800949-Bakshi Ka Talab (NP),7,4,1,1,1



T43: 1 CSV file(s)

File: T43_0926__Sheet1.csv
Shape: (16, 3)


,0,1,2
0,Table 43: Number of beds in medical institutio...,NaN,NaN
1,Sr. No.,Name of the town,"Number of beds in medical institutions per 10,..."
2,1,2,3
3,1,800946-Malihabad (NP),18
4,2,800947-Mahona (NP),6
5,3,800948-Itaunja (NP),44
6,4,800949-Bakshi Ka Talab (NP),8
7,5,800950-Kakori (NP),21


In [ ]:
from pathlib import Path
import pandas as pd
import re

OUT_DIR = Path("/content/drive/MyDrive/lucknow_census_processed")

VILLAGE_FILE = OUT_DIR / "village_data_lucknow.csv"
SELECTED_FILE = OUT_DIR / "village_data_lucknow_selected.csv"

# Read headers only
full_columns = pd.read_csv(VILLAGE_FILE, nrows=0, dtype=str).columns
selected_columns = set(
    pd.read_csv(SELECTED_FILE, nrows=0, dtype=str).columns
)

# Categories used to identify potentially useful fields
categories = {
    "administrative_and_population": [
        "state code", "district code", "district name",
        "village code", "village name", "block code",
        "block name", "cd block", "population",
        "household", "area", "latitude", "longitude"
    ],
    "healthcare": [
        "medical", "hospital", "dispensary", "chc", "phc",
        "sub-centre", "sub centre", "health centre",
        "maternity", "child welfare", "nursing",
        "allopathic", "ayurvedic", "unani", "homeopathic",
        "doctor", "medical practitioner", "medicine shop",
        "family welfare", "clinic", "bed"
    ],
    "education": [
        "school", "college", "educational", "literacy"
    ],
    "water_and_sanitation": [
        "drinking water", "water supply", "tap water",
        "well", "hand pump", "tubewell", "tube well",
        "toilet", "latrine", "drainage", "sewerage",
        "sanitation"
    ],
    "electricity_and_communications": [
        "electricity", "power supply", "telephone",
        "mobile", "internet", "post office", "telegraph"
    ],
    "transport_and_connectivity": [
        "road", "bus", "railway", "rail station",
        "navigable", "transport", "distance", "nearest town"
    ],
    "other_amenities": [
        "bank", "atm", "agricultural", "market",
        "fair price", "ration", "recreation", "library",
        "community", "public distribution"
    ]
}

audit_rows = []

for col in full_columns:
    normalized = re.sub(r"\s+", " ", str(col)).strip().lower()

    matched_categories = [
        category
        for category, keywords in categories.items()
        if any(keyword in normalized for keyword in keywords)
    ]

    audit_rows.append({
        "column_name": col,
        "present_in_selected_file": col in selected_columns,
        "candidate_categories": "; ".join(matched_categories),
        "potentially_relevant": bool(matched_categories)
    })

audit = pd.DataFrame(audit_rows)

audit.to_csv(
    OUT_DIR / "village_field_audit.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Full village columns:", len(full_columns))
print("Selected file columns:", len(selected_columns))
print("Potentially relevant fields:", int(audit["potentially_relevant"].sum()))
print(
    "Potentially relevant fields missing from selected file:",
    int((
        audit["potentially_relevant"]
        & ~audit["present_in_selected_file"]
    ).sum())
)

print("\nPotentially relevant fields missing from selected file:")
display(
    audit[
        audit["potentially_relevant"]
        & ~audit["present_in_selected_file"]
    ].reset_index(drop=True)
)

print("\nAudit saved to:", OUT_DIR / "village_field_audit.csv")

Full village columns: 396
Selected file columns: 73
Potentially relevant fields: 295
Potentially relevant fields missing from selected file: 227

Potentially relevant fields missing from selected file:


,column_name,present_in_selected_file,candidate_categories,potentially_relevant
0,Sub District Head Quarter (Distance in km),False,transport_and_connectivity,True
1,District Head Quarter (Distance in km),False,transport_and_connectivity,True
2,Within the State/UT (Distance in km),False,transport_and_connectivity,True
3,"Outside the State/UT distance, if nearest one ...",False,transport_and_connectivity,True
4,Total Households,False,administrative_and_population,True
...,...,...,...,...
222,Tanks/Lakes Area (in Hectares),False,administrative_and_population,True
223,Waterfall Area (in Hectares),False,administrative_and_population,True
224,Other Source (specify) Area (in Hectares),False,administrative_and_population,True
225,Nearest Town Name,False,transport_and_connectivity,True



Audit saved to: /content/drive/MyDrive/lucknow_census_processed/village_field_audit.csv


In [ ]:
from pathlib import Path
import pandas as pd
import re

OUT_DIR = Path("/content/drive/MyDrive/lucknow_census_processed")

INPUT_FILE = OUT_DIR / "village_data_lucknow.csv"
OUTPUT_FILE = OUT_DIR / "village_data_lucknow_selected_v2.csv"
REPORT_FILE = OUT_DIR / "village_field_selection_report.csv"

# Read all fields as strings to preserve codes and source values.
df = pd.read_csv(
    INPUT_FILE,
    dtype=str,
    encoding="utf-8-sig",
    low_memory=False
)

# Normalize whitespace in column names for matching.
def normalize_name(name):
    return re.sub(r"\s+", " ", str(name)).strip().lower()

normalized = {col: normalize_name(col) for col in df.columns}

# Always retain important identifiers and population fields.
required_keywords = [
    "state code",
    "state name",
    "district code",
    "district name",
    "sub district code",
    "sub district name",
    "village code",
    "village name",
    "cd block code",
    "cd block name",
    "total population",
    "total households",
    "area of village",
]

# Relevant fields across the categories needed for our project.
category_keywords = {
    "healthcare": [
        "medical", "hospital", "dispensary", "chc", "phc",
        "sub-centre", "sub centre", "health centre",
        "maternity", "child welfare", "nursing home",
        "allopathic", "ayurvedic", "unani", "homeopathic",
        "medical practitioner", "medicine shop",
        "family welfare", "clinic", "in-patient", "out-patient"
    ],
    "education": [
        "school", "college", "educational institution",
        "vocational training", "iti"
    ],
    "water_and_sanitation": [
        "drinking water", "tap water", "covered well",
        "uncovered well", "hand pump", "tube wells",
        "borehole", "drainage", "sanitation", "latrine",
        "toilet", "waste disposal", "bio-gas"
    ],
    "electricity_and_communications": [
        "power supply", "electricity", "telephone",
        "mobile phone", "mobile coverage", "internet",
        "common service centre", "post office", "telegraph"
    ],
    "transport_and_connectivity": [
        "bus service", "railway station", "railway",
        "all weather road", "approach by pucca road",
        "approach by", "road", "navigable waterway",
        "nearest town name", "nearest town distance",
        "distance from village"
    ],
    "other_amenities": [
        "bank", "atm", "agricultural credit",
        "public distribution", "pds shop", "market",
        "mandi", "community centre", "public library",
        "sports club", "recreation centre",
        "birth and death registration"
    ],
}

# These are explanatory notes, not village-level observations.
exclude_patterns = [
    "if not available within the village",
    "distance range code",
    "viz; a for",
]

rows = []

for col in df.columns:
    name = normalized[col]

    is_required = any(keyword in name for keyword in required_keywords)
    matched_categories = [
        category
        for category, keywords in category_keywords.items()
        if any(keyword in name for keyword in keywords)
    ]

    is_explanatory_note = any(
        pattern in name for pattern in exclude_patterns
    )

    # Keep required identifiers regardless of category.
    keep = is_required or (
        bool(matched_categories) and not is_explanatory_note
    )

    rows.append({
        "column_name": col,
        "normalized_name": name,
        "required_identifier_or_population": is_required,
        "categories": "; ".join(matched_categories),
        "excluded_as_explanatory_note": is_explanatory_note,
        "selected": keep
    })

selection_report = pd.DataFrame(rows)

selected_columns = selection_report.loc[
    selection_report["selected"], "column_name"
].tolist()

selected_df = df[selected_columns].copy()

selected_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

selection_report.to_csv(
    REPORT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("Original rows:", len(df))
print("Original columns:", len(df.columns))
print("Revised selected columns:", len(selected_columns))
print("Output:", OUTPUT_FILE)
print("Selection report:", REPORT_FILE)

print("\nSelected fields by category:")
for category in category_keywords:
    count = selection_report["categories"].apply(
        lambda value: category in value.split("; ")
    ).sum()
    print(f"{category}: {count}")

print("\nSample of revised extract:")
display(selected_df.head(5))

Original rows: 807
Original columns: 396
Revised selected columns: 197
Output: /content/drive/MyDrive/lucknow_census_processed/village_data_lucknow_selected_v2.csv
Selection report: /content/drive/MyDrive/lucknow_census_processed/village_field_selection_report.csv

Selected fields by category:
healthcare: 61
education: 83
water_and_sanitation: 30
electricity_and_communications: 18
transport_and_connectivity: 12
other_amenities: 11

Sample of revised extract:


,State Code,State Name,District Code,District Name,Sub District Code,Sub District Name,Village Code,Village Name,CD Block Code,CD Block Name,...,Handicrafts Commodities (First),Agricultural Commodities (Second),Manufacturers Commodities (Second),Handicrafts Commodities (Second),Agricultural Commodities (Third),Manufacturers Commodities (Third),Handicrafts Commodities (Third),Wells/Tube Wells Area (in Hectares),Nearest Town Name,Nearest Town Distance from Village (in Km.)
0,09,UTTAR PRADESH,157,Lucknow,00819,Malihabad,142980,Vahir,0294,Mal,...,NaN,"WHEAT,PEACE",NaN,NaN,"POTATO,URD",NaN,NaN,115.44,MALIHABAAD,26
1,09,UTTAR PRADESH,157,Lucknow,00819,Malihabad,142981,Ahindar,0294,Mal,...,NaN,GEHUN ALOO MATAR,NaN,NaN,AAM,NaN,NaN,0,NaN,NaN
2,09,UTTAR PRADESH,157,Lucknow,00819,Malihabad,142982,Umraval,0294,Mal,...,NaN,GARI,NaN,NaN,AAM,NaN,NaN,251,NaN,NaN
3,09,UTTAR PRADESH,157,Lucknow,00819,Malihabad,142983,Rahata,0294,Mal,...,NaN,WHEAT,NaN,NaN,AAM,NaN,NaN,176.95,NaN,NaN
4,09,UTTAR PRADESH,157,Lucknow,00819,Malihabad,142984,Siswara,0294,Mal,...,NaN,GEHUN CHANA MATTAR MASOOR,NaN,NaN,URAD AAM,NaN,NaN,233.13,NaN,NaN


In [ ]:
from pathlib import Path
import pandas as pd

OUT_DIR = Path("/content/drive/MyDrive/lucknow_census_processed")

original = pd.read_csv(
    OUT_DIR / "village_data_lucknow.csv",
    dtype=str,
    encoding="utf-8-sig",
    low_memory=False
)

revised = pd.read_csv(
    OUT_DIR / "village_data_lucknow_selected_v2.csv",
    dtype=str,
    encoding="utf-8-sig",
    low_memory=False
)

print("Original shape:", original.shape)
print("Revised shape:", revised.shape)

# Compare row counts and village codes.
print("\nRow counts match:", len(original) == len(revised))

if "Village Code" in original.columns and "Village Code" in revised.columns:
    original_codes = original["Village Code"].fillna("").tolist()
    revised_codes = revised["Village Code"].fillna("").tolist()

    print("Village codes and order match:", original_codes == revised_codes)
    print("Unique original village codes:", original["Village Code"].nunique())
    print("Unique revised village codes:", revised["Village Code"].nunique())

# Check that population values are unchanged.
population_columns = [
    col for col in original.columns
    if "population" in col.lower()
]

print("\nPopulation fields retained:")
for col in population_columns:
    if col in revised.columns:
        same = (
            original[col].fillna("").tolist()
            == revised[col].fillna("").tolist()
        )
        print(f"{col}: retained={same}")
    else:
        print(f"{col}: NOT RETAINED — review selection rules")

Original shape: (807, 396)
Revised shape: (807, 197)

Row counts match: True
Village codes and order match: True
Unique original village codes: 807
Unique revised village codes: 807

Population fields retained:
Total Population of Village: retained=True
Total Male Population of Village: NOT RETAINED — review selection rules
Total Female Population of Village: NOT RETAINED — review selection rules
Total Scheduled Castes Population of Village: NOT RETAINED — review selection rules
Total Scheduled Castes  Male Population of Village: NOT RETAINED — review selection rules
Total Scheduled Castes  Female Population of Village: NOT RETAINED — review selection rules
Total Scheduled Tribes Population of Village: NOT RETAINED — review selection rules
Total Scheduled Tribes  Male Population of Village: NOT RETAINED — review selection rules
Total Scheduled Tribes  Female Population of Village: NOT RETAINED — review selection rules


In [ ]:
from pathlib import Path
import pandas as pd
import re

OUT_DIR = Path("/content/drive/MyDrive/lucknow_census_processed")

FULL_FILE = OUT_DIR / "village_data_lucknow.csv"
V2_FILE = OUT_DIR / "village_data_lucknow_selected_v2.csv"

V3_FILE = OUT_DIR / "village_data_lucknow_selected_v3.csv"
REPORT_FILE = OUT_DIR / "village_v3_validation_report.csv"

# Read all source fields as strings.
full_df = pd.read_csv(
    FULL_FILE, dtype=str, encoding="utf-8-sig", low_memory=False
)

v2_df = pd.read_csv(
    V2_FILE, dtype=str, encoding="utf-8-sig", low_memory=False
)

def normalize_column(name):
    return re.sub(r"\s+", " ", str(name)).strip().lower()

# Preserve all existing v2 fields, in their original order.
selected_columns = list(v2_df.columns)

# Add all population and household fields from the full dataset.
added_columns = []

for col in full_df.columns:
    name = normalize_column(col)

    relevant = (
        "population" in name
        or "household" in name
        or name == "area of village"
    )

    if relevant and col not in selected_columns:
        selected_columns.append(col)
        added_columns.append(col)

# Build v3 directly from the full source to retain source values.
v3_df = full_df[selected_columns].copy()

v3_df.to_csv(
    V3_FILE, index=False, encoding="utf-8-sig"
)

# Validate row counts, village codes and all retained source values.
checks = []

checks.append({
    "check": "Row count matches full source",
    "passed": len(v3_df) == len(full_df),
    "details": f"Full={len(full_df)}, v3={len(v3_df)}"
})

if "Village Code" in full_df.columns and "Village Code" in v3_df.columns:
    codes_match = (
        full_df["Village Code"].fillna("").tolist()
        == v3_df["Village Code"].fillna("").tolist()
    )
    checks.append({
        "check": "Village codes and order match",
        "passed": codes_match,
        "details": f"Unique codes={v3_df['Village Code'].nunique()}"
    })

missing_population_fields = [
    col for col in full_df.columns
    if "population" in normalize_column(col)
    and col not in v3_df.columns
]

checks.append({
    "check": "All population fields retained",
    "passed": len(missing_population_fields) == 0,
    "details": (
        "All retained" if not missing_population_fields
        else "; ".join(missing_population_fields)
    )
})

# Confirm all selected values match the corresponding full-source values.
values_match = all(
    full_df[col].fillna("").tolist() == v3_df[col].fillna("").tolist()
    for col in v3_df.columns
)

checks.append({
    "check": "All selected values match full source",
    "passed": values_match,
    "details": f"Compared {len(v3_df.columns)} columns"
})

report = pd.DataFrame(checks)
report.to_csv(REPORT_FILE, index=False, encoding="utf-8-sig")

print("Original shape:", full_df.shape)
print("V2 shape:", v2_df.shape)
print("V3 shape:", v3_df.shape)
print("\nNewly added fields:")
for col in added_columns:
    print("-", col)

print("\nValidation:")
display(report)

print("\nPopulation-related columns in v3:")
display(pd.DataFrame({
    "column_name": [
        col for col in v3_df.columns
        if "population" in normalize_column(col)
    ]
}))

Original shape: (807, 396)
V2 shape: (807, 197)
V3 shape: (807, 205)

Newly added fields:
- Total Male Population of Village
- Total Female Population of Village
- Total Scheduled Castes Population of Village
- Total Scheduled Castes  Male Population of Village
- Total Scheduled Castes  Female Population of Village
- Total Scheduled Tribes Population of Village
- Total Scheduled Tribes  Male Population of Village
- Total Scheduled Tribes  Female Population of Village

Validation:


,check,passed,details
0,Row count matches full source,True,"Full=807, v3=807"
1,Village codes and order match,True,Unique codes=807
2,All population fields retained,True,All retained
3,All selected values match full source,True,Compared 205 columns



Population-related columns in v3:


,column_name
0,Total Population of Village
1,Total Male Population of Village
2,Total Female Population of Village
3,Total Scheduled Castes Population of Village
4,Total Scheduled Castes Male Population of Vil...
5,Total Scheduled Castes Female Population of V...
6,Total Scheduled Tribes Population of Village
7,Total Scheduled Tribes Male Population of Vil...
8,Total Scheduled Tribes Female Population of V...


In [ ]:
from pathlib import Path
import pandas as pd

OUT_DIR = Path("/content/drive/MyDrive/lucknow_census_processed")

HAMLET_FILE = OUT_DIR / "hamlet_data_raw.csv"
VILLAGE_FILE = OUT_DIR / "village_data_lucknow.csv"

# Load the Lucknow village identifiers.
villages = pd.read_csv(
    VILLAGE_FILE,
    dtype=str,
    encoding="utf-8-sig",
    low_memory=False
)

# Read a sample of the hamlet file without assuming it has a header.
hamlet_sample = pd.read_csv(
    HAMLET_FILE,
    header=None,
    dtype=str,
    encoding="utf-8-sig",
    nrows=20,
    low_memory=False
)

print("Village dataset shape:", villages.shape)
print("Hamlet sample shape:", hamlet_sample.shape)

print("\nVillage columns:")
print(villages.columns.tolist())

print("\nFirst 20 hamlet rows:")
display(hamlet_sample)

# Inspect the first column names and missing values across the sample.
print("\nNon-empty values by column in the hamlet sample:")
for col in hamlet_sample.columns:
    values = hamlet_sample[col].dropna().astype(str).tolist()
    print(f"\nColumn {col}:")
    print(values[:8])

Village dataset shape: (807, 396)
Hamlet sample shape: (20, 7)

Village columns:
['State Code', 'State Name', 'District Code', 'District Name', 'Sub District Code', 'Sub District Name', 'Village Code', 'Village Name', 'CD Block Code', 'CD Block Name', 'Gram Panchayat Code', 'Gram Panchayat Name', 'Reference Year', 'Sub District Head Quarter (Name)', 'Sub District Head Quarter (Distance in km)', 'District Head Quarter  (Name)', 'District Head Quarter  (Distance in km)', 'Nearest Statutory Town  (Name)', 'Nearest Statutory Town (Distance in km)', 'Within the State/UT (Name)', 'Within the State/UT (Distance in km)', 'Outside the State/UT, if nearest one is not within the State/UT (Name)', 'Outside the State/UT distance, if nearest one is not within the State/UT (Distance in km)', 'Total Geographical Area (in Hectares)', 'Total   Households ', 'Total Population of Village', 'Total Male Population of Village', 'Total Female Population of Village', 'Total Scheduled Castes Population of Villa

,0,1,2,3,4,5,6
0,Hamlet / Habitatation Information,NaN,NaN,NaN,NaN,NaN,NaN
1,Village Code,Village Name,Population of Village,Hamlet (Serial Number),Hamlet Name,Distance from the Village (in km),Number of Households in the Village
2,108885,Faizabad,7047,1,JANIPUR MAJRA FAIZABAD,2,0
3,108885,Faizabad,7047,2,MAGANPURA MAJARA,4,0
4,108896,Sherpur Pelon,3542,1,SHREPUR,0,361
5,108897,Habibpur Alis Tapoban,1316,1,HABIBPUR,0,111
6,108897,Habibpur Alis Tapoban,1316,2,MOMHMMAD PUR,2,35
7,108899,Barkalan,814,1,BADKALA,0,110
8,108901,Inderpur Talara,140,1,INDARPUR TALDA,0,28
9,108902,Rampur,216,1,RAMPUR,0,42



Non-empty values by column in the hamlet sample:

Column 0:
['Hamlet / Habitatation Information', 'Village Code', '108885', '108885', '108896', '108897', '108897', '108899']

Column 1:
['Village Name', 'Faizabad', 'Faizabad', 'Sherpur Pelon', 'Habibpur Alis Tapoban', 'Habibpur Alis Tapoban', 'Barkalan', 'Inderpur Talara']

Column 2:
['Population of Village', '7047', '7047', '3542', '1316', '1316', '814', '140']

Column 3:
['Hamlet                   (Serial Number)', '1', '2', '1', '1', '2', '1', '1']

Column 4:
['Hamlet Name', 'JANIPUR MAJRA FAIZABAD', 'MAGANPURA MAJARA', 'SHREPUR', 'HABIBPUR', 'MOMHMMAD PUR', 'BADKALA', 'INDARPUR TALDA']

Column 5:
['Distance from the Village  (in km)', '2', '4', '0', '0', '2', '0', '0']

Column 6:
['Number of  Households in the Village', '0', '0', '361', '111', '35', '110', '28']


In [ ]:
import pandas as pd
from pathlib import Path

OUT_DIR = Path("/content/drive/MyDrive/lucknow_census_processed")

HAMLET_FILE = OUT_DIR / "hamlet_data_raw.csv"

# Read a larger sample to locate repeated header rows.
sample = pd.read_csv(
    HAMLET_FILE,
    header=None,
    dtype=str,
    encoding="utf-8-sig",
    nrows=500,
    low_memory=False
)

print("Rows containing possible header labels:")
for col in sample.columns:
    matches = sample[col].fillna("").str.contains(
        r"village code|hamlet|district code|village name",
        case=False,
        regex=True
    )

    if matches.any():
        print(f"\nColumn {col}:")
        display(sample.loc[matches].head(10))

# Show the exact identifier format in the village master.
villages = pd.read_csv(
    OUT_DIR / "village_data_lucknow.csv",
    dtype=str,
    encoding="utf-8-sig",
    low_memory=False
)

print("\nFirst 15 Lucknow village codes:")
display(villages[["Village Code", "Village Name"]].head(15))

Rows containing possible header labels:

Column 0:


,0,1,2,3,4,5,6
0,Hamlet / Habitatation Information,NaN,NaN,NaN,NaN,NaN,NaN
1,Village Code,Village Name,Population of Village,Hamlet (Serial Number),Hamlet Name,Distance from the Village (in km),Number of Households in the Village



Column 1:


,0,1,2,3,4,5,6
1,Village Code,Village Name,Population of Village,Hamlet (Serial Number),Hamlet Name,Distance from the Village (in km),Number of Households in the Village



Column 3:


,0,1,2,3,4,5,6
1,Village Code,Village Name,Population of Village,Hamlet (Serial Number),Hamlet Name,Distance from the Village (in km),Number of Households in the Village



Column 4:


,0,1,2,3,4,5,6
1,Village Code,Village Name,Population of Village,Hamlet (Serial Number),Hamlet Name,Distance from the Village (in km),Number of Households in the Village



First 15 Lucknow village codes:


,Village Code,Village Name
0,142980,Vahir
1,142981,Ahindar
2,142982,Umraval
3,142983,Rahata
4,142984,Siswara
5,142985,Gorava Baravki
6,142986,Saspan
7,142987,Kamaluddin Nagar
8,142988,Gopal Pur
9,142989,Tilan


In [ ]:
from pathlib import Path
import pandas as pd

OUT_DIR = Path("/content/drive/MyDrive/lucknow_census_processed")

HAMLET_FILE = OUT_DIR / "hamlet_data_raw.csv"
VILLAGE_FILE = OUT_DIR / "village_data_lucknow.csv"

OUTPUT_FILE = OUT_DIR / "hamlet_data_lucknow.csv"
REPORT_FILE = OUT_DIR / "hamlet_filter_report.csv"
UNMATCHED_FILE = OUT_DIR / "hamlet_unmatched_village_codes.csv"

# --------------------------------------------------
# 1. Load the Lucknow village master
# --------------------------------------------------
villages = pd.read_csv(
    VILLAGE_FILE,
    dtype=str,
    encoding="utf-8-sig",
    low_memory=False
)

def normalize_code(series):
    """Normalize numeric-looking codes without losing leading zeros."""
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

lucknow_codes = set(
    normalize_code(villages["Village Code"]).dropna()
)
lucknow_codes.discard("")

# --------------------------------------------------
# 2. Read the hamlet CSV without assuming a header
# --------------------------------------------------
raw = pd.read_csv(
    HAMLET_FILE,
    header=None,
    dtype=str,
    encoding="utf-8-sig",
    keep_default_na=False,
    low_memory=False
)

print("Raw hamlet shape:", raw.shape)

if raw.shape[1] != 7:
    raise ValueError(
        f"Expected 7 hamlet columns, found {raw.shape[1]}. "
        "Inspect the file before proceeding."
    )

# Row 0 is the title; row 1 contains the actual headers.
headers = raw.iloc[1].astype(str).str.strip().tolist()

# Data begins from row 2.
hamlets = raw.iloc[2:].copy()
hamlets.columns = headers
hamlets = hamlets.reset_index(drop=True)

# Remove completely empty rows.
hamlets = hamlets.loc[
    ~hamlets.apply(
        lambda row: all(str(v).strip() == "" for v in row),
        axis=1
    )
].copy()

# --------------------------------------------------
# 3. Normalize codes and filter to Lucknow
# --------------------------------------------------
code_column = "Village Code"

if code_column not in hamlets.columns:
    raise ValueError(
        f"Could not find '{code_column}'. "
        f"Detected headers: {hamlets.columns.tolist()}"
    )

hamlets["_normalized_village_code"] = normalize_code(
    hamlets[code_column]
)

valid_code_mask = (
    hamlets["_normalized_village_code"].notna()
    & hamlets["_normalized_village_code"].ne("")
)

lucknow_mask = (
    valid_code_mask
    & hamlets["_normalized_village_code"].isin(lucknow_codes)
)

lucknow_hamlets = hamlets.loc[lucknow_mask].copy()
outside_lucknow = hamlets.loc[
    valid_code_mask & ~lucknow_mask
].copy()

# Retain the original seven columns; remove helper column.
lucknow_hamlets = lucknow_hamlets[headers].reset_index(drop=True)

# --------------------------------------------------
# 4. Save the cleaned Lucknow hamlet file
# --------------------------------------------------
lucknow_hamlets.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

# --------------------------------------------------
# 5. Identify village codes with and without hamlet rows
# --------------------------------------------------
hamlet_codes = set(
    normalize_code(lucknow_hamlets[code_column]).dropna()
)
hamlet_codes.discard("")

village_codes_without_hamlet_rows = sorted(
    lucknow_codes - hamlet_codes
)

hamlet_codes_not_in_lucknow = sorted(
    set(normalize_code(outside_lucknow[code_column]).dropna())
    - lucknow_codes
)

pd.DataFrame({
    "Village Code": hamlet_codes_not_in_lucknow
}).to_csv(
    UNMATCHED_FILE,
    index=False,
    encoding="utf-8-sig"
)

# --------------------------------------------------
# 6. Save a validation report
# --------------------------------------------------
report = pd.DataFrame([
    {
        "metric": "Raw CSV rows including title/header",
        "value": len(raw)
    },
    {
        "metric": "Rows after removing title/header and blank rows",
        "value": len(hamlets)
    },
    {
        "metric": "Rows retained for Lucknow",
        "value": len(lucknow_hamlets)
    },
    {
        "metric": "Rows excluded as non-Lucknow or invalid-code rows",
        "value": len(hamlets) - len(lucknow_hamlets)
    },
    {
        "metric": "Unique village codes in Lucknow master",
        "value": len(lucknow_codes)
    },
    {
        "metric": "Unique Lucknow village codes represented in hamlet data",
        "value": len(hamlet_codes)
    },
    {
        "metric": "Lucknow village codes with no hamlet rows",
        "value": len(village_codes_without_hamlet_rows)
    },
    {
        "metric": "Unique non-Lucknow codes found in raw hamlet data",
        "value": len(hamlet_codes_not_in_lucknow)
    }
])

report.to_csv(
    REPORT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nCleaned hamlet shape:", lucknow_hamlets.shape)
print("Cleaned file:", OUTPUT_FILE)
print("\nValidation report:")
display(report)

print("\nFirst 10 cleaned Lucknow hamlet rows:")
display(lucknow_hamlets.head(10))

print("\nSample Lucknow villages with no hamlet rows:")
display(
    villages[
        normalize_code(villages["Village Code"]).isin(
            village_codes_without_hamlet_rows
        )
    ][["Village Code", "Village Name"]].head(20)
)

Raw hamlet shape: (165083, 7)

Cleaned hamlet shape: (1607, 7)
Cleaned file: /content/drive/MyDrive/lucknow_census_processed/hamlet_data_lucknow.csv

Validation report:


,metric,value
0,Raw CSV rows including title/header,165083
1,Rows after removing title/header and blank rows,165081
2,Rows retained for Lucknow,1607
3,Rows excluded as non-Lucknow or invalid-code rows,163474
4,Unique village codes in Lucknow master,807
5,Unique Lucknow village codes represented in ha...,600
6,Lucknow village codes with no hamlet rows,207
7,Unique non-Lucknow codes found in raw hamlet data,56303



First 10 cleaned Lucknow hamlet rows:


,Village Code,Village Name,Population of Village,Hamlet (Serial Number),Hamlet Name,Distance from the Village (in km),Number of Households in the Village
0,142980,Vahir,1244,1,BAGWARA,1,105
1,142980,Vahir,1244,2,LACHANA KHERA,1.5,20
2,142980,Vahir,1244,3,GULAL KHERA,2,75
3,142981,Ahindar,3876,1,KARAUNDI,1.5,70
4,142981,Ahindar,3876,2,MAHIMAKHEDA,1,200
5,142981,Ahindar,3876,3,MUNDKHEDA,1.500,25
6,142981,Ahindar,3876,4,SHIVPOOR,1,75
7,142981,Ahindar,3876,5,PARNANKHEDA,1,100
8,142981,Ahindar,3876,6,RAIHAR,1,60
9,142982,Umraval,3380,1,UMRAVAL,0,300



Sample Lucknow villages with no hamlet rows:


,Village Code,Village Name
7,142987,Kamaluddin Nagar
14,142994,Muriyara
18,142998,Visunpur
22,143002,Shahmau
25,143005,Jindana
30,143010,Devari Ganja
31,143011,Akbar Pur
42,143022,Pakara
43,143023,Devari Bharat
46,143026,Ataria


In [ ]:
from pathlib import Path
import pandas as pd

OUT_DIR = Path("/content/drive/MyDrive/lucknow_census_processed")

hamlets = pd.read_csv(
    OUT_DIR / "hamlet_data_lucknow.csv",
    dtype=str,
    encoding="utf-8-sig",
    keep_default_na=False
)

villages = pd.read_csv(
    OUT_DIR / "village_data_lucknow.csv",
    dtype=str,
    encoding="utf-8-sig",
    low_memory=False
)

def normalize_code(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

village_codes = set(normalize_code(villages["Village Code"]).dropna())
hamlet_codes = normalize_code(hamlets["Village Code"])

valid_codes = hamlet_codes.isin(village_codes)

print("Hamlet output rows:", len(hamlets))
print("All output codes belong to Lucknow:", bool(valid_codes.all()))
print("Rows with invalid or unmatched codes:", int((~valid_codes).sum()))
print("Unique village codes represented:", hamlet_codes.nunique())
print("Duplicate hamlet rows:", int(hamlets.duplicated().sum()))

print("\nColumn names:")
print(hamlets.columns.tolist())

print("\nRows per village — highest counts:")
display(
    hamlets.groupby(
        ["Village Code", "Village Name"],
        dropna=False
    ).size()
    .reset_index(name="hamlet_rows")
    .sort_values("hamlet_rows", ascending=False)
    .head(10)
)

Hamlet output rows: 1607
All output codes belong to Lucknow: True
Rows with invalid or unmatched codes: 0
Unique village codes represented: 600
Duplicate hamlet rows: 0

Column names:
['Village Code', 'Village Name', 'Population of Village', 'Hamlet                   (Serial Number)', 'Hamlet Name', 'Distance from the Village  (in km)', 'Number of  Households in the Village']

Rows per village — highest counts:


,Village Code,Village Name,hamlet_rows
353,143447,Ganeshpur Rahmanpur,26
355,143453,Jugor,22
6,142986,Saspan,17
567,143747,Utrawan,14
2,142982,Umraval,14
62,143074,Jindaur,14
561,143737,Dayal Pur,13
541,143714,Hulas Khera,12
11,142992,Thari,11
499,143666,Aneya Khargapur,11


In [ ]:
from pathlib import Path
import pandas as pd

OUT_DIR = Path("/content/drive/MyDrive/lucknow_census_processed")

# 1. Inspect the extracted Town Directory
town_file = OUT_DIR / "town_data_lucknow.csv"

towns = pd.read_csv(
    town_file,
    dtype=str,
    encoding="utf-8-sig",
    low_memory=False
)

print("Town Directory shape:", towns.shape)
print("\nTown columns:")
print(towns.columns.tolist())

# Display key administrative fields where available
key_columns = [
    col for col in towns.columns
    if any(term in col.lower() for term in [
        "town code", "town name", "district code",
        "district name", "population", "urban status",
        "class of town"
    ])
]

print("\nRelevant town fields:")
display(towns[key_columns].head(20))

# 2. Inspect the inventory of Census workbook exports
inventory_file = OUT_DIR / "zip_workbook_inventory.csv"

if inventory_file.exists():
    inventory = pd.read_csv(
        inventory_file,
        dtype=str,
        encoding="utf-8-sig"
    )

    print("\nWorkbook inventory columns:")
    print(inventory.columns.tolist())

    print("\nInventory preview:")
    display(inventory.head(15))
else:
    print("\nWorkbook inventory not found:", inventory_file)

# 3. Find exported CSV files whose names or contents suggest
#    town, ward, PCA, population, or urban tables.
csv_dir = OUT_DIR / "census_appendices_raw_csv"

matches = []

for file in csv_dir.rglob("*.csv"):
    name_match = any(term in file.name.lower() for term in [
        "pca", "ward", "town", "urban", "population"
    ])

    if name_match:
        matches.append({
            "file": str(file.relative_to(OUT_DIR)),
            "size_bytes": file.stat().st_size
        })

matches_df = pd.DataFrame(matches)

print("\nPotential urban/PCA files found:", len(matches_df))
if not matches_df.empty:
    display(matches_df.sort_values("file").reset_index(drop=True))

Town Directory shape: (12, 429)

Town columns:
['State Code', 'State Name', 'District Code', 'District Name', 'Sub District Code', 'Sub District Name', 'Town Code', 'Town Name', 'Total   Households ', 'Total Population of Town', 'Total Male Population of Town', 'Total Female Population of Town', 'Total Scheduled Castes Population of Town', 'Total Scheduled Castes  Male Population of Town', 'Total Scheduled Castes  Female Population of Town', 'Total Scheduled Tribes Population of Town', 'Total Scheduled Tribes  Male Population of Town', 'Total Scheduled Tribes  Female Population of Town', 'Reference Year', 'Class', 'Civic Status of Town', 'Name of Taluk/Tahsil/Police Station/Islands etc.', 'Name of CD Block', 'Area (sq. km.)', 'Number of Households including Houseless Households (2011 Census)', 'Scheduled Castes Population (2011 Census)', 'Scheduled Tribes Population (2011 Census)', 'Town Population (Census 1901)', 'Growth Rate Town (Census 1901)', 'Town Population (Census 1911)', 'Grow

,District Code,District Name,Sub District Code,Sub District Name,Town Code,Town Name,Total Population of Town,Total Male Population of Town,Total Female Population of Town,Total Scheduled Castes Population of Town,...,Nearest Village/Town Name.24,Nearest Village/Town Name.25,Nearest Village/Town Name.26,Nearest Village/Town Name.27,Nearest Village/Town Name.28,Nearest Village/Town Name.29,Nearest Village/Town Name.30,Nearest Village/Town Name.31,Nearest Village/Town Name.32,Nearest Village/Town Name.33
0,157,LUCKNOW,00819,MALIHABAD,800946,Malihabad (NP),17818,9334,8484,1547,...,LUCKNOW,NaN,LUCKNOW,LUCKNOW,LUCKNOW,LUCKNOW,NaN,LUCKNOW,LUCKNOW,LUCKNOW
1,157,LUCKNOW,00820,BAKSHI KA TALAB,800947,Mahona (NP),8557,4510,4047,2360,...,LUCKNOW,NaN,LUCKNOW,LUCKNOW,LUCKNOW,LUCKNOW,LUCKNOW,BKT,BKT,BKT
2,157,LUCKNOW,00820,BAKSHI KA TALAB,800948,Itaunja (NP),7305,3962,3343,796,...,LUCKNOW,NaN,LUCKNOW,LUCKNOW,LUCKNOW,LUCKNOW,LUCKNOW,LUCKNOW,LUCKNOW,LUCKNOW
3,157,LUCKNOW,00820,BAKSHI KA TALAB,800949,Bakshi Ka Talab (NP),49166,25657,23509,14840,...,LUCKNOW,NaN,LUCKNOW,LUCKNOW,LUCKNOW,LUCKNOW,LUCKNOW,NaN,NaN,NaN
4,157,LUCKNOW,00821,LUCKNOW,143558,Kalli Pashchim (CT),12157,6347,5810,6722,...,MOHANLALGANJ,NaN,LUCKNOW,LUCKNOW,LUCKNOW,LUCKNOW,LCUKNOW,LCUKNOW,LUCKNOW,LUCKNOW
5,157,LUCKNOW,00821,LUCKNOW,143559,Sarsawan (CT),10655,5548,5107,2728,...,LUCKNOW,LUCKNOW,LUCKNOW,LUCKNOW,LUCKNOW,LUCKNOW,LUCKNOW,LUCKNOW,LUCKNOW,LUCKNOW
6,157,LUCKNOW,00821,LUCKNOW,800950,Kakori (NP),19403,10157,9246,2564,...,MOHAN ROAD LUCKNOW,NaN,MOHAN ROAD,LUCKNOW,GUDAMBA,LUCKNOW,NaN,NaN,KAISERBAGH LUCKNOW,KAISERBAGH LUCKNOW
7,157,LUCKNOW,00821,LUCKNOW,800951,Lucknow (M Corp.),2817105,1460970,1356135,302938,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,157,LUCKNOW,00821,LUCKNOW,800952,Lucknow (CB),63003,36586,26417,8915,...,LUCKNOW,LUCKNOW,LUCKNOW,LUCKNOW,NaN,NaN,NaN,NaN,NaN,NaN
9,157,LUCKNOW,00822,MOHANLALGANJ,800953,Gosainganj (NP),9649,5026,4623,1837,...,LUCKNOW,LUCKNOW,LUCKNOW,LUCKNOW,LUCKNOW,LUCKNOW,LUCKNOW,LUCKNOW,LUCKNOW,LUCKNOW



Workbook inventory columns:
['workbook', 'sheet', 'rows', 'columns', 'raw_csv', 'status']

Inventory preview:


,workbook,sheet,rows,columns,raw_csv,status
0,Appendix_IA_0926.xls,Sheet1,15.0,8.0,Appendix_IA_0926__Sheet1.csv,OK
1,Appendix_IB_0926.xls,Sheet1,15.0,7.0,Appendix_IB_0926__Sheet1.csv,OK
2,Appendix_IC_0926.xls,Sheet1,15.0,7.0,Appendix_IC_0926__Sheet1.csv,OK
3,Appendix_II_0926.xls,Sheet1,49.0,12.0,Appendix_II_0926__Sheet1.csv,OK
4,Appendix_I_0926.xls,Sheet1,16.0,71.0,Appendix_I_0926__Sheet1.csv,OK
5,Appendix_VD_0926.xls,Sheet1,838.0,124.0,Appendix_VD_0926__Sheet1.csv,OK
6,Appendix_VIIA_0926.xls,Sheet1,873.0,3.0,Appendix_VIIA_0926__Sheet1.csv,OK
7,Appendix_VIIB_0926.xls,Sheet1,873.0,3.0,Appendix_VIIB_0926__Sheet1.csv,OK
8,Appendix_VI_0926.xls,Sheet1,14.0,6.0,Appendix_VI_0926__Sheet1.csv,OK
9,Appendix_V_0926.xls,Sheet1,14.0,6.0,Appendix_V_0926__Sheet1.csv,OK



Potential urban/PCA files found: 7


,file,size_bytes
0,census_appendices_raw_csv/Town Statement-III_0...,2592
1,census_appendices_raw_csv/Town Statement-II_09...,1945
2,census_appendices_raw_csv/Town Statement-IV_09...,1774
3,census_appendices_raw_csv/Town Statement-I_092...,3292
4,census_appendices_raw_csv/Town Statement-VII_0...,885
5,census_appendices_raw_csv/Town Statement-VI_09...,1008
6,census_appendices_raw_csv/Town Statement-V_092...,3520


In [ ]:
from pathlib import Path
import pandas as pd
import re

OUT_DIR = Path("/content/drive/MyDrive/lucknow_census_processed")
EXTRACTED_DIR = OUT_DIR / "lucknow_zip_extracted"

# 1. List all extracted Excel workbooks
workbooks = sorted(EXTRACTED_DIR.rglob("*.xls"))

print("Total extracted XLS workbooks:", len(workbooks))
print("\nWorkbooks relevant to population, wards, or PCA:")

for file in workbooks:
    name = file.name.lower()

    if any(term in name for term in [
        "pca", "ward", "population", "town", "appendix_vii"
    ]) or re.search(r"t\d+_", name):
        print(file.name)

# 2. Search the workbook inventory for likely population tables
inventory_file = OUT_DIR / "zip_workbook_inventory.csv"

if inventory_file.exists():
    inv = pd.read_csv(inventory_file, dtype=str)

    print("\nPotentially relevant inventory entries:")
    mask = inv["workbook"].fillna("").str.contains(
        r"pca|ward|population|town|T0[1-9]|T[1-4][0-9]",
        case=False,
        regex=True
    )

    display(
        inv.loc[mask, ["workbook", "sheet", "rows", "columns", "status"]]
        .reset_index(drop=True)
    )

# 3. Search the extracted Part B text
pdf_text = OUT_DIR / "DH_2011_0926_PART_B_DCHB_LUCKNOW_extracted.txt"

if pdf_text.exists():
    text = pdf_text.read_text(encoding="utf-8", errors="ignore")
    lines = text.splitlines()

    terms = [
        "primary census abstract",
        "ward",
        "urban",
        "town",
        "population",
        "schedule"
    ]

    print("\nRelevant references in Part B text:")
    for term in terms:
        matches = [
            (i + 1, line.strip())
            for i, line in enumerate(lines)
            if term in line.lower()
        ]

        print(f"\n{term!r}: {len(matches)} matching lines")
        for line_number, line in matches[:8]:
            print(f"Line {line_number}: {line[:250]}")
else:
    print("\nPart B extracted text not found at:", pdf_text)

Total extracted XLS workbooks: 62

Workbooks relevant to population, wards, or PCA:
Appendix_VIIA_0926.xls
Appendix_VIIB_0926.xls
T01_0926.xls
T02_0926.xls
T03_0926.xls
T04_0926.xls
T05_0926.xls
T06_0926.xls
T07_0926.xls
T08_0926.xls
T09_0926.xls
T10_0926.xls
T11_0926.xls
T12_0926.xls
T13_0926.xls
T14_0926.xls
T15_0926.xls
T16_0926.xls
T17_0926.xls
T18_0926.xls
T19_0926.xls
T20_0926.xls
T21_0926.xls
T22_0926.xls
T23_0926.xls
T24_0926.xls
T25_0926.xls
T26_0926.xls
T27_0926.xls
T28_0926.xls
T29_0926.xls
T30_0926.xls
T31_0926.xls
T32_0926.xls
T33_0926.xls
T34_0926.xls
T35_0926.xls
T36_0926.xls
T37_0926.xls
T38_0926.xls
T39_0926.xls
T40_0926.xls
T41_0926.xls
T42_0926.xls
T43_0926.xls
T44_0926.xls
T45_0926.xls
Town Statement-III_0926.xls
Town Statement-II_0926.xls
Town Statement-IV_0926.xls
Town Statement-I_0926.xls
Town Statement-VII_0926.xls
Town Statement-VI_0926.xls
Town Statement-V_0926.xls

Potentially relevant inventory entries:


,workbook,sheet,rows,columns,status
0,T01_0926.xls,Sheet1,13.0,13.0,OK
1,T02_0926.xls,Sheet1,16.0,27.0,OK
2,T03_0926.xls,Sheet1,19.0,15.0,OK
3,T04_0926.xls,Sheet1,18.0,7.0,OK
4,T05_0926.xls,Sheet1,9.0,5.0,OK
5,T06_0926.xls,Sheet1,12.0,3.0,OK
6,T07_0926.xls,Sheet1,16.0,5.0,OK
7,T08_0926.xls,Sheet1,17.0,4.0,OK
8,T09_0926.xls,Sheet1,20.0,7.0,OK
9,T10_0926.xls,Sheet1,13.0,6.0,OK



Relevant references in Part B text:

'primary census abstract': 244 matching lines
Line 12: PRIMARY CENSUS ABSTRACT (PCA)
Line 32: PRIMARY CENSUS ABSTRACT (PCA)
Line 414: PRIMARY CENSUS ABSTRACT
Line 531: PRIMARY CENSUS ABSTRACT
Line 682: 9 Section - I Primary Census Abstract (PCA)
Line 683: (i) Brief note on Primary Census Abstract 22
Line 684: (ii) District Primary Census Abstract 27
Line 685: (iii) Appendix to District Primary Census Abstract Total, Scheduled Castes and 35

'ward': 7213 matching lines
Line 732: each Village and Town and ward of the District. The Primary Census Abstract (PCA) part of
Line 835: and for urban areas up to the ward level for each town or city in the shape of
Line 1031: unit i.e. of each village and town and ward of the district. The non census data
Line 1147: brothers. For his meritorious services he was awarded the title of Sadat Khan
Line 1170: vicissitudes for his reign. From the time of Shuja-ud-daula onwards, the Nawabs
Line 1173: contributed signi

In [ ]:
from pathlib import Path
import pandas as pd

OUT_DIR = Path("/content/drive/MyDrive/lucknow_census_processed")
CSV_DIR = OUT_DIR / "census_appendices_raw_csv"

for n in range(1, 11):
    prefix = f"T{n:02d}_0926__"
    files = sorted(CSV_DIR.glob(prefix + "*.csv"))

    if not files:
        print(f"\nT{n:02d}: No exported CSV found")
        continue

    for file in files:
        df = pd.read_csv(
            file,
            header=None,
            dtype=str,
            encoding="utf-8-sig"
        )

        print(f"\n{'=' * 60}")
        print(f"{file.name} | shape={df.shape}")
        display(df.head(12))


T01_0926__Sheet1.csv | shape=(13, 13)


,0,1,2,3,4,5,6,7,8,9,10,11,12
0,Table 1:Decadal change in population of Tahsil...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SI. NO.,Tahsil,Population,NaN,NaN,NaN,NaN,NaN,Percentage decadal variation 2001-2011,NaN,NaN,percentage urban population,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,2001,NaN,NaN,2011,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,Total,Rural,Urban,Total,Rural,Urban,Total,Rural,Urban,2001,2011
6,1,2,3,4,5,6,7,8,9,10,11,12,13
7,1,Malihabad,312888,297080,15808,368453,350635,17818,17.758750735087315,18.02713073919483,12.71508097165992,5.052287080361023,4.8358949445383805
8,2,Bakshi Ka Talab,276134,262918,13216,349654,284626,65028,26.62475464810563,8.256566686191132,392.03995157384986,4.786082119550653,18.597813838823523
9,3,Lucknow,2652116,389747,2262369,3391208,468885,2922323,27.86801180642174,20.304967068380257,29.170926581826397,85.30430041521562,86.17351103205701



T02_0926__Sheet1.csv | shape=(16, 27)


,0,1,2,3,4,5,6,7,8,9,...,17,18,19,20,21,22,23,24,25,26
0,Table 2: Number and percentage of inhabited vi...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Sr.No.,C.D. Block,Total number of inhabited villages,Total rural population,NaN,NaN,Number and percentage of villages,Population less than 200,NaN,Number and percentage of villages,...,NaN,Number and percentage of villages,Population 2000 - 4999,NaN,Number and percentage of villages,Population 5000 - 9999,NaN,Number and percentage of villages,Population 10000 and above,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,Persons,Males,Females,NaN,Males,Females,NaN,...,Females,NaN,Males,Females,NaN,Males,Females,NaN,Males,Females
6,1,2,3,4,5,6,7,8,9,10,...,18,19,20,21,22,23,24,25,26,27
7,1,0294-Mal,87,170962,89553,81409,3 ( 3 %),243,242,6 ( 7 %),...,20630,28 ( 32 %),45952,41693,4 ( 5 %),13095,12023,0 ( 0 %),0,0
8,2,0295-Malihabad,98,179673,94343,85330,2 ( 2 %),169,136,15 ( 15 %),...,24404,22 ( 22 %),34393,31059,5 ( 5 %),16132,14919,1 ( 1 %),6544,5779
9,3,0296-Bakshi-Ka-Talab,160,239938,126133,113805,2 ( 1 %),154,126,14 ( 9 %),...,36443,36 ( 22 %),54813,49262,2 ( 1 %),7643,7042,0 ( 0 %),0,0



T03_0926__Sheet1.csv | shape=(19, 15)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"Table 3: New towns, de- notified, declassified...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,(a),New,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,(i),Statutory town,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,(ii),Census town,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,(b),Denotified,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,(i),Statutory towns of 2001 census denotified and ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NIL,NaN


... (10 outputs left out)


T09_0926__Sheet1.csv | shape=(20, 7)


,0,1,2,3,4,5,6
0,Table 9: Sex ratio of population in the age g...,NaN,NaN,NaN,NaN,NaN,NaN
1,Sr.No.,Name of Sub-district,Total/ Rural/ Urban,Total population in 0-6 age group,NaN,NaN,Sex ratio for 0-6 age group
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,Persons,Males,Females,NaN
4,1,2,3,4,5,6,7
5,1,00819-Malihabad,Total,52917,27413,25504,930
6,NaN,NaN,Rural,50733,26302,24431,929
7,NaN,NaN,Urban,2184,1111,1073,966
8,2,00820-Bakshi Ka Talab,Total,52599,27533,25066,910
9,NaN,NaN,Rural,43673,22924,20749,905



T10_0926__Sheet1.csv | shape=(13, 6)


,0,1,2,3,4,5
0,Table 10: Sex ratio of population in the age g...,NaN,NaN,NaN,NaN,NaN
1,Sr.No.,Name of CD Block,Total population in 0-6 age group,NaN,NaN,Sex ratio for 0-6 age group
2,NaN,NaN,Persons,Males,Females,NaN
3,1,2,3,4,5,6
4,1,0294-Mal,25340,13306,12034,904
5,2,0295-Malihabad,25393,12996,12397,954
6,3,0296-Bakshi-Ka-Talab,37324,19597,17727,905
7,4,0297-Chinhat,19388,10064,9324,926
8,5,0298-Kakori,21067,10993,10074,916
9,6,0299-Sarojaninagar,29461,15347,14114,920


In [ ]:
from pathlib import Path
import xlrd
import traceback

EXTRACT_DIR = OUT_DIR / "lucknow_zip_extracted"

for number in [42, 43]:
    matches = list(EXTRACT_DIR.rglob(f"T{number}_0926.xls"))

    print(f"\n{'=' * 50}")
    print(f"T{number}: found {len(matches)} matching files")

    for path in matches:
        print("File:", path)

        try:
            wb = xlrd.open_workbook(str(path))
            print("Sheets:", wb.sheet_names())

            for sheet in wb.sheets():
                print(
                    f"Sheet: {sheet.name} | "
                    f"rows={sheet.nrows}, columns={sheet.ncols}"
                )
                for row in range(min(8, sheet.nrows)):
                    print(sheet.row_values(row))

        except Exception:
            traceback.print_exc()


T42: found 1 matching files
File: /content/drive/MyDrive/lucknow_census_processed/lucknow_zip_extracted/0926-Lucknow/T42_0926.xls
*** formula/tFunc unknown FuncID:186
*** formula/tFunc unknown FuncID:186
*** formula/tFunc unknown FuncID:186
*** formula/tFunc unknown FuncID:186
*** formula/tFunc unknown FuncID:186
*** formula/tFunc unknown FuncID:186

T43: found 1 matching files
File: /content/drive/MyDrive/lucknow_census_processed/lucknow_zip_extracted/0926-Lucknow/T43_0926.xls


Traceback (most recent call last):
  File "/tmp/ipykernel_859/3179332798.py", line 17, in <cell line: 0>
    wb = xlrd.open_workbook(str(path))
  File "/usr/local/lib/python3.13/dist-packages/xlrd/__init__.py", line 172, in open_workbook
    bk = open_workbook_xls(
        filename=filename,
    ...<8 lines>...
        ignore_workbook_corruption=ignore_workbook_corruption,
    )
  File "/usr/local/lib/python3.13/dist-packages/xlrd/book.py", line 104, in open_workbook_xls
    bk.parse_globals()
    ~~~~~~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/xlrd/book.py", line 1251, in parse_globals
    self.names_epilogue()
    ~~~~~~~~~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/xlrd/book.py", line 1045, in names_epilogue
    evaluate_name_formula(self, nobj, namex, blah=blah)
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/xlrd/formula.py", line 1142, in evaluate_name_formula
    assert len(tgtobj.stack) 

*** formula/tFunc unknown FuncID:186
*** formula/tFunc unknown FuncID:186
*** formula/tFunc unknown FuncID:186
*** formula/tFunc unknown FuncID:186
*** formula/tFunc unknown FuncID:186
*** formula/tFunc unknown FuncID:186


Traceback (most recent call last):
  File "/tmp/ipykernel_859/3179332798.py", line 17, in <cell line: 0>
    wb = xlrd.open_workbook(str(path))
  File "/usr/local/lib/python3.13/dist-packages/xlrd/__init__.py", line 172, in open_workbook
    bk = open_workbook_xls(
        filename=filename,
    ...<8 lines>...
        ignore_workbook_corruption=ignore_workbook_corruption,
    )
  File "/usr/local/lib/python3.13/dist-packages/xlrd/book.py", line 104, in open_workbook_xls
    bk.parse_globals()
    ~~~~~~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/xlrd/book.py", line 1251, in parse_globals
    self.names_epilogue()
    ~~~~~~~~~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/xlrd/book.py", line 1045, in names_epilogue
    evaluate_name_formula(self, nobj, namex, blah=blah)
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/xlrd/formula.py", line 1142, in evaluate_name_formula
    assert len(tgtobj.stack) 

In [ ]:
!pip -q install "xlwings" "python-calamine"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.0/935.0 kB 43.8 MB/s eta 0:00:00


In [ ]:
from pathlib import Path

EXTRACT_DIR = Path(
    "/content/drive/MyDrive/lucknow_census_processed/"
    "lucknow_zip_extracted/0926-Lucknow"
)

for number in [42, 43]:
    path = EXTRACT_DIR / f"T{number}_0926.xls"

    print("\n" + "=" * 60)
    print(f"TABLE T{number}: {path.name}")

    try:
        from python_calamine import CalamineWorkbook

        wb = CalamineWorkbook.from_path(str(path))

        print("Workbook opened successfully.")
        print("Sheets:", wb.sheet_names)

        for sheet_name in wb.sheet_names:
            sheet = wb.get_sheet_by_name(sheet_name)
            rows = sheet.to_python()

            print(f"\nSheet: {sheet_name}")
            print(f"Rows: {len(rows)}")

            for row in rows[:12]:
                print(row)

    except Exception as e:
        print(f"Alternative reader failed: {type(e).__name__}: {e}")


TABLE T42: T42_0926.xls
Workbook opened successfully.
Sheets: ['Sheet1']

Sheet: Sheet1
Rows: 19
['Table 42: Schools/ colleges per 10,000 population in towns, 2011', '', '', '', '', '', '']
['Sr. No. ', 'Name of the town', 'Type of educational institution (Approx. numbers)', '', '', '', '']
['', '', 'Primary ', ' Middle', 'Secondary / matriculation', 'Senior secondary', 'College*']
[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]
[1.0, '800946-Malihabad (NP)', 9.0, 4.0, 1.0, 1.0, 0.0]
[2.0, '800947-Mahona (NP)', 7.0, 4.0, 0.0, 0.0, 0.0]
[3.0, '800948-Itaunja (NP)', 11.0, 7.0, 3.0, 3.0, 0.0]
[4.0, '800949-Bakshi Ka Talab (NP)', 7.0, 4.0, 1.0, 1.0, 1.0]
[5.0, '800950-Kakori (NP)', 15.0, 9.0, 3.0, 1.0, 1.0]
[6.0, '800951-Lucknow (M Corp.)', 6.0, 3.0, 2.0, 1.0, 0.0]
[7.0, '800952-Lucknow (CB)', 3.0, 3.0, 2.0, 1.0, 0.0]
[8.0, '143558-Kalli Pashchim (CT)', 7.0, 1.0, 0.0, 0.0, 0.0]

TABLE T43: T43_0926.xls
Workbook opened successfully.
Sheets: ['Sheet1']

Sheet: Sheet1
Rows: 16
['Table 43: Number of beds

In [ ]:
from pathlib import Path
import csv
from python_calamine import CalamineWorkbook

BASE_DIR = Path(
    "/content/drive/MyDrive/lucknow_census_processed"
)
EXTRACT_DIR = BASE_DIR / "lucknow_zip_extracted" / "0926-Lucknow"
OUTPUT_DIR = BASE_DIR / "recovered_tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for number in [42, 43]:
    source = EXTRACT_DIR / f"T{number}_0926.xls"
    workbook = CalamineWorkbook.from_path(str(source))

    for sheet_name in workbook.sheet_names:
        rows = workbook.get_sheet_by_name(sheet_name).to_python()

        output = OUTPUT_DIR / f"T{number}_0926.csv"

        with output.open("w", newline="", encoding="utf-8-sig") as f:
            writer = csv.writer(f)
            writer.writerows(rows)

        print(f"Saved: {output}")
        print(f"Rows exported: {len(rows)}")

print("\nRecovery complete.")

Saved: /content/drive/MyDrive/lucknow_census_processed/recovered_tables/T42_0926.csv
Rows exported: 19
Saved: /content/drive/MyDrive/lucknow_census_processed/recovered_tables/T43_0926.csv
Rows exported: 16

Recovery complete.


In [ ]:
import pandas as pd

for number in [42, 43]:
    path = OUTPUT_DIR / f"T{number}_0926.csv"
    df = pd.read_csv(path, header=None)

    print(f"\nT{number}: {df.shape[0]} rows × {df.shape[1]} columns")
    print(df.head(10).to_string(index=False, header=False))


T42: 19 rows × 7 columns
Table 42: Schools/ colleges per 10,000 population in towns, 2011                         NaN                                               NaN     NaN                       NaN              NaN      NaN
                                                        Sr. No.             Name of the town Type of educational institution (Approx. numbers)     NaN                       NaN              NaN      NaN
                                                             NaN                         NaN                                          Primary   Middle Secondary / matriculation Senior secondary College*
                                                             1.0                         2.0                                               3.0     4.0                       5.0              6.0      7.0
                                                             1.0       800946-Malihabad (NP)                                               9.0     4.0            

In [ ]:

from pathlib import Path
import pandas as pd

BASE_DIR = Path(
    "/content/drive/MyDrive/lucknow_census_processed"
)

files_to_check = [
    "village_data_lucknow.csv",
    "village_data_lucknow_selected.csv",
    "town_data_lucknow.csv",
    "slum_data_lucknow.csv",
    "village_population_by_cd_block.csv",
    "data_dictionary.csv",
]

for filename in files_to_check:
    path = BASE_DIR / filename

    print("\n" + "=" * 75)
    print("FILE:", filename)

    if not path.exists():
        print("NOT FOUND")
        continue

    try:
        df = pd.read_csv(path, low_memory=False)

        print("Shape:", df.shape)
        print("Columns:")
        print(df.columns.tolist())

        print("\nFirst 3 rows:")
        print(df.head(3).to_string(index=False))

    except Exception as e:
        print("Could not read:", e)



FILE: village_data_lucknow.csv
Shape: (807, 396)
Columns:
['State Code', 'State Name', 'District Code', 'District Name', 'Sub District Code', 'Sub District Name', 'Village Code', 'Village Name', 'CD Block Code', 'CD Block Name', 'Gram Panchayat Code', 'Gram Panchayat Name', 'Reference Year', 'Sub District Head Quarter (Name)', 'Sub District Head Quarter (Distance in km)', 'District Head Quarter  (Name)', 'District Head Quarter  (Distance in km)', 'Nearest Statutory Town  (Name)', 'Nearest Statutory Town (Distance in km)', 'Within the State/UT (Name)', 'Within the State/UT (Distance in km)', 'Outside the State/UT, if nearest one is not within the State/UT (Name)', 'Outside the State/UT distance, if nearest one is not within the State/UT (Distance in km)', 'Total Geographical Area (in Hectares)', 'Total   Households ', 'Total Population of Village', 'Total Male Population of Village', 'Total Female Population of Village', 'Total Scheduled Castes Population of Village', 'Total Scheduled 

In [ ]:

from pathlib import Path
import pandas as pd

BASE_DIR = Path(
    "/content/drive/MyDrive/lucknow_census_processed"
)

# 1. Rural village population
villages = pd.read_csv(
    BASE_DIR / "village_data_lucknow.csv",
    low_memory=False
)

village_pop_col = "Total Population of Village"

villages[village_pop_col] = pd.to_numeric(
    villages[village_pop_col], errors="coerce"
)

print("RURAL VILLAGES")
print("Village records:", len(villages))
print("Unique village codes:", villages["Village Code"].nunique())
print("Sum of village population:", int(villages[village_pop_col].sum()))
print("\nPopulation by CD block:")
print(
    villages.groupby("CD Block Name", dropna=False)[village_pop_col]
    .sum()
    .sort_values(ascending=False)
    .to_string()
)

# 2. Urban town population
towns = pd.read_csv(
    BASE_DIR / "town_data_lucknow.csv",
    low_memory=False
)

print("\nURBAN TOWNS")
print("Town records:", len(towns))
print(
    towns[[
        "Town Code",
        "Town Name",
        "Total Population of Town"
    ]].to_string(index=False)
)

# 3. Existing extraction summary
summary_path = BASE_DIR / "village_population_by_cd_block.csv"

if summary_path.exists():
    print("\nEXISTING CD-BLOCK SUMMARY")
    print(pd.read_csv(summary_path).to_string(index=False))


RURAL VILLAGES
Village records: 807
Unique village codes: 807
Sum of village population: 1550842

Population by CD block:
CD Block Name
Mohanlalganj       248512
Bakshi-Ka-Talab    239938
Sarojaninagar      224045
Gosaiganj          198184
Malihabad          179673
Mal                170962
Kakori             152277
Chinhat            137251

URBAN TOWNS
Town records: 12
 Town Code            Town Name  Total Population of Town
    800946       Malihabad (NP)                     17818
    800947          Mahona (NP)                      8557
    800948         Itaunja (NP)                      7305
    800949 Bakshi Ka Talab (NP)                     49166
    143558  Kalli Pashchim (CT)                     12157
    143559        Sarsawan (CT)                     10655
    800950          Kakori (NP)                     19403
    800951    Lucknow (M Corp.)                   2817105
    800952         Lucknow (CB)                     63003
    800953      Gosainganj (NP)               

In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path(
    "/content/drive/MyDrive/lucknow_census_processed"
)

villages = pd.read_csv(
    BASE_DIR / "village_data_lucknow.csv",
    low_memory=False
)

pop_col = "Total Population of Village"
code_col = "Village Code"

villages[pop_col] = pd.to_numeric(
    villages[pop_col], errors="coerce"
)

print("=== RURAL DATA QUALITY CHECK ===")
print("Total records:", len(villages))
print("Missing village codes:", villages[code_col].isna().sum())
print("Duplicate village codes:", villages[code_col].duplicated().sum())
print("Missing population:", villages[pop_col].isna().sum())
print("Zero population:", (villages[pop_col] == 0).sum())
print("Negative population:", (villages[pop_col] < 0).sum())

print("\n=== POPULATION SUMMARY ===")
print(villages[pop_col].describe())

print("\n=== VILLAGES WITH MISSING OR ZERO POPULATION ===")
print(
    villages.loc[
        villages[pop_col].isna() | (villages[pop_col] <= 0),
        ["Village Code", "Village Name", "CD Block Name", pop_col]
    ].to_string(index=False)
)

print("\n=== POTENTIAL PCA-RELATED COLUMNS ===")
keywords = [
    "population", "male", "female", "literacy",
    "household", "scheduled caste", "scheduled tribe",
    "worker", "marginal", "main worker", "non worker",
    "age 0-6", "total population"
]

for col in villages.columns:
    if any(k in col.lower() for k in keywords):
        print(col)

=== RURAL DATA QUALITY CHECK ===
Total records: 807
Missing village codes: 0
Duplicate village codes: 0
Missing population: 0
Zero population: 4
Negative population: 0

=== POPULATION SUMMARY ===
count      807.000000
mean      1921.737299
std       1766.490193
min          0.000000
25%        844.000000
50%       1422.000000
75%       2412.000000
max      15658.000000
Name: Total Population of Village, dtype: float64

=== VILLAGES WITH MISSING OR ZERO POPULATION ===
 Village Code Village Name   CD Block Name  Total Population of Village
       143080 Raghu Rampur       Malihabad                            0
       143152    Sahilabad       Malihabad                            0
       143248   Prithvipur Bakshi-Ka-Talab                            0
       143356     Autrouli          Kakori                            0

=== POTENTIAL PCA-RELATED COLUMNS ===
Total   Households 
Total Population of Village
Total Male Population of Village
Total Female Population of Village
Total Schedul

In [ ]:
!pip -q install pdfplumber

from pathlib import Path
import pdfplumber
import re

PDF_PATH = Path(
    "/content/drive/MyDrive/Lucknow_Census_Raw/"
    "DH_2011_0926_PART_B_DCHB_LUCKNOW.pdf"
)

if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"PDF not found: {PDF_PATH}\n"
        "Check the filename and your Google Drive folder."
    )

keywords = [
    "primary census abstract",
    "p.c.a.",
    "ward",
    "total population",
    "scheduled caste",
    "scheduled tribe",
    "literates",
    "main workers",
    "marginal workers",
]

matches = []

with pdfplumber.open(PDF_PATH) as pdf:
    print("Total PDF pages:", len(pdf.pages))

    for page_number, page in enumerate(pdf.pages, start=1):
        text = page.extract_text() or ""
        lower_text = text.lower()

        found = [k for k in keywords if k in lower_text]

        if found:
            matches.append({
                "page": page_number,
                "keywords": ", ".join(found),
                "preview": re.sub(r"\s+", " ", text)[:450]
            })

print("\n=== MATCHING PAGES ===")

for item in matches:
    print(f"\nPDF page {item['page']}")
    print("Matched terms:", item["keywords"])
    print("Preview:", item["preview"])

# Save results for later use
import pandas as pd

output_dir = Path(
    "/content/drive/MyDrive/lucknow_census_processed"
)
output_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame(matches).to_csv(
    output_dir / "part_b_pca_candidate_pages.csv",
    index=False
)

print("\nSaved candidate pages to part_b_pca_candidate_pages.csv")

Total PDF pages: 350

=== MATCHING PAGES ===

PDF page 1
Matched terms: primary census abstract
Preview: Census of India 2011 UTTAR PRADESH PART XII-B SERIES-10 DISTRICT CENSUS HANDBOOK LUCKNOW VILLAGE AND TOWN WISE PRIMARY CENSUS ABSTRACT (PCA) DIRECTORATE OF CENSUS OPERATIONS UTTAR PRADESH

PDF page 4
Matched terms: primary census abstract
Preview: CENSUS OF INDIA 2011 UTTAR PRADESH SERIES-10 PART XII - B DISTRICT CENSUS HANDBOOK LUCKNOW VILLAGE AND TOWN WISE PRIMARY CENSUS ABSTRACT (PCA) Directorate of Census Operations UTTAR PRADESH

PDF page 6
Matched terms: main workers
Preview: Census House: A ‘census house’ is a building or part of a building used or recognized as a separate unit because of having a separate main entrance from the road or common courtyard or staircase, etc. It may be occupied or vacant. It may be used for residential or non- residential purpose or both. If a building has a number of Flats or Blocks/Wings, which are independent of one another having separate ent

In [ ]:
!pip -q install pdfplumber

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [ ]:
!pip -q install pdfplumber

import pdfplumber
from pathlib import Path

pdf_path = Path(
    "/content/drive/MyDrive/Lucknow_Census_Raw/"
    "DH_2011_0926_PART_B_DCHB_LUCKNOW.pdf"
)

with pdfplumber.open(pdf_path) as pdf:
    for page_no in range(307, 311):
        page = pdf.pages[page_no - 1]

        print("\n" + "=" * 70)
        print(f"PDF PAGE {page_no}")
        print("=" * 70)

        text = page.extract_text()
        print(text[:10000] if text else "No extractable text")

        tables = page.extract_tables()
        print("\nTables detected:", len(tables))

        for table_no, table in enumerate(tables, start=1):
            print(f"\nTable {table_no}")
            for row in table[:5]:
                print(row)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 58.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.

PDF PAGE 307
DISTRICT CENSUS HANDBOOK : LUCKNOW
URBAN PRIMARY
Area in Total population (including
Location Square institutional and houseless
code Kilometr Number of population) Population in the age-group 0-6
number Name of Town/ Ward e households Persons Males 

In [ ]:

from google.colab import drive
import os

mount_path = "/content/gdrive"

os.makedirs(mount_path, exist_ok=True)

drive.mount(mount_path)


Mounted at /content/gdrive


In [ ]:

import pdfplumber
import pandas as pd
import re
from pathlib import Path

pdf_path = Path(
    "/content/gdrive/MyDrive/Lucknow_Census_Raw/"
    "DH_2011_0926_PART_B_DCHB_LUCKNOW.pdf"
)

output_dir = Path(
    "/content/gdrive/MyDrive/lucknow_census_processed"
)
output_dir.mkdir(parents=True, exist_ok=True)

records = []

with pdfplumber.open(pdf_path) as pdf:
    # PCA begins on printed page 288, corresponding to PDF page 307.
    # Extend the range to capture the subsequent worker-category tables.
    for pdf_page_no in range(307, 314):
        page = pdf.pages[pdf_page_no - 1]
        text = page.extract_text()

        if not text:
            continue

        for line in text.splitlines():
            line = line.strip()

            if line:
                records.append({
                    "pdf_page": pdf_page_no,
                    "raw_line": line
                })

raw_df = pd.DataFrame(records)

output_path = output_dir / "lucknow_urban_pca_raw_lines.csv"
raw_df.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Extracted lines:", len(raw_df))
print("\nSample lines:")
print(raw_df.head(25).to_string(index=False))


Saved: /content/gdrive/MyDrive/lucknow_census_processed/lucknow_urban_pca_raw_lines.csv
Extracted lines: 473

Sample lines:
 pdf_page                                                                           raw_line
      307                                                 DISTRICT CENSUS HANDBOOK : LUCKNOW
      307                                                                      URBAN PRIMARY
      307                                                Area in Total population (including
      307                                        Location Square institutional and houseless
      307                code Kilometr Number of population) Population in the age-group 0-6
      307 number Name of Town/ Ward e households Persons Males Females Persons Males Females
      307                                                               1 2 3 4 5 6 7 8 9 10
      307       800946 Malihabad (NP) 6.00 3 ,032 1 7,818 9 ,334 8 ,484 2 ,184 1 ,111 1 ,073
      307                              

In [ ]:

import pdfplumber
from pathlib import Path

pdf_path = Path(
    "/content/gdrive/MyDrive/Lucknow_Census_Raw/"
    "DH_2011_0926_PART_B_DCHB_LUCKNOW.pdf"
)

with pdfplumber.open(pdf_path) as pdf:
    for page_no in [307, 308, 309, 310]:
        page = pdf.pages[page_no - 1]

        print("\n" + "=" * 90)
        print(f"PAGE {page_no} — WORD POSITIONS")
        print("=" * 90)

        words = page.extract_words(
            x_tolerance=1,
            y_tolerance=3,
            keep_blank_chars=False
        )

        # Show words from the first few data rows only
        shown_rows = []
        for word in words:
            if not shown_rows or abs(
                word["top"] - shown_rows[-1][0]["top"]
            ) > 3:
                shown_rows.append([word])
            else:
                shown_rows[-1].append(word)

        for row in shown_rows[:12]:
            print(
                " | ".join(
                    f'{w["text"]} (x={w["x0"]:.1f})'
                    for w in row
                )
            )



PAGE 307 — WORD POSITIONS
DISTRICT (x=346.4) | CENSUS (x=385.8) | HANDBOOK (x=421.5) | : (x=469.8) | LUCKNOW (x=474.3)
URBAN (x=429.1) | PRIMARY (x=467.1)
Area (x=161.2) | in (x=180.7) | Total (x=267.1) | population (x=288.3) | (including (x=329.4)
Location (x=26.8) | Square (x=162.0) | institutional (x=269.2) | and (x=314.7) | houseless (x=330.0)
code (x=34.3) | Kilometr (x=158.3) | Number (x=202.0) | of (x=233.6) | population) (x=296.0) | Population (x=397.7) | in (x=439.4) | the (x=448.7) | age-group (x=462.0) | 0-6 (x=500.8)
number (x=29.0) | Name (x=69.2) | of (x=92.8) | Town/ (x=102.5) | Ward (x=128.1) | e (x=172.6) | households (x=201.1) | Persons (x=257.6) | Males (x=305.5) | Females (x=347.2) | Persons (x=400.2) | Males (x=444.7) | Females (x=482.2)
1 (x=40.8) | 2 (x=106.7) | 3 (x=172.6) | 4 (x=219.6) | 5 (x=269.5) | 6 (x=314.3) | 7 (x=359.8) | 8 (x=412.0) | 9 (x=453.4) | 10 (x=492.5)
800946 (x=24.7) | Malihabad (x=64.0) | (NP) (x=105.7) | 6.00 (x=176.8) | 3 (x=224.4) | ,032 

In [ ]:

import pdfplumber
import pandas as pd
import re
from pathlib import Path

pdf_path = Path(
    "/content/gdrive/MyDrive/Lucknow_Census_Raw/"
    "DH_2011_0926_PART_B_DCHB_LUCKNOW.pdf"
)

output_dir = Path(
    "/content/gdrive/MyDrive/lucknow_census_processed"
)
output_dir.mkdir(parents=True, exist_ok=True)

def clean_number(text):
    """Join digit fragments such as '1 ,588' into 1588."""
    text = text.strip().replace(",", "").replace(" ", "")
    if not text or text == "-":
        return None
    if re.fullmatch(r"\d+(\.\d+)?", text):
        return float(text) if "." in text else int(text)
    return None

records = []

with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[306]  # PDF page 307

    words = page.extract_words(
        x_tolerance=1,
        y_tolerance=3,
        keep_blank_chars=False
    )

    # Group words into horizontal rows
    rows = []
    for word in words:
        if not rows or abs(word["top"] - rows[-1][0]["top"]) > 3:
            rows.append([word])
        else:
            rows[-1].append(word)

    # The first 7 rows are titles and column headings.
    for row in rows[7:]:
        row = sorted(row, key=lambda w: w["x0"])
        text = " ".join(w["text"] for w in row)

        # Town summary rows begin with a 6-digit town code.
        town_match = re.match(r"^(\d{6})\s+(.+?)\s+(\d+\.\d+)", text)

        # Ward rows begin with a four-digit ward code.
        ward_match = re.match(r"^(\d{4})\s+Ward\s+No\.\s*(\d+)", text)

        if town_match:
            town_code = town_match.group(1)
            town_name = town_match.group(2).strip()
            row_type = "town"
            ward_code = None
            ward_name = None
        elif ward_match:
            row_type = "ward"
            ward_code = ward_match.group(1)
            ward_name = f"Ward No. {ward_match.group(2)}"
            town_name = None
            town_code = None
        else:
            continue

        # Print identified rows for validation before extracting values.
        records.append({
            "row_type": row_type,
            "town_code": town_code,
            "town_name": town_name,
            "ward_code": ward_code,
            "ward_name": ward_name,
            "raw_text": text
        })

df = pd.DataFrame(records)

print("Rows identified:", len(df))
print(df.head(20).to_string(index=False))

test_path = output_dir / "urban_pca_page307_row_test.csv"
df.to_csv(test_path, index=False)
print("\nSaved test file:", test_path)


Rows identified: 60
row_type town_code      town_name ward_code   ward_name                                                                     raw_text
    town    800946 Malihabad (NP)      None        None 800946 Malihabad (NP) 6.00 3 ,032 1 7,818 9 ,334 8 ,484 2 ,184 1 ,111 1 ,073
    ward      None           None      0001  Ward No. 1                          0001 Ward No. 1 2 56 1 ,588 8 44 7 44 1 98 1 06 9 2
    ward      None           None      0002  Ward No. 2                           0002 Ward No. 2 2 37 1 ,377 7 19 6 58 1 54 7 5 7 9
    ward      None           None      0003  Ward No. 3                           0003 Ward No. 3 2 90 1 ,654 9 04 7 50 1 80 9 4 8 6
    ward      None           None      0004  Ward No. 4                           0004 Ward No. 4 2 02 1 ,175 5 87 5 88 1 62 7 2 9 0
    ward      None           None      0005  Ward No. 5                         0005 Ward No. 5 2 61 1 ,714 8 63 8 51 2 36 1 20 1 16
    ward      None           None      0006  Ward

In [ ]:

import pdfplumber
import pandas as pd
import re
from pathlib import Path

pdf_path = Path(
    "/content/gdrive/MyDrive/Lucknow_Census_Raw/"
    "DH_2011_0926_PART_B_DCHB_LUCKNOW.pdf"
)

output_dir = Path(
    "/content/gdrive/MyDrive/lucknow_census_processed"
)
output_dir.mkdir(parents=True, exist_ok=True)

def parse_numeric_cell(words):
    """Join fragments within one numeric column."""
    if not words:
        return None

    text = "".join(w["text"] for w in sorted(words, key=lambda w: w["x0"]))
    text = text.replace(",", "").replace(" ", "")

    if text in ("", "-"):
        return None

    if re.fullmatch(r"\d+(?:\.\d+)?", text):
        return float(text) if "." in text else int(text)

    return None

# Page 307 numeric columns:
# area, households, total population, males, females,
# age 0-6 total, age 0-6 males, age 0-6 females.
column_ranges = [
    (155, 195),   # area
    (195, 250),   # households
    (250, 300),   # total population
    (300, 345),   # population males
    (345, 395),   # population females
    (395, 440),   # age 0-6 total
    (440, 480),   # age 0-6 males
    (480, 540),   # age 0-6 females
]

records = []
current_town_code = None
current_town_name = None

with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[306]  # PDF page 307

    words = page.extract_words(
        x_tolerance=1,
        y_tolerance=3,
        keep_blank_chars=False
    )

    # Group words by their vertical position.
    rows = []
    for word in words:
        if not rows or abs(word["top"] - rows[-1][0]["top"]) > 3:
            rows.append([word])
        else:
            rows[-1].append(word)

    for row in rows:
        row = sorted(row, key=lambda w: w["x0"])
        row_text = " ".join(w["text"] for w in row)

        town_match = re.match(
            r"^(\d{6})\s+(.+?)\s+(\d+\.\d+)",
            row_text
        )
        ward_match = re.match(
            r"^(\d{4})\s+Ward\s+No\.\s*(\d+)",
            row_text
        )

        if town_match:
            current_town_code = town_match.group(1)
            current_town_name = town_match.group(2).strip()
            row_type = "town"
            ward_code = None
            ward_name = None

        elif ward_match and current_town_code:
            row_type = "ward"
            ward_code = ward_match.group(1)
            ward_name = f"Ward No. {ward_match.group(2)}"

        else:
            continue

        values = []
        for left, right in column_ranges:
            cell_words = [
                w for w in row
                if left <= w["x0"] < right
            ]
            values.append(parse_numeric_cell(cell_words))

        records.append({
            "row_type": row_type,
            "town_code": current_town_code,
            "town_name": current_town_name,
            "ward_code": ward_code,
            "ward_name": ward_name,
            "area_sq_km": values[0],
            "households": values[1],
            "population_total": values[2],
            "population_male": values[3],
            "population_female": values[4],
            "age_0_6_total": values[5],
            "age_0_6_male": values[6],
            "age_0_6_female": values[7],
        })

df = pd.DataFrame(records)

test_path = output_dir / "urban_pca_page307_numeric_test.csv"
df.to_csv(test_path, index=False)

print("Rows extracted:", len(df))
print("\nFirst 15 records:")
print(df.head(15).to_string(index=False))

print("\nTown summary rows:")
print(
    df[df["row_type"] == "town"][
        ["town_code", "town_name", "households", "population_total"]
    ].to_string(index=False)
)

print("\nSaved:", test_path)


Rows extracted: 60

First 15 records:
row_type town_code      town_name ward_code   ward_name  area_sq_km  households  population_total  population_male  population_female  age_0_6_total  age_0_6_male  age_0_6_female
    town    800946 Malihabad (NP)      None        None        6.00        3032             17818             9334               8484           2184          1111            1073
    ward    800946 Malihabad (NP)      0001  Ward No. 1         NaN         256              1588              844                744            198           106              92
    ward    800946 Malihabad (NP)      0002  Ward No. 2         NaN         237              1377              719                658            154            75              79
    ward    800946 Malihabad (NP)      0003  Ward No. 3         NaN         290              1654              904                750            180            94              86
    ward    800946 Malihabad (NP)      0004  Ward No. 4         NaN

In [ ]:

import pandas as pd

towns = df[df["row_type"] == "town"].copy()
wards = df[df["row_type"] == "ward"].copy()

# 1. Check total ward population against town population
ward_sums = (
    wards.groupby(["town_code", "town_name"], as_index=False)
    .agg(
        ward_count=("ward_code", "count"),
        ward_population=("population_total", "sum"),
        ward_households=("households", "sum"),
    )
)

validation = towns[
    ["town_code", "town_name", "population_total", "households"]
].merge(
    ward_sums,
    on=["town_code", "town_name"],
    how="left"
)

validation["population_difference"] = (
    validation["ward_population"] - validation["population_total"]
)

validation["household_difference"] = (
    validation["ward_households"] - validation["households"]
)

print("=== TOWN VS WARD VALIDATION ===")
print(validation.to_string(index=False))

# 2. Check male + female = total population
check_rows = df[
    df["population_total"].notna()
    & df["population_male"].notna()
    & df["population_female"].notna()
].copy()

check_rows["sex_total_difference"] = (
    check_rows["population_male"]
    + check_rows["population_female"]
    - check_rows["population_total"]
)

print("\n=== MALE/FEMALE POPULATION CHECK ===")
print(
    "Rows checked:", len(check_rows),
    "| Mismatches:",
    (check_rows["sex_total_difference"] != 0).sum()
)

print(
    check_rows.loc[
        check_rows["sex_total_difference"] != 0,
        ["town_code", "town_name", "ward_code",
         "population_total", "population_male",
         "population_female", "sex_total_difference"]
    ].to_string(index=False)
)

# 3. Check age 0–6 totals
age_rows = df[
    df["age_0_6_total"].notna()
    & df["age_0_6_male"].notna()
    & df["age_0_6_female"].notna()
].copy()

age_rows["age_difference"] = (
    age_rows["age_0_6_male"]
    + age_rows["age_0_6_female"]
    - age_rows["age_0_6_total"]
)

print("\n=== AGE 0–6 CHECK ===")
print(
    "Rows checked:", len(age_rows),
    "| Mismatches:", (age_rows["age_difference"] != 0).sum()
)


=== TOWN VS WARD VALIDATION ===
town_code            town_name  population_total  households  ward_count  ward_population  ward_households  population_difference  household_difference
   800946       Malihabad (NP)             17818        3032          13            17818             3032                      0                     0
   800947          Mahona (NP)              8557        1455          10             8557             1455                      0                     0
   800948         Itaunja (NP)              7305        1246          10             7305             1246                      0                     0
   800949 Bakshi Ka Talab (NP)             49166        8728          23            49166             8728                      0                     0

=== MALE/FEMALE POPULATION CHECK ===
Rows checked: 60 | Mismatches: 0
Empty DataFrame
Columns: [town_code, town_name, ward_code, population_total, population_male, population_female, sex_total_difference]
In

In [ ]:

import pdfplumber
from pathlib import Path

pdf_path = Path(
    "/content/gdrive/MyDrive/Lucknow_Census_Raw/"
    "DH_2011_0926_PART_B_DCHB_LUCKNOW.pdf"
)

with pdfplumber.open(pdf_path) as pdf:
    for page_no in range(311, 314):
        page = pdf.pages[page_no - 1]

        print("\n" + "=" * 80)
        print(f"PDF PAGE {page_no}")
        print("=" * 80)

        words = page.extract_words(
            x_tolerance=1,
            y_tolerance=3,
            keep_blank_chars=False
        )

        # Group words by their vertical position.
        rows = []
        for word in words:
            if (
                not rows
                or abs(word["top"] - rows[-1][0]["top"]) > 3
            ):
                rows.append([word])
            else:
                rows[-1].append(word)

        # Print the heading and first 8 data rows.
        for row in rows[:15]:
            row = sorted(row, key=lambda w: w["x0"])
            print(" | ".join(w["text"] for w in row))



PDF PAGE 311
DISTRICT | CENSUS | HANDBOOK | : | LUCKNOW
URBAN | PRIMARY
Industrial | category
Location
code | Marginal | workers | Cultivators | Agricultural | labourers
number | Name | of | Town/ | Ward | Persons | Males | Females | Persons | Males | Females | Persons | Males | Females
1 | 2 | 41 | 42 | 43 | 44 | 45 | 46 | 47 | 48 | 49
800946 | Malihabad | (NP) | 1 | ,844 | 1 | ,487 | 3 | 57 | 4 | 1 | 3 | 6 | 5 | 3 | 84 | 3 | 10 | 7 | 4
0001 | Ward | No. | 1 | 3 | 28 | 2 | 69 | 5 | 9 | 3 | 3 | - | 1 | 44 | 1 | 20 | 2 | 4
0002 | Ward | No. | 2 | 9 | 4 | 5 | 9 | 3 | 5 | - | - | - | 8 | 5 | 3
0003 | Ward | No. | 3 | 1 | 76 | 1 | 13 | 6 | 3 | 1 | 1 | 9 | 2 | 2 | 6 | 1 | 8 | 8
0004 | Ward | No. | 4 | 3 | 3 | 1 | 8 | 1 | 5 | - | - | - | 1 | 1 | -
0005 | Ward | No. | 5 | 2 | 35 | 1 | 93 | 4 | 2 | 4 | 2 | 2 | 6 | 5 | 1
0006 | Ward | No. | 6 | 1 | 17 | 1 | 06 | 1 | 1 | 1 | 1 | - | 2 | 2 | 2 | 0 | 2
0007 | Ward | No. | 7 | 5 | 9 | 4 | 9 | 1 | 0 | - | - | - | 8 | 7 | 1

PDF PAGE 312
PRIMARY | C

In [ ]:

import pdfplumber
from pathlib import Path

pdf_path = Path(
    "/content/gdrive/MyDrive/Lucknow_Census_Raw/"
    "DH_2011_0926_PART_B_DCHB_LUCKNOW.pdf"
)

keywords = [
    "Non-workers",
    "PRIMARY CENSUS ABSTRACT",
    "Industrial category",
    "Location code",
    "Name of Town"
]

with pdfplumber.open(pdf_path) as pdf:
    for page_no in range(307, 351):
        text = pdf.pages[page_no - 1].extract_text() or ""
        lower_text = text.lower()

        matches = [
            keyword for keyword in keywords
            if keyword.lower() in lower_text
        ]

        if matches:
            print(
                f"PDF page {page_no}: "
                + ", ".join(matches)
            )


PDF page 307: Name of Town
PDF page 308: PRIMARY CENSUS ABSTRACT, Name of Town
PDF page 309: Name of Town
PDF page 310: PRIMARY CENSUS ABSTRACT, Industrial category, Name of Town
PDF page 311: Industrial category, Name of Town
PDF page 312: Non-workers, PRIMARY CENSUS ABSTRACT, Name of Town
PDF page 313: Name of Town
PDF page 314: PRIMARY CENSUS ABSTRACT, Name of Town
PDF page 315: Name of Town
PDF page 316: PRIMARY CENSUS ABSTRACT, Industrial category, Name of Town
PDF page 317: Industrial category, Name of Town
PDF page 318: Non-workers, PRIMARY CENSUS ABSTRACT, Name of Town
PDF page 319: Name of Town
PDF page 320: PRIMARY CENSUS ABSTRACT, Name of Town
PDF page 321: Name of Town
PDF page 322: PRIMARY CENSUS ABSTRACT, Industrial category, Name of Town
PDF page 323: Industrial category, Name of Town
PDF page 324: Non-workers, PRIMARY CENSUS ABSTRACT, Name of Town
PDF page 325: Name of Town
PDF page 326: PRIMARY CENSUS ABSTRACT, Name of Town
PDF page 327: Name of Town
PDF page 328: PRIM

In [ ]:
import pdfplumber
from pathlib import Path

pdf_path = Path(
    "/content/gdrive/MyDrive/Lucknow_Census_Raw/"
    "DH_2011_0926_PART_B_DCHB_LUCKNOW.pdf"
)

keywords = [
    "Non-workers",
    "PRIMARY CENSUS ABSTRACT",
    "Industrial category",
    "Name of Town",
    "Location code",
    "Urban PCA",
    "Rural PCA",
]

with pdfplumber.open(pdf_path) as pdf:
    for page_no in range(331, min(351, len(pdf.pages) + 1)):
        text = pdf.pages[page_no - 1].extract_text() or ""
        matches = [
            keyword for keyword in keywords
            if keyword.lower() in text.lower()
        ]

        if matches:
            print(f"PDF page {page_no}: {', '.join(matches)}")

In [ ]:

import pdfplumber
from pathlib import Path

pdf_path = Path(
    "/content/gdrive/MyDrive/Lucknow_Census_Raw/"
    "DH_2011_0926_PART_B_DCHB_LUCKNOW.pdf"
)

print("PDF exists:", pdf_path.exists())

if pdf_path.exists():
    with pdfplumber.open(pdf_path) as pdf:
        print("Total PDF pages:", len(pdf.pages))

        for page_no in range(331, min(351, len(pdf.pages) + 1)):
            text = pdf.pages[page_no - 1].extract_text() or ""

            print(f"\n--- PDF page {page_no} ---")
            print("Extracted characters:", len(text))
            print(repr(text[:350]))


PDF exists: True
Total PDF pages: 350

--- PDF page 331 ---
Extracted characters: 0
''

--- PDF page 332 ---
Extracted characters: 110
'Section – II\nTables based on Households\nAmenities and Assets\n(Rural /Urban) at District and\nSub-District level'

--- PDF page 333 ---
Extracted characters: 0
''

--- PDF page 334 ---
Extracted characters: 1687
'Table -1: Households by Ownership Status and by Number of Dwelling rooms occupied in the District, 2011\nName of Ownership Number of dwelling Number of households\ndistrict status rooms Total Scheduled Castes Scheduled Tribes\nTotal Rural Urban Total Rural Urban Total Rural Urban\n1 2 3 4 5 6 7 8 9 10 11 12\nOwned No exclusive room 22,188 13,478 8,710 7'

--- PDF page 335 ---
Extracted characters: 1528
'Table - 2 : Percentage distribution of Households living in Permanent,\nSl. No. Name of Sub - District Total\nPermanent Semi permanent Temporary Permanent\n1 2 3 4 5 6\n1 Sub-District - Malihabad (0915700819) 27,822 (42.78%) 5,438 (8.36%) 30,

In [ ]:

import pdfplumber
from pathlib import Path

pdf_path = Path(
    "/content/gdrive/MyDrive/Lucknow_Census_Raw/"
    "DH_2011_0926_PART_B_DCHB_LUCKNOW.pdf"
)

with pdfplumber.open(pdf_path) as pdf:
    for page_no in range(307, 331):
        text = pdf.pages[page_no - 1].extract_text() or ""
        lines = [line.strip() for line in text.splitlines() if line.strip()]

        print(f"\n{'=' * 60}")
        print(f"PDF PAGE {page_no} | Characters: {len(text)}")

        for line in lines[:5]:
            print(line[:200])



PDF PAGE 307 | Characters: 3459
DISTRICT CENSUS HANDBOOK : LUCKNOW
URBAN PRIMARY
Area in Total population (including
Location Square institutional and houseless
code Kilometr Number of population) Population in the age-group 0-6

PDF PAGE 308 | Characters: 3055
PRIMARY CENSUS ABSTRACT
CENSUS ABSTRACT
Scheduled Castes population Scheduled Tribes population Literates
Persons Males Females Persons Males Females Persons Males Females Name of Town/ Ward
11 12 13 14 15 16 17 18 19 2

PDF PAGE 309 | Characters: 3890
DISTRICT CENSUS HANDBOOK : LUCKNOW
URBAN PRIMARY
Location
code Illiterates Total workers Main workers
number Name of Town/ Ward Persons Males Females Persons Males Females Persons Males Females

PDF PAGE 310 | Characters: 3388
PRIMARY CENSUS ABSTRACT
CENSUS ABSTRACT
Industrial category of main workers
Household industry
Cultivators Agricultural labourers workers Other workers Name of Town/

PDF PAGE 311 | Characters: 3094
DISTRICT CENSUS HANDBOOK : LUCKNOW
URBAN PRIMARY
Industria

In [ ]:

import pdfplumber
from pathlib import Path

pdf_path = Path(
    "/content/gdrive/MyDrive/Lucknow_Census_Raw/"
    "DH_2011_0926_PART_B_DCHB_LUCKNOW.pdf"
)

with pdfplumber.open(pdf_path) as pdf:
    for page_no in [308, 309, 310, 311, 312]:
        page = pdf.pages[page_no - 1]
        print(f"\n{'=' * 70}\nPDF PAGE {page_no}")

        text = page.extract_text(
            x_tolerance=1,
            y_tolerance=3
        ) or ""

        for line in text.splitlines()[:12]:
            print(line)



PDF PAGE 308
PRIMARY CENSUS ABSTRACT
CENSUS ABSTRACT
Scheduled Castes population Scheduled Tribes population Literates
Persons Males Females Persons Males Females Persons Males Females Name of Town/ Ward
11 12 13 14 15 16 17 18 19 2
1 ,547 8 39 7 08 - - - 1 0,854 6 ,037 4 ,817 Malihabad (NP)
3 23 1 69 1 54 - - - 7 94 4 60 3 34 Ward No. 1
1 22 8 3 3 9 - - - 8 64 4 95 3 69 Ward No. 2
2 73 1 50 1 23 - - - 9 05 5 50 3 55 Ward No. 3
2 17 1 18 9 9 - - - 6 84 3 66 3 18 Ward No. 4
4 0 2 0 2 0 - - - 7 82 4 03 3 79 Ward No. 5
7 9 4 1 3 8 - - - 9 21 5 34 3 87 Ward No. 6

PDF PAGE 309
DISTRICT CENSUS HANDBOOK : LUCKNOW
URBAN PRIMARY
Location
code Illiterates Total workers Main workers
number Name of Town/ Ward Persons Males Females Persons Males Females Persons Males Females
1 2 20 21 22 23 24 25 26 27 28
800946 Malihabad (NP) 6 ,964 3 ,297 3 ,667 5 ,886 5 ,066 8 20 4 ,042 3 ,579 4 63
0001 Ward No. 1 7 94 3 84 4 10 5 16 4 41 7 5 1 88 1 72 1 6
0002 Ward No. 2 5 13 2 24 2 89 4 68 4 05 6 3 3 74 3 46

In [ ]:
import pdfplumber
from pathlib import Path

pdf_path = Path(
    "/content/gdrive/MyDrive/Lucknow_Census_Raw/"
    "DH_2011_0926_PART_B_DCHB_LUCKNOW.pdf"
)

with pdfplumber.open(pdf_path) as pdf:
    for page_no in [308, 310, 312]:
        page = pdf.pages[page_no - 1]

        print(f"\n{'=' * 90}")
        print(f"PDF PAGE {page_no}")
        print(f"{'=' * 90}")

        words = page.extract_words(
            x_tolerance=1,
            y_tolerance=3,
            keep_blank_chars=False
        )

        # Group words by their approximate vertical position.
        rows = {}
        for word in words:
            y = round(word["top"], 1)
            rows.setdefault(y, []).append(word)

        # Print the first few rows containing data.
        shown = 0
        for y in sorted(rows):
            row = sorted(rows[y], key=lambda w: w["x0"])

            text = " ".join(w["text"] for w in row)
            if any(
                marker in text
                for marker in [
                    "Malihabad", "Ward No. 1", "800946", "0001"
                ]
            ):
                print(f"\nY = {y}")
                print(
                    " | ".join(
                        f"{w['text']} (x={w['x0']:.1f}-{w['x1']:.1f})"
                        for w in row
                    )
                )
                shown += 1

            if shown >= 3:
                break


PDF PAGE 308

Y = 159.1
Malihabad (x=413.6-453.1) | (NP) (x=455.4-471.9)

Y = 169.2
Ward (x=413.6-433.1) | No. (x=435.4-447.8) | 1 (x=450.1-454.6)

Y = 261.0
Ward (x=413.6-433.1) | No. (x=435.4-447.8) | 10 (x=450.1-459.0)

PDF PAGE 310

Y = 159.1
Malihabad (x=455.5-495.0) | (NP) (x=497.3-513.7)

Y = 169.2
Ward (x=455.5-475.0) | No. (x=477.3-489.7) | 1 (x=492.0-496.5)

Y = 261.0
Ward (x=455.5-475.0) | No. (x=477.3-489.7) | 10 (x=492.0-500.9)

PDF PAGE 312

Y = 159.1
Malihabad (x=398.3-437.7) | (NP) (x=440.0-456.5)

Y = 159.2
4 (x=42.4-46.8) | 14 (x=46.8-55.7) | 3 (x=79.3-83.8) | 37 (x=83.8-92.7) | 7 (x=120.7-125.2) | 7 (x=125.2-129.6) | 1 (x=157.0-161.4) | ,005 (x=161.4-177.0) | 8 (x=204.4-208.8) | 04 (x=208.8-217.7) | 2 (x=241.3-245.8) | 01 (x=245.8-254.7) | 1 (x=281.3-285.8) | 1,932 (x=285.7-305.8) | 4 (x=327.1-331.6) | ,268 (x=331.6-347.2) | 7 (x=372.2-376.7) | ,664 (x=376.7-392.3) | 800946 (x=490.8-517.5)

Y = 169.2
Ward (x=398.3-417.8) | No. (x=420.0-432.5) | 1 (x=434.8-439.2)


In [ ]:
import pdfplumber
from pathlib import Path

pdf_path = Path(
    "/content/gdrive/MyDrive/Lucknow_Census_Raw/"
    "DH_2011_0926_PART_B_DCHB_LUCKNOW.pdf"
)

with pdfplumber.open(pdf_path) as pdf:
    for page_no in [308, 310, 312]:
        page = pdf.pages[page_no - 1]
        words = page.extract_words(
            x_tolerance=1,
            y_tolerance=3,
            keep_blank_chars=False
        )

        print(f"\n{'=' * 90}")
        print(f"PDF PAGE {page_no}")
        print(f"{'=' * 90}")

        # Find the vertical positions of the town and Ward No. 1.
        target_y = set()

        for w in words:
            if w["text"] == "Malihabad":
                target_y.add(round(w["top"], 1))
            elif w["text"] == "Ward":
                target_y.add(round(w["top"], 1))

        for y in sorted(target_y):
            row = [
                w for w in words
                if abs(w["top"] - y) <= 0.8
            ]
            row.sort(key=lambda w: w["x0"])

            print(f"\nY = {y}")
            print(
                " | ".join(
                    f"{w['text']} [{w['x0']:.1f}-{w['x1']:.1f}]"
                    for w in row
                )
            )


PDF PAGE 308

Y = 131.1
Persons [30.1-58.3] | Males [74.6-96.6] | Females [112.1-141.8] | Persons [158.8-187.0] | Males [203.3-225.2] | Females [240.7-270.5] | Persons [289.2-317.4] | Males [337.4-359.4] | Females [376.8-406.6] | Name [418.9-440.2] | of [442.4-450.0] | Town/ [452.1-475.5] | Ward [477.8-497.7]

Y = 159.1
1 [40.1-44.6] | ,547 [44.5-60.2] | 8 [88.2-92.7] | 39 [92.6-101.6] | 7 [129.6-134.1] | 08 [134.0-143.0] | - [177.2-179.9] | - [218.6-221.3] | - [260.0-262.7] | 1 [296.6-301.1] | 0,854 [301.1-321.2] | 6 [346.2-350.7] | ,037 [350.6-366.3] | 4 [387.6-392.1] | ,817 [392.0-407.7] | Malihabad [413.6-453.1] | (NP) [455.4-471.9]

Y = 169.2
3 [46.8-51.3] | 23 [51.2-60.2] | 1 [88.2-92.7] | 69 [92.6-101.6] | 1 [129.6-134.1] | 54 [134.0-143.0] | - [177.2-179.9] | - [218.6-221.3] | - [260.0-262.7] | 7 [307.8-312.3] | 94 [312.2-321.2] | 4 [352.9-357.4] | 60 [357.4-366.3] | 3 [394.3-398.8] | 34 [398.8-407.7] | Ward [413.6-433.1] | No. [435.4-447.8] | 1 [450.1-454.6]

Y = 179.4
1 [46.

In [ ]:

# STEP 1: Extract word-level data from urban PCA pages 307–312

!pip -q install pdfplumber pandas

import os
import pdfplumber
import pandas as pd

PDF_PATH = (
    "/content/gdrive/MyDrive/Lucknow_Census_Raw/"
    "DH_2011_0926_PART_B_DCHB_LUCKNOW.pdf"
)

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RAW_CSV = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_words_307_312.csv"
)

if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(
        f"PDF not found at:\n{PDF_PATH}\n"
        "Check the Google Drive mount and file name."
    )

records = []

with pdfplumber.open(PDF_PATH) as pdf:
    print("PDF pages:", len(pdf.pages))

    # These are PDF page numbers, counted from 1.
    for page_num in range(307, 313):
        page = pdf.pages[page_num - 1]

        words = page.extract_words(
            x_tolerance=1,
            y_tolerance=3,
            keep_blank_chars=False
        )

        for word in words:
            records.append({
                "pdf_page": page_num,
                "text": word["text"],
                "x0": word["x0"],
                "x1": word["x1"],
                "top": word["top"],
                "bottom": word["bottom"]
            })

words_df = pd.DataFrame(records)
words_df.to_csv(RAW_CSV, index=False)

print("Words extracted:", len(words_df))
print("Saved to:", RAW_CSV)

display(words_df.head(20))


PDF pages: 350
Words extracted: 7228
Saved to: /content/gdrive/MyDrive/lucknow_census_processed/lucknow_urban_pca_words_307_312.csv


,pdf_page,text,x0,x1,top,bottom
0,307,DISTRICT,346.440000,383.503939,67.648479,75.688379
1,307,CENSUS,385.779231,419.225215,67.648479,75.688379
2,307,HANDBOOK,421.532666,467.472655,67.648479,75.688379
3,307,:,469.780106,472.015198,67.648479,75.688379
4,307,LUCKNOW,474.330689,515.302020,67.648479,75.688379
5,307,URBAN,429.120000,464.597520,84.071360,94.031360
6,307,PRIMARY,467.117400,515.433360,84.071360,94.031360
7,307,Area,161.160000,178.476000,96.584000,105.584000
8,307,in,180.717000,187.728000,96.584000,105.584000
9,307,Total,267.120000,286.047000,97.304000,106.304000


In [ ]:

# STEP 2: Inspect the first town row on each of the six pages

# Approximate vertical position of the first town row, based on
# your earlier diagnostics. We inspect a range to allow small shifts.
for page_num in range(307, 313):
    page_words = words_df[words_df["pdf_page"] == page_num].copy()

    # Keep words in the upper data-table area.
    # The first town row was around y=159 on the sample pages.
    row_words = page_words[
        page_words["top"].between(145, 175)
    ].sort_values(["top", "x0"])

    print(f"\n{'=' * 25} PDF PAGE {page_num} {'=' * 25}")
    display(row_words[["text", "x0", "x1", "top"]].round(1))



========================= PDF PAGE 307 =========================


,text,x0,x1,top
51,Malihabad,64.0,103.4,159.1
52,(NP),105.7,122.2,159.1
50,800946,24.7,51.4,159.2
53,6.00,176.8,192.4,159.2
54,3,224.4,228.9,159.2
55,",032",228.8,244.5,159.2
56,1,265.1,269.6,159.2
57,"7,818",269.5,289.6,159.2
58,9,313.9,318.4,159.2
59,",334",318.4,334.0,159.2



========================= PDF PAGE 308 =========================


,text,x0,x1,top
1181,Malihabad,413.6,453.1,159.1
1182,(NP),455.4,471.9,159.1
1166,1,40.1,44.6,159.2
1167,",547",44.5,60.2,159.2
1168,8,88.2,92.7,159.2
1169,39,92.6,101.6,159.2
1170,7,129.6,134.1,159.2
1171,08,134.0,143.0,159.2
1172,-,177.2,179.9,159.2
1173,-,218.6,221.3,159.2



========================= PDF PAGE 309 =========================


,text,x0,x1,top
2262,Malihabad,64.0,103.4,159.1
2263,(NP),105.7,122.2,159.1
2261,800946,24.7,51.4,159.2
2264,6,171.8,176.3,159.2
2265,",964",176.3,191.9,159.2
2266,3,213.2,217.7,159.2
2267,",297",217.7,233.3,159.2
2268,3,253.9,258.4,159.2
2269,",667",258.4,274.0,159.2
2270,5,298.3,302.8,159.2


... (2 outputs left out)


========================= PDF PAGE 311 =========================


,text,x0,x1,top
4952,Malihabad,64.0,103.4,159.1
4953,(NP),105.7,122.2,159.1
4951,800946,24.7,51.4,159.2
4954,1,171.1,175.6,159.2
4955,",844",175.6,191.2,159.2
4956,1,214.1,218.6,159.2
4957,",487",218.5,234.2,159.2
4958,3,260.0,264.5,159.2
4959,57,264.5,273.4,159.2
4960,4,306.7,311.2,159.2



========================= PDF PAGE 312 =========================


,text,x0,x1,top
6098,Malihabad,398.3,437.7,159.1
6099,(NP),440.0,456.5,159.1
6080,4,42.4,46.8,159.2
6081,14,46.8,55.7,159.2
6082,3,79.3,83.8,159.2
6083,37,83.8,92.7,159.2
6084,7,120.7,125.2,159.2
6085,7,125.2,129.6,159.2
6086,1,157.0,161.4,159.2
6087,",005",161.4,177.0,159.2


In [ ]:

# STEP 4: Create a preliminary consolidated urban PCA CSV
# Important: this is a parsing/validation stage, not yet the final dataset.

import re
import os
import pandas as pd
import numpy as np

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"
RAW_CSV = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_words_307_312.csv"
)

words_df = pd.read_csv(RAW_CSV)

# Column ranges are based on the coordinate diagnostics you supplied.
# Each interval represents one numeric field.
FIELD_BINS = {
    307: [
        ("area_sq_km", 155, 198),
        ("households", 198, 250),
        ("population_total", 250, 300),
        ("population_male", 300, 345),
        ("population_female", 345, 395),
        ("age_0_6_total", 395, 440),
        ("age_0_6_male", 440, 480),
        ("age_0_6_female", 480, 540),
    ],
    308: [
        ("sc_total", 25, 70),
        ("sc_male", 70, 110),
        ("sc_female", 110, 150),
        ("st_total", 150, 195),
        ("st_male", 195, 235),
        ("st_female", 235, 275),
        ("literate_total", 275, 325),
        ("literate_male", 325, 370),
        ("literate_female", 370, 410),
    ],
    309: [
        ("illiterate_total", 160, 205),
        ("illiterate_male", 205, 245),
        ("illiterate_female", 245, 285),
        ("workers_total", 285, 330),
        ("workers_male", 330, 370),
        ("workers_female", 370, 410),
        ("main_workers_total", 410, 455),
        ("main_workers_male", 455, 495),
        ("main_workers_female", 495, 540),
    ],
    310: [
        ("cultivators_total", 25, 65),
        ("cultivators_male", 65, 100),
        ("cultivators_female", 100, 135),
        ("agri_labourers_total", 135, 170),
        ("agri_labourers_male", 170, 205),
        ("agri_labourers_female", 205, 235),
        ("household_industry_main_total", 235, 270),
        ("household_industry_main_male", 270, 305),
        ("household_industry_main_female", 305, 340),
        ("other_main_workers_total", 340, 380),
        ("other_main_workers_male", 380, 420),
        ("other_main_workers_female", 420, 455),
    ],
    311: [
        ("marginal_workers_total", 160, 205),
        ("marginal_workers_male", 205, 245),
        ("marginal_workers_female", 245, 285),
        ("marginal_cultivators_total", 285, 325),
        ("marginal_cultivators_male", 325, 365),
        ("marginal_cultivators_female", 365, 400),
        ("marginal_agri_labourers_total", 400, 440),
        ("marginal_agri_labourers_male", 440, 480),
        ("marginal_agri_labourers_female", 480, 520),
    ],
    312: [
        ("marginal_household_industry_total", 25, 65),
        ("marginal_household_industry_male", 65, 100),
        ("marginal_household_industry_female", 100, 140),
        ("marginal_other_workers_total", 140, 180),
        ("marginal_other_workers_male", 180, 220),
        ("marginal_other_workers_female", 220, 260),
        ("nonworkers_total", 260, 310),
        ("nonworkers_male", 310, 355),
        ("nonworkers_female", 355, 395),
    ],
}

def join_fragments(tokens):
    """Join adjacent PDF fragments, e.g. ['1', ',547'] -> '1,547'."""
    tokens = sorted(tokens, key=lambda z: z["x0"])
    return "".join(str(t["text"]).strip() for t in tokens)

def parse_number(value):
    """Parse a reconstructed numeric value without turning dashes into zero."""
    value = str(value).strip().replace(",", "").replace(" ", "")
    if value in {"-", "—", "–", ""}:
        return value if value else None
    if re.fullmatch(r"\d+(?:\.\d+)?", value):
        return float(value) if "." in value else int(value)
    return value

# Use page 307 as the reference for row identities.
p307 = words_df[words_df["pdf_page"] == 307].copy()
p307["row_y"] = p307["top"].round(1)

# Candidate data rows are those with a 6-digit town code or a 4-digit ward code.
id_words = p307[
    p307["text"].astype(str).str.fullmatch(r"(?:\d{6}|\d{4})", na=False)
    & p307["x0"].between(20, 60)
].copy()

# Keep only the main town/ward data region, excluding page headings.
id_words = id_words[id_words["top"].between(140, 820)].copy()

reference_rows = []
for _, item in id_words.iterrows():
    code = str(item["text"]).zfill(6 if len(str(item["text"])) == 6 else 4)
    y = float(item["top"])

    # Name appears to the right of town codes on page 307.
    name_tokens = p307[
        (p307["top"].sub(y).abs() <= 1.0)
        & (p307["x0"] >= 60)
        & (p307["x0"] < 155)
    ].sort_values("x0")

    name = " ".join(name_tokens["text"].astype(str).tolist())

    reference_rows.append({
        "row_y": y,
        "geo_code": code,
        "geography_type": "town" if len(code) == 6 else "ward",
        "geography_name_raw": name if len(code) == 6 else None,
    })

reference = pd.DataFrame(reference_rows).drop_duplicates(
    subset=["geo_code", "row_y"]
).sort_values("row_y").reset_index(drop=True)

# Extract numeric fields from each page by matching rows on Y-coordinate.
for page_num, bins in FIELD_BINS.items():
    page_words = words_df[words_df["pdf_page"] == page_num].copy()

    for field_name, xmin, xmax in bins:
        values = []

        for _, refrow in reference.iterrows():
            y = refrow["row_y"]

            # Match words close to the reference row's vertical position.
            row_tokens = page_words[
                (page_words["top"].sub(y).abs() <= 1.0)
                & (page_words["x0"] >= xmin)
                & (page_words["x0"] < xmax)
            ]

            # Only combine fragments that appear to be numeric or dashes.
            row_tokens = row_tokens[
                row_tokens["text"].astype(str).str.fullmatch(
                    r"[\d,.\-—–]+", na=False
                )
            ]

            raw_value = join_fragments(
                row_tokens[["text", "x0"]].to_dict("records")
            )
            values.append(parse_number(raw_value))

        reference[field_name] = values

# Save preliminary data and a diagnostic report.
preliminary_path = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_preliminary_307_312.csv"
)
reference.to_csv(preliminary_path, index=False)

diagnostic_path = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_alignment_diagnostic.csv"
)

diagnostic = reference[
    ["row_y", "geo_code", "geography_type", "geography_name_raw",
     "population_total", "population_male", "population_female"]
].copy()

diagnostic["population_sum_matches"] = (
    pd.to_numeric(diagnostic["population_male"], errors="coerce")
    + pd.to_numeric(diagnostic["population_female"], errors="coerce")
    == pd.to_numeric(diagnostic["population_total"], errors="coerce")
)

diagnostic.to_csv(diagnostic_path, index=False)

print("Reference rows:", len(reference))
print("Preliminary CSV:", preliminary_path)
print("Diagnostic CSV:", diagnostic_path)

display(reference.head(15))
display(diagnostic.head(15))


Reference rows: 60
Preliminary CSV: /content/gdrive/MyDrive/lucknow_census_processed/lucknow_urban_pca_preliminary_307_312.csv
Diagnostic CSV: /content/gdrive/MyDrive/lucknow_census_processed/lucknow_urban_pca_alignment_diagnostic.csv


,row_y,geo_code,geography_type,geography_name_raw,area_sq_km,households,population_total,population_male,population_female,age_0_6_total,...,marginal_agri_labourers_female,marginal_household_industry_total,marginal_household_industry_male,marginal_household_industry_female,marginal_other_workers_total,marginal_other_workers_male,marginal_other_workers_female,nonworkers_total,nonworkers_male,nonworkers_female
0,159.208479,800946,town,Malihabad (NP),6.00,3032,17818,9334,8484,2184,...,74,414,337,77,1005,804,201,11932,4268,7664
1,169.408479,0001,ward,None,NaN,256,1588,844,744,198,...,24,22,15,7,159,131,28,1072,403,669
2,179.608479,0002,ward,None,NaN,237,1377,719,658,154,...,3,4,2,2,82,52,30,909,314,595
3,189.808479,0003,ward,None,NaN,290,1654,904,750,180,...,8,82,49,33,57,37,20,1091,442,649
4,200.008479,0004,ward,None,NaN,202,1175,587,588,162,...,-,6,2,4,26,15,11,814,265,549
5,210.208479,0005,ward,None,NaN,261,1714,863,851,236,...,1,27,20,7,198,166,32,1174,395,779
6,220.408479,0006,ward,None,NaN,242,1345,721,624,143,...,2,5,3,2,89,82,7,956,362,594
7,230.608479,0007,ward,None,NaN,258,1653,845,808,216,...,1,34,31,3,17,11,6,1174,398,776
8,240.808479,0008,ward,None,NaN,221,1208,641,567,124,...,-,-,-,-,24,11,13,594,207,387
9,251.008479,0009,ward,None,NaN,183,1103,563,540,128,...,-,3,3,-,18,12,6,804,295,509


,row_y,geo_code,geography_type,geography_name_raw,population_total,population_male,population_female,population_sum_matches
0,159.208479,800946,town,Malihabad (NP),17818,9334,8484,True
1,169.408479,0001,ward,None,1588,844,744,True
2,179.608479,0002,ward,None,1377,719,658,True
3,189.808479,0003,ward,None,1654,904,750,True
4,200.008479,0004,ward,None,1175,587,588,True
5,210.208479,0005,ward,None,1714,863,851,True
6,220.408479,0006,ward,None,1345,721,624,True
7,230.608479,0007,ward,None,1653,845,808,True
8,240.808479,0008,ward,None,1208,641,567,True
9,251.008479,0009,ward,None,1103,563,540,True


In [ ]:

import pandas as pd
import numpy as np
import os

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"
CSV_PATH = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_preliminary_307_312.csv"
)

df = pd.read_csv(CSV_PATH, dtype={"geo_code": str})

# Keep numeric codes intact, including leading zeros in ward codes.
df["geo_code"] = df["geo_code"].astype(str)

towns = df[df["geography_type"] == "town"].copy()
wards = df[df["geography_type"] == "ward"].copy()

print("Town rows:", len(towns))
print("Ward rows:", len(wards))
print("\nTown codes and names:")
display(towns[["geo_code", "geography_name_raw",
               "population_total", "households"]])

# Check ward sums against town totals for key additive fields.
fields_to_check = [
    "households",
    "population_total",
    "population_male",
    "population_female",
    "age_0_6_total",
    "age_0_6_male",
    "age_0_6_female",
    "sc_total",
    "st_total",
    "literate_total",
    "illiterate_total",
    "workers_total",
    "main_workers_total",
    "marginal_workers_total",
    "nonworkers_total",
]

# The PDF lists wards immediately after their town.
# Use row order to group each ward under the preceding town.
df = df.sort_values("row_y").reset_index(drop=True)
df["parent_town_code"] = None
current_town = None

for i, row in df.iterrows():
    if row["geography_type"] == "town":
        current_town = row["geo_code"]
    else:
        df.loc[i, "parent_town_code"] = current_town

ward_rows = df[df["geography_type"] == "ward"].copy()
town_rows = df[df["geography_type"] == "town"].copy()

results = []

for _, town in town_rows.iterrows():
    child = ward_rows[
        ward_rows["parent_town_code"] == town["geo_code"]
    ]

    for field in fields_to_check:
        if field not in df.columns:
            continue

        town_value = pd.to_numeric(
            pd.Series([town[field]]), errors="coerce"
        ).iloc[0]
        ward_values = pd.to_numeric(child[field], errors="coerce")

        # Only compare when every ward has a numeric value.
        if pd.notna(town_value) and ward_values.notna().all():
            ward_sum = ward_values.sum()
            matches = np.isclose(town_value, ward_sum)
        else:
            ward_sum = ward_values.sum(min_count=1)
            matches = None

        results.append({
            "town_code": town["geo_code"],
            "town_name": town["geography_name_raw"],
            "field": field,
            "town_value": town_value,
            "ward_sum": ward_sum,
            "matches": matches,
            "ward_count": len(child),
        })

validation = pd.DataFrame(results)

validation_path = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_town_ward_validation.csv"
)
validation.to_csv(validation_path, index=False)

print("\nValidation results:")
display(validation)

print("\nSaved validation report:", validation_path)
print("\nMatch summary:")
display(validation["matches"].value_counts(dropna=False))


Town rows: 4
Ward rows: 56

Town codes and names:


,geo_code,geography_name_raw,population_total,households
0,800946,Malihabad (NP),17818,3032
14,800947,Mahona (NP),8557,1455
25,800948,Itaunja (NP),7305,1246
36,800949,Bakshi Ka Talab (NP),49166,8728



Validation results:


,town_code,town_name,field,town_value,ward_sum,matches,ward_count
0,800946,Malihabad (NP),households,3032.0,3032.0,True,13
1,800946,Malihabad (NP),population_total,17818.0,17818.0,True,13
2,800946,Malihabad (NP),population_male,9334.0,9334.0,True,13
3,800946,Malihabad (NP),population_female,8484.0,8484.0,True,13
4,800946,Malihabad (NP),age_0_6_total,2184.0,2184.0,True,13
5,800946,Malihabad (NP),age_0_6_male,1111.0,1111.0,True,13
6,800946,Malihabad (NP),age_0_6_female,1073.0,1073.0,True,13
7,800946,Malihabad (NP),sc_total,1547.0,1547.0,True,13
8,800946,Malihabad (NP),st_total,NaN,NaN,None,13
9,800946,Malihabad (NP),literate_total,10854.0,10854.0,True,13



Saved validation report: /content/gdrive/MyDrive/lucknow_census_processed/lucknow_urban_pca_town_ward_validation.csv

Match summary:


,count
matches,
True,52
None,8


In [ ]:
import os
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"

validation_path = os.path.join(
    OUTPUT_DIR,
    "lucknow_urban_pca_town_ward_validation.csv"
)

validation = pd.read_csv(validation_path)

unresolved = validation[validation["matches"].isna()]

print("Unresolved validation checks:", len(unresolved))

display(
    unresolved[
        [
            "town_code",
            "town_name",
            "field",
            "town_value",
            "ward_sum",
            "matches",
            "ward_count"
        ]
    ]
)

Unresolved validation checks: 8


,town_code,town_name,field,town_value,ward_sum,matches,ward_count
8,800946,Malihabad (NP),st_total,NaN,NaN,NaN,13
22,800947,Mahona (NP),sc_total,2360.0,2360.0,NaN,10
23,800947,Mahona (NP),st_total,NaN,NaN,NaN,10
37,800948,Itaunja (NP),sc_total,796.0,796.0,NaN,10
38,800948,Itaunja (NP),st_total,NaN,NaN,NaN,10
52,800949,Bakshi Ka Talab (NP),sc_total,14840.0,14840.0,NaN,23
53,800949,Bakshi Ka Talab (NP),st_total,NaN,NaN,NaN,23
58,800949,Bakshi Ka Talab (NP),marginal_workers_total,4839.0,4839.0,NaN,23


In [ ]:
import os
import numpy as np
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"

INPUT_CSV = os.path.join(
    OUTPUT_DIR,
    "lucknow_urban_pca_preliminary_307_312.csv"
)

df = pd.read_csv(INPUT_CSV)

check_fields = [
    "households",
    "population_total",
    "population_male",
    "population_female",
    "age_0_6_total",
    "age_0_6_male",
    "age_0_6_female",
    "sc_total",
    "st_total",
    "literate_total",
    "illiterate_total",
    "workers_total",
    "main_workers_total",
    "marginal_workers_total",
    "nonworkers_total",
]

results = []

towns = df[df["geography_type"] == "town"]
wards = df[df["geography_type"] == "ward"]

for _, town in towns.iterrows():
    town_code = str(town["geo_code"])

    # Select wards belonging to this town, using the row order
    # between this town's record and the next town's record.
    town_position = df.index.get_loc(town.name)
    next_towns = towns[towns.index > town.name]

    if len(next_towns):
        next_position = df.index.get_loc(next_towns.index[0])
        town_wards = df.iloc[town_position + 1:next_position]
    else:
        town_wards = df.iloc[town_position + 1:]

    town_wards = town_wards[
        town_wards["geography_type"] == "ward"
    ]

    for field in check_fields:
        town_value = pd.to_numeric(
            pd.Series([town.get(field)]), errors="coerce"
        ).iloc[0]

        ward_values = pd.to_numeric(
            town_wards[field], errors="coerce"
        )

        if pd.isna(town_value) and ward_values.isna().all():
            status = "BOTH_MISSING"
            ward_sum = np.nan

        elif pd.isna(town_value) or ward_values.isna().any():
            status = "MISSING_VALUES_REVIEW"
            ward_sum = ward_values.sum(min_count=1)

        else:
            ward_sum = ward_values.sum()

            status = (
                "MATCH"
                if np.isclose(town_value, ward_sum)
                else "MISMATCH"
            )

        results.append({
            "town_code": town_code,
            "town_name": town["geography_name_raw"],
            "field": field,
            "town_value": town_value,
            "ward_sum": ward_sum,
            "ward_count": len(town_wards),
            "status": status,
        })

validation_fixed = pd.DataFrame(results)

validation_path = os.path.join(
    OUTPUT_DIR,
    "lucknow_urban_pca_validation_v2.csv"
)

validation_fixed.to_csv(validation_path, index=False)

print("Validation status summary:")
display(validation_fixed["status"].value_counts().to_frame("count"))

print("\nFields requiring review:")
display(
    validation_fixed[
        validation_fixed["status"].isin(
            ["MISMATCH", "MISSING_VALUES_REVIEW"]
        )
    ]
)

print("\nValidation saved to:", validation_path)

Validation status summary:


,count
status,
MATCH,52
BOTH_MISSING,4
MISSING_VALUES_REVIEW,4



Fields requiring review:


,town_code,town_name,field,town_value,ward_sum,ward_count,status
22,800947,Mahona (NP),sc_total,2360.0,2360.0,10,MISSING_VALUES_REVIEW
37,800948,Itaunja (NP),sc_total,796.0,796.0,10,MISSING_VALUES_REVIEW
52,800949,Bakshi Ka Talab (NP),sc_total,14840.0,14840.0,23,MISSING_VALUES_REVIEW
58,800949,Bakshi Ka Talab (NP),marginal_workers_total,4839.0,4839.0,23,MISSING_VALUES_REVIEW



Validation saved to: /content/gdrive/MyDrive/lucknow_census_processed/lucknow_urban_pca_validation_v2.csv


In [ ]:
import os
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"

INPUT_CSV = os.path.join(
    OUTPUT_DIR,
    "lucknow_urban_pca_preliminary_307_312.csv"
)

df = pd.read_csv(INPUT_CSV)

fields_to_check = [
    ("800947", "sc_total"),
    ("800948", "sc_total"),
    ("800949", "sc_total"),
    ("800949", "marginal_workers_total"),
]

towns = df[df["geography_type"] == "town"]

for town_code, field in fields_to_check:
    town_matches = towns[
        towns["geo_code"].astype(str) == town_code
    ]

    if town_matches.empty:
        print(f"Town code {town_code} not found.")
        continue

    town = town_matches.iloc[0]
    town_position = df.index.get_loc(town.name)

    next_towns = towns[towns.index > town.name]

    if len(next_towns):
        next_position = df.index.get_loc(next_towns.index[0])
        subset = df.iloc[town_position + 1:next_position]
    else:
        subset = df.iloc[town_position + 1:]

    wards = subset[subset["geography_type"] == "ward"].copy()
    wards[field] = pd.to_numeric(wards[field], errors="coerce")

    print(
        f"\n{town['geography_name_raw']} | "
        f"{field} | Town total: {town[field]}"
    )

    display(
        wards.loc[
            wards[field].isna(),
            ["geo_code", "geography_type", field]
        ]
    )


Mahona (NP) | sc_total | Town total: 2360


,geo_code,geography_type,sc_total
24,10,ward,NaN



Itaunja (NP) | sc_total | Town total: 796


,geo_code,geography_type,sc_total
27,2,ward,NaN



Bakshi Ka Talab (NP) | sc_total | Town total: 14840


,geo_code,geography_type,sc_total
57,21,ward,NaN



Bakshi Ka Talab (NP) | marginal_workers_total | Town total: 4839


,geo_code,geography_type,marginal_workers_total
57,21,ward,NaN


In [ ]:
import pdfplumber
import pandas as pd

PDF_PATH = (
    "/content/gdrive/MyDrive/Lucknow_Census_Raw/"
    "DH_2011_0926_PART_B_DCHB_LUCKNOW.pdf"
)

# PDF page 308 contains SC/ST totals.
# PDF page 311 contains marginal-worker totals.
targets = {
    308: ["0002", "0010", "0021"],
    311: ["0002", "0010", "0021"],
}

with pdfplumber.open(PDF_PATH) as pdf:
    for page_num, target_codes in targets.items():
        page = pdf.pages[page_num - 1]
        words = pd.DataFrame(
            page.extract_words(x_tolerance=1, y_tolerance=3)
        )

        words["top"] = words["top"].astype(float)
        words["x0"] = words["x0"].astype(float)

        print(f"\nPDF page {page_num}")

        # Display the identifier and numeric words by row.
        # We will use the output to locate the target rows.
        subset = words[
            words["text"].isin(target_codes)
            | words["text"].str.fullmatch(r"\d[\d,]*|-", na=False)
        ].sort_values(["top", "x0"])

        display(subset[["top", "x0", "text"]].reset_index(drop=True))


PDF page 308


,top,x0,text
0,142.904000,39.600000,11
1,142.904000,81.000000,12
2,142.904000,122.400000,13
3,142.904000,168.240000,14
4,142.904000,209.640000,15
...,...,...,...
896,761.008479,357.360000,3
897,761.008479,361.806065,6
898,761.008479,398.760000,2
899,761.008479,403.206065,8



PDF page 311


,top,x0,text
0,142.904000,40.80,1
1,142.904000,106.68,2
2,142.904000,171.00,41
3,142.904000,212.88,42
4,142.904000,253.92,43
...,...,...,...
963,761.008479,386.76,1
964,761.008479,429.00,1
965,761.008479,466.80,1
966,761.008479,497.52,-


In [ ]:
import pdfplumber
import pandas as pd

PDF_PATH = (
    "/content/gdrive/MyDrive/Lucknow_Census_Raw/"
    "DH_2011_0926_PART_B_DCHB_LUCKNOW.pdf"
)

targets = {
    308: ["0002", "0010", "0021"],
    311: ["0021"],
}

with pdfplumber.open(PDF_PATH) as pdf:
    for page_num, ward_codes in targets.items():
        page = pdf.pages[page_num - 1]

        words = pd.DataFrame(
            page.extract_words(x_tolerance=1, y_tolerance=3)
        )

        words["top"] = words["top"].astype(float)
        words["x0"] = words["x0"].astype(float)

        print(f"\n--- PDF page {page_num} ---")

        for ward_code in ward_codes:
            # Ward identifiers may be fragmented or stored with
            # different leading-zero formats.
            code_rows = words[
                words["text"].astype(str).str.strip().isin(
                    [ward_code, str(int(ward_code))]
                )
                & words["x0"].between(15, 70)
            ]

            if code_rows.empty:
                print(f"Ward {ward_code}: identifier not found in this range.")
                continue

            for _, code_row in code_rows.iterrows():
                y = code_row["top"]

                row_words = words[
                    (words["top"] - y).abs() < 2.0
                ].sort_values("x0")

                print(f"\nWard {ward_code}, row y={y:.2f}")
                display(row_words[["x0", "text"]].reset_index(drop=True))


--- PDF page 308 ---

Ward 0002, row y=189.81


,x0,text
0,46.800000,2
1,51.246065,73
2,88.200000,1
3,92.646065,50
4,129.600000,1
5,134.046065,23
6,177.240000,-
7,218.640000,-
8,260.040000,-
9,307.800000,9



Ward 0002, row y=200.01


,x0,text
0,46.800000,2
1,51.246065,17
2,88.200000,1
3,92.646065,18
4,134.040000,9
5,138.486065,9
6,177.240000,-
7,218.640000,-
8,260.040000,-
9,307.800000,6



Ward 0002, row y=230.61


,x0,text
0,51.240000,6
1,55.686065,2
2,92.640000,2
3,97.086065,7
4,134.040000,3
5,138.486065,5
6,177.240000,-
7,218.640000,-
8,260.040000,-
9,301.080000,1


... (26 outputs left out)


Ward 0002, row y=710.01


,x0,text
0,46.800000,2
1,51.246065,33
2,88.200000,1
3,92.646065,26
4,129.600000,1
5,134.046065,07
6,177.240000,-
7,218.640000,-
8,260.040000,-
9,307.800000,1


Ward 0010: identifier not found in this range.
Ward 0021: identifier not found in this range.

--- PDF page 311 ---

Ward 0021, row y=740.61


,x0,text
0,24.720000,0021
1,63.960000,Ward
2,85.691850,No.
3,100.453106,21
4,179.640000,-
5,222.600000,-
6,261.840000,-
7,304.080000,-
8,341.880000,-
9,379.680000,-


In [ ]:

import os
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"

validation_path = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_validation_v2.csv"
)
pca_path = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_preliminary_307_312.csv"
)

for path in [validation_path, pca_path]:
    print("\n" + "=" * 80)
    print("FILE:", os.path.basename(path))
    print("Exists:", os.path.exists(path))

    if os.path.exists(path):
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:")
        print(df.columns.tolist())
        print("\nFirst 5 rows:")
        display(df.head())
    else:
        print("File not found. Check the path.")



FILE: lucknow_urban_pca_validation_v2.csv
Exists: True
Shape: (60, 7)
Columns:
['town_code', 'town_name', 'field', 'town_value', 'ward_sum', 'ward_count', 'status']

First 5 rows:


,town_code,town_name,field,town_value,ward_sum,ward_count,status
0,800946,Malihabad (NP),households,3032.0,3032.0,13,MATCH
1,800946,Malihabad (NP),population_total,17818.0,17818.0,13,MATCH
2,800946,Malihabad (NP),population_male,9334.0,9334.0,13,MATCH
3,800946,Malihabad (NP),population_female,8484.0,8484.0,13,MATCH
4,800946,Malihabad (NP),age_0_6_total,2184.0,2184.0,13,MATCH



FILE: lucknow_urban_pca_preliminary_307_312.csv
Exists: True
Shape: (60, 60)
Columns:
['row_y', 'geo_code', 'geography_type', 'geography_name_raw', 'area_sq_km', 'households', 'population_total', 'population_male', 'population_female', 'age_0_6_total', 'age_0_6_male', 'age_0_6_female', 'sc_total', 'sc_male', 'sc_female', 'st_total', 'st_male', 'st_female', 'literate_total', 'literate_male', 'literate_female', 'illiterate_total', 'illiterate_male', 'illiterate_female', 'workers_total', 'workers_male', 'workers_female', 'main_workers_total', 'main_workers_male', 'main_workers_female', 'cultivators_total', 'cultivators_male', 'cultivators_female', 'agri_labourers_total', 'agri_labourers_male', 'agri_labourers_female', 'household_industry_main_total', 'household_industry_main_male', 'household_industry_main_female', 'other_main_workers_total', 'other_main_workers_male', 'other_main_workers_female', 'marginal_workers_total', 'marginal_workers_male', 'marginal_workers_female', 'marginal_cul

,row_y,geo_code,geography_type,geography_name_raw,area_sq_km,households,population_total,population_male,population_female,age_0_6_total,...,marginal_agri_labourers_female,marginal_household_industry_total,marginal_household_industry_male,marginal_household_industry_female,marginal_other_workers_total,marginal_other_workers_male,marginal_other_workers_female,nonworkers_total,nonworkers_male,nonworkers_female
0,159.208479,800946,town,Malihabad (NP),6.0,3032,17818,9334,8484,2184,...,74,414,337,77,1005,804,201,11932,4268,7664
1,169.408479,1,ward,NaN,NaN,256,1588,844,744,198,...,24,22,15,7,159,131,28,1072,403,669
2,179.608479,2,ward,NaN,NaN,237,1377,719,658,154,...,3,4,2,2,82,52,30,909,314,595
3,189.808479,3,ward,NaN,NaN,290,1654,904,750,180,...,8,82,49,33,57,37,20,1091,442,649
4,200.008479,4,ward,NaN,NaN,202,1175,587,588,162,...,-,6,2,4,26,15,11,814,265,549


In [ ]:

import os
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"

source_dashes = pd.DataFrame([
    {
        "town": "Mahona (NP)",
        "town_code": "800947",
        "ward": "Ward No. 10",
        "field": "sc_total",
        "source_value": "-",
        "interpretation": "Source-reported dash; do not assume zero"
    },
    {
        "town": "Itaunja (NP)",
        "town_code": "800948",
        "ward": "Ward No. 2",
        "field": "sc_total",
        "source_value": "-",
        "interpretation": "Source-reported dash; do not assume zero"
    },
    {
        "town": "Bakshi Ka Talab (NP)",
        "town_code": "800949",
        "ward": "Ward No. 21",
        "field": "sc_total",
        "source_value": "-",
        "interpretation": "Source-reported dash; do not assume zero"
    },
    {
        "town": "Bakshi Ka Talab (NP)",
        "town_code": "800949",
        "ward": "Ward No. 21",
        "field": "marginal_workers_total",
        "source_value": "-",
        "interpretation": "Source-reported dash; do not assume zero"
    }
])

audit_path = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_source_dash_audit.csv"
)

source_dashes.to_csv(audit_path, index=False)

print("Source-dash audit saved to:")
print(audit_path)
display(source_dashes)



Source-dash audit saved to:
/content/gdrive/MyDrive/lucknow_census_processed/lucknow_urban_pca_source_dash_audit.csv


,town,town_code,ward,field,source_value,interpretation
0,Mahona (NP),800947,Ward No. 10,sc_total,-,Source-reported dash; do not assume zero
1,Itaunja (NP),800948,Ward No. 2,sc_total,-,Source-reported dash; do not assume zero
2,Bakshi Ka Talab (NP),800949,Ward No. 21,sc_total,-,Source-reported dash; do not assume zero
3,Bakshi Ka Talab (NP),800949,Ward No. 21,marginal_workers_total,-,Source-reported dash; do not assume zero


In [ ]:

import os
import numpy as np
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"

INPUT_PATH = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_preliminary_307_312.csv"
)
AUDIT_PATH = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_source_dash_audit.csv"
)

df = pd.read_csv(INPUT_PATH, dtype={"geo_code": str})
dash_audit = pd.read_csv(AUDIT_PATH, dtype=str)

# Keep the original order; each town row starts a new group.
df["town_code"] = pd.NA
df["town_name"] = pd.NA

current_code = None
current_name = None

for idx, row in df.iterrows():
    if row["geography_type"] == "town":
        current_code = str(row["geo_code"]).strip()
        current_name = row["geography_name_raw"]
        df.at[idx, "town_code"] = current_code
        df.at[idx, "town_name"] = current_name
    elif row["geography_type"] == "ward":
        df.at[idx, "town_code"] = current_code
        df.at[idx, "town_name"] = current_name

# Preserve the extracted ward number as a string identifier.
df["ward_code"] = pd.NA
ward_mask = df["geography_type"].eq("ward")
df.loc[ward_mask, "ward_code"] = (
    df.loc[ward_mask, "geo_code"].astype(str).str.zfill(4)
)

# Reconstruct a readable ward label without overwriting source text.
df["ward_name"] = pd.NA
df.loc[ward_mask, "ward_name"] = (
    "Ward No. " +
    pd.to_numeric(df.loc[ward_mask, "geo_code"], errors="coerce")
      .astype("Int64").astype(str)
)

# Convert dash markers to NaN only in numeric measurement columns.
metadata_cols = {
    "row_y", "geo_code", "geography_type", "geography_name_raw",
    "town_code", "town_name", "ward_code", "ward_name"
}
value_cols = [c for c in df.columns if c not in metadata_cols]

dash_counts = {}
for col in value_cols:
    values = df[col].astype("string").str.strip()
    dash_mask = values.isin(["-", "–", "—"])
    count = int(dash_mask.sum())
    if count:
        dash_counts[col] = count
    df[col] = pd.to_numeric(values.mask(dash_mask), errors="coerce")

# Save cleaned copy.
clean_path = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_cleaned_307_312.csv"
)
df.to_csv(clean_path, index=False)

# Report parent mapping and dash conversions.
mapping = df.loc[
    :, ["geography_type", "geo_code", "geography_name_raw",
        "town_code", "town_name", "ward_code", "ward_name"]
]

mapping_path = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_town_ward_mapping.csv"
)
mapping.to_csv(mapping_path, index=False)

print("Cleaned file:", clean_path)
print("Mapping file:", mapping_path)
print("\nRows by geography type:")
print(df["geography_type"].value_counts(dropna=False))

print("\nTown/ward mapping:")
display(mapping.head(18))

print("\nDash markers converted to missing values by column:")
display(pd.DataFrame(
    list(dash_counts.items()),
    columns=["column", "dash_count"]
))


Cleaned file: /content/gdrive/MyDrive/lucknow_census_processed/lucknow_urban_pca_cleaned_307_312.csv
Mapping file: /content/gdrive/MyDrive/lucknow_census_processed/lucknow_urban_pca_town_ward_mapping.csv

Rows by geography type:
geography_type
ward    56
town     4
Name: count, dtype: int64

Town/ward mapping:


,geography_type,geo_code,geography_name_raw,town_code,town_name,ward_code,ward_name
0,town,800946,Malihabad (NP),800946,Malihabad (NP),<NA>,<NA>
1,ward,0001,NaN,800946,Malihabad (NP),0001,Ward No. 1
2,ward,0002,NaN,800946,Malihabad (NP),0002,Ward No. 2
3,ward,0003,NaN,800946,Malihabad (NP),0003,Ward No. 3
4,ward,0004,NaN,800946,Malihabad (NP),0004,Ward No. 4
5,ward,0005,NaN,800946,Malihabad (NP),0005,Ward No. 5
6,ward,0006,NaN,800946,Malihabad (NP),0006,Ward No. 6
7,ward,0007,NaN,800946,Malihabad (NP),0007,Ward No. 7
8,ward,0008,NaN,800946,Malihabad (NP),0008,Ward No. 8
9,ward,0009,NaN,800946,Malihabad (NP),0009,Ward No. 9



Dash markers converted to missing values by column:


,column,dash_count
0,sc_total,3
1,sc_male,3
2,sc_female,4
3,st_total,60
4,st_male,60
5,st_female,60
6,cultivators_total,2
7,cultivators_male,2
8,cultivators_female,25
9,agri_labourers_total,3


In [ ]:

import os
import numpy as np
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"

CLEAN_PATH = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_cleaned_307_312.csv"
)
OLD_VALIDATION_PATH = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_validation_v2.csv"
)
AUDIT_PATH = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_source_dash_audit.csv"
)

df = pd.read_csv(
    CLEAN_PATH,
    dtype={"town_code": str, "geo_code": str, "ward_code": str}
)
old_validation = pd.read_csv(
    OLD_VALIDATION_PATH,
    dtype={"town_code": str}
)
dash_audit = pd.read_csv(AUDIT_PATH, dtype=str)

# Map audit field names to town codes and ward numbers.
audit_keys = set()
for _, r in dash_audit.iterrows():
    ward_num = "".join(ch for ch in str(r["ward"]) if ch.isdigit())
    audit_keys.add((
        str(r["town_code"]).strip(),
        ward_num.zfill(4),
        str(r["field"]).strip()
    ))

results = []

for _, check in old_validation.iterrows():
    town_code = str(check["town_code"]).strip()
    field = check["field"]

    town_rows = df[
        (df["geography_type"] == "town") &
        (df["town_code"] == town_code)
    ]
    ward_rows = df[
        (df["geography_type"] == "ward") &
        (df["town_code"] == town_code)
    ]

    if town_rows.empty or field not in df.columns:
        results.append({
            **check.to_dict(),
            "ward_sum_recalculated": np.nan,
            "ward_count_recalculated": len(ward_rows),
            "source_dash_count": np.nan,
            "status_recalculated": "REVIEW"
        })
        continue

    town_value = pd.to_numeric(
        town_rows.iloc[0][field], errors="coerce"
    )
    ward_values = pd.to_numeric(ward_rows[field], errors="coerce")
    ward_sum = ward_values.sum(min_count=1)
    missing_count = int(ward_values.isna().sum())

    source_dash_count = sum(
        (town_code, str(w).zfill(4), field) in audit_keys
        for w in ward_rows["ward_code"].dropna()
    )

    if pd.isna(town_value) and ward_values.isna().all():
        status = "BOTH_MISSING"
    elif source_dash_count > 0 and missing_count > 0:
        status = "SOURCE_DASH"
    elif missing_count > 0:
        status = "MISSING_VALUES_REVIEW"
    elif pd.isna(town_value):
        status = "TOWN_VALUE_MISSING"
    elif np.isclose(float(town_value), float(ward_sum), atol=0.01):
        status = "MATCH"
    else:
        status = "MISMATCH"

    results.append({
        **check.to_dict(),
        "ward_sum_recalculated": ward_sum,
        "ward_count_recalculated": len(ward_rows),
        "source_dash_count": source_dash_count,
        "status_recalculated": status
    })

validation = pd.DataFrame(results)

out_path = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_validation_v3.csv"
)
validation.to_csv(out_path, index=False)

print("Saved:", out_path)
print("\nValidation status summary:")
print(validation["status_recalculated"].value_counts(dropna=False))

print("\nAnything needing review:")
review = validation[
    validation["status_recalculated"].isin([
        "MISMATCH", "MISSING_VALUES_REVIEW",
        "TOWN_VALUE_MISSING", "REVIEW"
    ])
]
display(review)

print("\nSource-dash checks:")
display(validation[
    validation["status_recalculated"] == "SOURCE_DASH"
][[
    "town_code", "town_name", "field",
    "town_value", "ward_sum_recalculated",
    "source_dash_count", "status_recalculated"
]])


Saved: /content/gdrive/MyDrive/lucknow_census_processed/lucknow_urban_pca_validation_v3.csv

Validation status summary:
status_recalculated
MATCH           52
BOTH_MISSING     4
SOURCE_DASH      4
Name: count, dtype: int64

Anything needing review:


,town_code,town_name,field,town_value,ward_sum,ward_count,status,ward_sum_recalculated,ward_count_recalculated,source_dash_count,status_recalculated



Source-dash checks:


,town_code,town_name,field,town_value,ward_sum_recalculated,source_dash_count,status_recalculated
22,800947,Mahona (NP),sc_total,2360.0,2360.0,1,SOURCE_DASH
37,800948,Itaunja (NP),sc_total,796.0,796.0,1,SOURCE_DASH
52,800949,Bakshi Ka Talab (NP),sc_total,14840.0,14840.0,1,SOURCE_DASH
58,800949,Bakshi Ka Talab (NP),marginal_workers_total,4839.0,4839.0,1,SOURCE_DASH


In [ ]:

!pip -q install pdfplumber

import os
import pdfplumber
import pandas as pd

PDF_PATH = (
    "/content/gdrive/MyDrive/Lucknow_Census_Raw/"
    "DH_2011_0926_PART_B_DCHB_LUCKNOW.pdf"
)

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

records = []

with pdfplumber.open(PDF_PATH) as pdf:
    print("Total PDF pages:", len(pdf.pages))

    # PDF viewer pages 313–318 correspond to indices 312–317.
    for page_number in range(313, 319):
        page = pdf.pages[page_number - 1]

        words = page.extract_words(
            x_tolerance=1,
            y_tolerance=3,
            keep_blank_chars=False
        )

        for word in words:
            records.append({
                "pdf_page": page_number,
                "x0": word["x0"],
                "x1": word["x1"],
                "top": word["top"],
                "bottom": word["bottom"],
                "text": word["text"]
            })

words_df = pd.DataFrame(records)

output_path = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_words_313_318.csv"
)
words_df.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Extracted word records:", len(words_df))

for page_number in range(313, 319):
    page_words = words_df[words_df["pdf_page"] == page_number]

    print(f"\n--- PDF page {page_number} ---")
    print("Word count:", len(page_words))

    # Print the upper portion of the page to inspect headings and IDs.
    sample = page_words[
        page_words["top"].between(100, 260)
    ].sort_values(["top", "x0"])

    print(" ".join(sample["text"].astype(str).tolist())[:3500])


Total PDF pages: 350
Saved: /content/gdrive/MyDrive/lucknow_census_processed/lucknow_urban_pca_words_313_318.csv
Extracted word records: 7839

--- PDF page 313 ---
Word count: 1132
Location Square institutional and houseless code Kilometr Number of population) Population in the age-group 0-6 number Name of Town/ Ward e households Persons Males Females Persons Males Females 1 2 3 4 5 6 7 8 9 10 Kakori (NP) 800950 9.05 3 ,409 1 9,403 1 0,157 9 ,246 2 ,618 1 ,350 1 ,268 Ward No. 1 0001 3 02 1 ,644 8 52 7 92 2 59 1 33 1 26 Ward No. 2 0002 2 65 1 ,508 7 99 7 09 1 85 9 2 9 3 Ward No. 3 0003 2 62 1 ,433 7 39 6 94 1 87 9 2 9 5 Ward No. 4 0004 3 01 1 ,882 9 91 8 91 3 00 1 56 1 44 Ward No. 5 0005 2 31 1 ,268 6 40 6 28 1 47 6 4 8 3 Ward No. 6 0006 2 64 1 ,451 7 83 6 68 1 80 1 07 7 3 Ward No. 7 0007 2 30 1 ,342 6 92 6 50 1 99 9 7 1 02 Ward No. 8 0008 2 04 1 ,084 5 93 4 91 9 8 5 3 4 5 Ward No. 9 0009 2 76 1 ,475 7 74 7 01 1 86 8 6 1 00

--- PDF page 314 ---
Word count: 1208
Scheduled Castes populat

In [ ]:

import os
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"
WORDS_PATH = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_words_313_318.csv"
)

words = pd.read_csv(WORDS_PATH)

# Group tokens into visual rows using their vertical positions.
# Small differences in top coordinates are normal in PDF extraction.
def group_words_into_rows(page_df, tolerance=2.5):
    page_df = page_df.sort_values(["top", "x0"]).copy()
    rows = []
    current = []
    current_y = None

    for _, word in page_df.iterrows():
        y = float(word["top"])

        if current_y is None or abs(y - current_y) <= tolerance:
            current.append(word)
            if current_y is None:
                current_y = y
            else:
                current_y = sum(float(w["top"]) for w in current) / len(current)
        else:
            rows.append(current)
            current = [word]
            current_y = y

    if current:
        rows.append(current)

    output = []
    for row in rows:
        row = sorted(row, key=lambda w: float(w["x0"]))
        output.append({
            "row_y": round(sum(float(w["top"]) for w in row) / len(row), 2),
            "text": " ".join(str(w["text"]) for w in row)
        })

    return pd.DataFrame(output)

all_rows = []

for page_no in range(313, 319):
    page_df = words[words["pdf_page"] == page_no]
    rows = group_words_into_rows(page_df)

    # Keep likely table content, excluding the very top running header.
    rows = rows[rows["row_y"].between(100, 820)].copy()
    rows.insert(0, "pdf_page", page_no)
    all_rows.append(rows)

    print(f"\n{'=' * 30} PDF PAGE {page_no} {'=' * 30}")
    for _, r in rows.iterrows():
        print(f"{r['row_y']:7.2f} | {r['text']}")

rows_df = pd.concat(all_rows, ignore_index=True)
rows_path = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_rows_313_318_review.csv"
)
rows_df.to_csv(rows_path, index=False)

print("\nSaved row review to:", rows_path)



============================== PDF PAGE 313 ==============================
 108.54 | Location Square institutional and houseless
 120.06 | code Kilometr Number of population) Population in the age-group 0-6
 131.20 | number Name of Town/ Ward e households Persons Males Females Persons Males Females
 142.90 | 1 2 3 4 5 6 7 8 9 10
 159.20 | 800950 Kakori (NP) 9.05 3 ,409 1 9,403 1 0,157 9 ,246 2 ,618 1 ,350 1 ,268
 169.37 | 0001 Ward No. 1 3 02 1 ,644 8 52 7 92 2 59 1 33 1 26
 179.57 | 0002 Ward No. 2 2 65 1 ,508 7 99 7 09 1 85 9 2 9 3
 189.77 | 0003 Ward No. 3 2 62 1 ,433 7 39 6 94 1 87 9 2 9 5
 199.97 | 0004 Ward No. 4 3 01 1 ,882 9 91 8 91 3 00 1 56 1 44
 210.17 | 0005 Ward No. 5 2 31 1 ,268 6 40 6 28 1 47 6 4 8 3
 220.37 | 0006 Ward No. 6 2 64 1 ,451 7 83 6 68 1 80 1 07 7 3
 230.57 | 0007 Ward No. 7 2 30 1 ,342 6 92 6 50 1 99 9 7 1 02
 240.77 | 0008 Ward No. 8 2 04 1 ,084 5 93 4 91 9 8 5 3 4 5
 250.97 | 0009 Ward No. 9 2 76 1 ,475 7 74 7 01 1 86 8 6 1 00
 261.17 | 0010 Ward No. 10 2

In [ ]:

import pandas as pd

# Page 313 values read from the source row review.
checks = pd.DataFrame([
    {
        "town": "Kakori (NP)",
        "town_population": 19403,
        "ward_populations": [
            1644, 1508, 1433, 1882, 1268, 1451, 1342,
            1084, 1475, 1517, 1953, 1258, 1588
        ]
    },
    {
        "town": "Lucknow (M Corp.)",
        "town_population": 2817105,
        "ward_populations": [
            43282, 41971, 17988, 22717, 21960, 38910, 18999,
            25553, 50574, 38053, 54513, 55227, 19668, 19946,
            46057, 55566, 16072, 23292, 23588, 19054, 68993,
            27010, 19263, 35400, 35312, 19965, 18326, 25218,
            22318, 53630, 11065, 52328, 17224, 15832, 31756,
            26795, 21207, 20544, 14829, 31412, 34903, 16816,
            24961, 29270, 29036
        ]
    }
])

for _, row in checks.iterrows():
    ward_sum = sum(row["ward_populations"])
    print(f"\n{row['town']}")
    print("Town population:", row["town_population"])
    print("Ward sum:", ward_sum)
    print("Difference:", ward_sum - row["town_population"])
    print("Status:", "MATCH" if ward_sum == row["town_population"] else "MISMATCH")



Kakori (NP)
Town population: 19403
Ward sum: 19403
Difference: 0
Status: MATCH

Lucknow (M Corp.)
Town population: 2817105
Ward sum: 1356403
Difference: -1460702
Status: MISMATCH


In [ ]:

import os
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"
INPUT_PATH = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_cleaned_307_312.csv"
)

df = pd.read_csv(
    INPUT_PATH,
    dtype={"town_code": str, "geo_code": str, "ward_code": str}
)

# The current cleaned file contains the first block only.
lucknow = df[df["town_code"] == "800951"].copy()

if lucknow.empty:
    print("Lucknow (M Corp.) is not in the current cleaned CSV.")
    print("That is expected if the CSV only contains PDF pages 307–312.")
else:
    print("Rows found:", len(lucknow))
    display(lucknow[
        ["geography_type", "geo_code", "town_name",
         "ward_name", "population_total"]
    ])

    town_rows = lucknow[lucknow["geography_type"] == "town"]
    ward_rows = lucknow[lucknow["geography_type"] == "ward"]

    if not town_rows.empty:
        town_population = pd.to_numeric(
            town_rows.iloc[0]["population_total"], errors="coerce"
        )
        ward_sum = pd.to_numeric(
            ward_rows["population_total"], errors="coerce"
        ).sum(min_count=1)

        print("\nTown population:", town_population)
        print("Ward population sum:", ward_sum)
        print("Ward count:", len(ward_rows))
        print("Difference:", ward_sum - town_population)


Lucknow (M Corp.) is not in the current cleaned CSV.
That is expected if the CSV only contains PDF pages 307–312.


In [ ]:

import os
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"

path = os.path.join(
    OUTPUT_DIR,
    "lucknow_urban_pca_words_313_318.csv"
)

words = pd.read_csv(path)

print("Shape:", words.shape)
print("\nColumns:")
print(words.columns.tolist())

display(words.head(15))


Shape: (7839, 6)

Columns:
['pdf_page', 'x0', 'x1', 'top', 'bottom', 'text']


,pdf_page,x0,x1,top,bottom,text
0,313,346.440000,383.503939,67.648479,75.688379,DISTRICT
1,313,385.779231,419.225215,67.648479,75.688379,CENSUS
2,313,421.532666,467.472655,67.648479,75.688379,HANDBOOK
3,313,469.780106,472.015198,67.648479,75.688379,:
4,313,474.330689,515.302020,67.648479,75.688379,LUCKNOW
5,313,429.120000,464.597520,84.071360,94.031360,URBAN
6,313,467.117400,515.433360,84.071360,94.031360,PRIMARY
7,313,161.160000,178.476000,96.584000,105.584000,Area
8,313,180.717000,187.728000,96.584000,105.584000,in
9,313,267.120000,286.047000,97.304000,106.304000,Total


In [ ]:

import os
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"

INPUT_PATH = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_words_313_318.csv"
)

words = pd.read_csv(INPUT_PATH)

# Sort words in reading order.
words = words.sort_values(
    ["pdf_page", "top", "x0"]
).reset_index(drop=True)

# Group words that are on approximately the same horizontal line.
# The tolerance may need adjustment after reviewing the output.
Y_TOLERANCE = 2.5

reconstructed = []

for page, page_df in words.groupby("pdf_page", sort=True):
    current_line = []
    current_y = None

    for _, word in page_df.iterrows():
        y = float(word["top"])

        if current_y is None or abs(y - current_y) <= Y_TOLERANCE:
            current_line.append(word)
            if current_y is None:
                current_y = y
            else:
                current_y = sum(
                    float(w["top"]) for w in current_line
                ) / len(current_line)
        else:
            if current_line:
                current_line = sorted(
                    current_line, key=lambda w: float(w["x0"])
                )
                reconstructed.append({
                    "pdf_page": int(page),
                    "top": round(
                        sum(float(w["top"]) for w in current_line)
                        / len(current_line), 2
                    ),
                    "line_text": " ".join(
                        str(w["text"]) for w in current_line
                    )
                })

            current_line = [word]
            current_y = y

    if current_line:
        current_line = sorted(
            current_line, key=lambda w: float(w["x0"])
        )
        reconstructed.append({
            "pdf_page": int(page),
            "top": round(
                sum(float(w["top"]) for w in current_line)
                / len(current_line), 2
            ),
            "line_text": " ".join(
                str(w["text"]) for w in current_line
            )
        })

lines = pd.DataFrame(reconstructed)

output_path = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_reconstructed_lines_313_318.csv"
)
lines.to_csv(output_path, index=False)

print("Reconstructed lines:", len(lines))
print("Saved to:", output_path)

# Show the lines containing town names, ward labels, and likely totals.
keywords = (
    "Kakori", "Lucknow", "Ward No.", "Ward", "Total",
    "Population", "800950", "800951"
)

mask = lines["line_text"].str.contains(
    "|".join(keywords), case=False, na=False
)

display(lines[mask].head(100))


Reconstructed lines: 405
Saved to: /content/gdrive/MyDrive/lucknow_census_processed/lucknow_urban_pca_reconstructed_lines_313_318.csv


,pdf_page,top,line_text
0,313,67.65,DISTRICT CENSUS HANDBOOK : LUCKNOW
2,313,97.02,Area in Total population (including
4,313,120.06,code Kilometr Number of population) Population...
5,313,131.20,number Name of Town/ Ward e households Persons...
7,313,159.20,"800950 Kakori (NP) 9.05 3 ,409 1 9,403 1 0,157..."
...,...,...,...
102,314,454.97,"7 ,258 3 ,849 3 ,409 8 34 4 51 3 83 3 4,165 1 ..."
103,314,465.17,"8 ,470 4 ,495 3 ,975 2 3 1 2 1 1 3 3,148 1 9,0..."
104,314,475.37,"2 ,855 1 ,460 1 ,395 9 4 5 1 1,722 6 ,391 5 ,3..."
105,314,485.57,"3 ,020 1 ,544 1 ,476 9 5 4 1 9,588 1 0,219 9 ,..."


In [ ]:

import os
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"

words = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "lucknow_urban_pca_words_313_318.csv"
    )
)

# Inspect the table body on page 313.
page = words[words["pdf_page"] == 313].copy()
page = page[
    (page["top"] >= 150) &
    (page["top"] <= 240)
].sort_values(["top", "x0"])

# Group words into approximate horizontal rows.
page["row_group"] = (page["top"] / 2.5).round().astype(int)

for row_id, group in page.groupby("row_group", sort=True):
    group = group.sort_values("x0")
    y = group["top"].mean()

    tokens = [
        f"x={r.x0:.1f}: {r.text}"
        for r in group.itertuples()
    ]

    print(f"\ny={y:.1f}")
    print(" | ".join(tokens))



y=159.2
x=24.7: 800950 | x=64.0: Kakori | x=91.2: (NP) | x=176.8: 9.05 | x=224.4: 3 | x=228.8: ,409 | x=265.1: 1 | x=269.5: 9,403 | x=309.5: 1 | x=313.9: 0,157 | x=360.6: 9 | x=365.0: ,246 | x=410.2: 2 | x=414.6: ,618 | x=451.6: 1 | x=456.0: ,350 | x=493.0: 1 | x=497.4: ,268

y=169.4
x=24.7: 0001 | x=64.0: Ward | x=85.7: No. | x=100.5: 1 | x=231.1: 3 | x=235.6: 02 | x=269.5: 1 | x=274.0: ,644 | x=320.6: 8 | x=325.1: 52 | x=367.3: 7 | x=371.8: 92 | x=416.9: 2 | x=421.3: 59 | x=458.3: 1 | x=462.7: 33 | x=499.7: 1 | x=504.1: 26

y=179.6
x=24.7: 0002 | x=64.0: Ward | x=85.7: No. | x=100.5: 2 | x=231.1: 2 | x=235.6: 65 | x=269.5: 1 | x=274.0: ,508 | x=320.6: 7 | x=325.1: 99 | x=367.3: 7 | x=371.8: 09 | x=416.9: 1 | x=421.3: 85 | x=462.7: 9 | x=467.2: 2 | x=504.1: 9 | x=508.6: 3

y=189.8
x=24.7: 0003 | x=64.0: Ward | x=85.7: No. | x=100.5: 3 | x=231.1: 2 | x=235.6: 62 | x=269.5: 1 | x=274.0: ,433 | x=320.6: 7 | x=325.1: 39 | x=367.3: 6 | x=371.8: 94 | x=416.9: 1 | x=421.3: 87 | x=462.7: 9 |

In [ ]:

import os
import re
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"

words = pd.read_csv(
    os.path.join(OUTPUT_DIR, "lucknow_urban_pca_words_313_318.csv")
)

page = words[words["pdf_page"] == 313].copy()
page = page[(page["top"] >= 150) & (page["top"] <= 310)]

# Cluster words by their vertical position.
page = page.sort_values(["top", "x0"]).reset_index(drop=True)

rows = []
for _, word in page.iterrows():
    y = float(word["top"])

    # Attach to the closest existing row if within tolerance.
    candidates = [
        i for i, row in enumerate(rows)
        if abs(row["mean_y"] - y) <= 2.0
    ]

    if candidates:
        i = min(candidates, key=lambda j: abs(rows[j]["mean_y"] - y))
        rows[i]["words"].append(word)
        rows[i]["mean_y"] = sum(
            float(w["top"]) for w in rows[i]["words"]
        ) / len(rows[i]["words"])
    else:
        rows.append({"mean_y": y, "words": [word]})

# Approximate column bands based on the coordinates you shared.
columns = {
    "code": (20, 55),
    "name": (55, 150),
    "area_sq_km": (150, 215),
    "households": (215, 260),
    "population_total": (260, 305),
    "population_male": (305, 355),
    "population_female": (355, 405),
    "age_0_6_total": (405, 450),
    "age_0_6_male": (450, 490),
    "age_0_6_female": (490, 535),
}

def join_tokens(tokens):
    # Preserve punctuation while repairing spaces introduced inside numbers.
    text = " ".join(str(t) for t in tokens).strip()
    text = re.sub(r"(?<=\d)\s+(?=[,\d])", "", text)
    text = re.sub(r"(?<=,)\s+(?=\d)", "", text)
    return text

parsed = []

for row in sorted(rows, key=lambda r: r["mean_y"]):
    record = {"pdf_page": 313, "top": round(row["mean_y"], 2)}

    for col_name, (left, right) in columns.items():
        tokens = [
            w for w in row["words"]
            if left <= float(w["x0"]) < right
        ]
        tokens = sorted(tokens, key=lambda w: float(w["x0"]))
        record[col_name] = join_tokens([w["text"] for w in tokens])

    # Keep rows that look like town/ward data, not table headings.
    if (
        re.fullmatch(r"\d{4,6}", record["code"])
        or record["name"].lower().startswith("ward")
    ):
        parsed.append(record)

result = pd.DataFrame(parsed)
out_path = os.path.join(
    OUTPUT_DIR, "lucknow_page313_population_column_test.csv"
)
result.to_csv(out_path, index=False)

print("Saved:", out_path)
display(result.head(20))


Saved: /content/gdrive/MyDrive/lucknow_census_processed/lucknow_page313_population_column_test.csv


,pdf_page,top,code,name,area_sq_km,households,population_total,population_male,population_female,age_0_6_total,age_0_6_male,age_0_6_female
0,313,159.20,800950,Kakori (NP),9.05,"3,409","19,403","10,157","9,246","2,618","1,350","1,268"
1,313,169.37,0001,Ward No. 1,,302,"1,644",852,792,259,133,126
2,313,179.57,0002,Ward No. 2,,265,"1,508",799,709,185,92,93
3,313,189.77,0003,Ward No. 3,,262,"1,433",739,694,187,92,95
4,313,199.97,0004,Ward No. 4,,301,"1,882",991,891,300,156,144
5,313,210.17,0005,Ward No. 5,,231,"1,268",640,628,147,64,83
6,313,220.37,0006,Ward No. 6,,264,"1,451",783,668,180,107,73
7,313,230.57,0007,Ward No. 7,,230,"1,342",692,650,199,97,102
8,313,240.77,0008,Ward No. 8,,204,"1,084",593,491,98,53,45
9,313,250.97,0009,Ward No. 9,,276,"1,475",774,701,186,86,100


In [ ]:

import os
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"

words = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "lucknow_urban_pca_words_313_318.csv"
    )
)

page = words[words["pdf_page"] == 313].copy()

# Print the table headers.
print("=== TABLE HEADERS ===")
headers = page[(page["top"] >= 95) & (page["top"] <= 145)]
for y, group in headers.groupby(
    headers["top"].round(0), sort=True
):
    group = group.sort_values("x0")
    print(f"\ny≈{y}")
    print(" | ".join(
        f"x={r.x0:.1f}: {r.text}"
        for r in group.itertuples()
    ))

# Print the Lucknow town row.
# Its baseline is approximately y=302 in the current extraction.
print("\n=== LUCKNOW TOWN ROW ===")
lucknow = page[
    (page["top"] >= 298) & (page["top"] <= 306)
].sort_values("x0")

for r in lucknow.itertuples():
    print(f"x0={r.x0:7.2f}  x1={r.x1:7.2f}  text={r.text}")


=== TABLE HEADERS ===

y≈97.0
x=161.2: Area | x=180.7: in | x=267.1: Total | x=288.3: population | x=329.4: (including

y≈108.0
x=26.8: Location | x=162.0: Square

y≈109.0
x=269.2: institutional | x=314.7: and | x=330.0: houseless

y≈120.0
x=34.3: code | x=158.3: Kilometr | x=202.0: Number | x=233.6: of | x=296.0: population) | x=397.7: Population | x=439.4: in | x=448.7: the | x=462.0: age-group | x=500.8: 0-6

y≈131.0
x=29.0: number | x=69.2: Name | x=92.8: of | x=102.5: Town/ | x=128.1: Ward | x=172.6: e | x=201.1: households | x=257.6: Persons | x=305.5: Males | x=347.2: Females | x=400.2: Persons | x=444.7: Males | x=482.2: Females

y≈143.0
x=40.8: 1 | x=106.7: 2 | x=172.6: 3 | x=219.6: 4 | x=269.5: 5 | x=314.3: 6 | x=359.8: 7 | x=412.0: 8 | x=453.4: 9 | x=492.5: 10

=== LUCKNOW TOWN ROW ===
x0=  24.72  x1=  51.42  text=800951
x0=  63.96  x1=  98.76  text=Lucknow
x0= 101.14  x1= 110.48  text=(M
x0= 112.89  x1= 136.56  text=Corp.)
x0= 167.88  x1= 192.41  text=348.80
x0= 213.24  x1=

In [ ]:

import os
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"

words = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "lucknow_urban_pca_words_313_318.csv"
    )
)

page = words[words["pdf_page"] == 313].copy()
lucknow = page[
    (page["top"] >= 298) & (page["top"] <= 306)
].sort_values("x0").copy()

# Column centers derived from the table's numbered header.
column_centers = {
    "area_sq_km": 180,
    "households": 220,
    "population_total": 270,
    "population_male": 315,
    "population_female": 360,
    "age_0_6_total": 415,
    "age_0_6_male": 455,
    "age_0_6_female": 495,
}

# Assign each token to the nearest column center.
def nearest_column(x):
    return min(
        column_centers,
        key=lambda col: abs(x - column_centers[col])
    )

lucknow["column"] = lucknow["x0"].apply(nearest_column)

# Join tokens assigned to the same column, in reading order.
values = {}
for col, group in lucknow.groupby("column", sort=False):
    tokens = group.sort_values("x0")["text"].astype(str).tolist()
    values[col] = "".join(tokens)

print("Reconstructed Lucknow town row:")
for col in column_centers:
    print(f"{col}: {values.get(col, '')}")


Reconstructed Lucknow town row:
area_sq_km: 800951Lucknow(MCorp.)348.80
households: 5,38,149
population_total: 28,17,105
population_male: 14,60,970
population_female: 13,56,135
age_0_6_total: 2,93,697
age_0_6_male: 1,54,226
age_0_6_female: 1,39,471


In [ ]:

import os
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"

words = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "lucknow_urban_pca_words_313_318.csv"
    )
)

page = words[words["pdf_page"] == 313].copy()

# Isolate the Lucknow town row.
row = page[
    (page["top"] >= 298) &
    (page["top"] <= 306)
].sort_values("x0")

# Use the table's numeric columns only.
row = row[row["x0"] >= 150].copy()

columns = {
    "area_sq_km": (150, 200),
    "households": (200, 250),
    "population_total": (250, 295),
    "population_male": (295, 340),
    "population_female": (340, 390),
    "age_0_6_total": (390, 435),
    "age_0_6_male": (435, 480),
    "age_0_6_female": (480, 530),
}

values = {}

for name, (left, right) in columns.items():
    tokens = row[
        (row["x0"] >= left) &
        (row["x0"] < right)
    ].sort_values("x0")["text"].astype(str).tolist()

    values[name] = "".join(tokens)

print("Lucknow (M Corp.) — reconstructed values")
for name, value in values.items():
    print(f"{name}: {value}")

# Check the arithmetic using the reconstructed values.
def number(value):
    return int(value.replace(",", ""))

population_ok = (
    number(values["population_male"]) +
    number(values["population_female"]) ==
    number(values["population_total"])
)

age_ok = (
    number(values["age_0_6_male"]) +
    number(values["age_0_6_female"]) ==
    number(values["age_0_6_total"])
)

print("\nPopulation male + female matches:", population_ok)
print("Age 0–6 male + female matches:", age_ok)


Lucknow (M Corp.) — reconstructed values
area_sq_km: 348.80
households: 5,38,149
population_total: 28,17,105
population_male: 14,60,970
population_female: 13,56,135
age_0_6_total: 2,93,697
age_0_6_male: 1,54,226
age_0_6_female: 1,39,471

Population male + female matches: True
Age 0–6 male + female matches: True


In [ ]:

import os
import re
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"

words = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "lucknow_urban_pca_words_313_318.csv"
    )
)

page = words[words["pdf_page"] == 313].copy()

# Column ranges verified against the page-313 header.
columns = {
    "households": (200, 250),
    "population_total": (250, 295),
    "population_male": (295, 340),
    "population_female": (340, 390),
    "age_0_6_total": (390, 435),
    "age_0_6_male": (435, 480),
    "age_0_6_female": (480, 530),
}

def parse_row(row_words):
    result = {}
    row_words = row_words.sort_values("x0")

    for field, (left, right) in columns.items():
        tokens = row_words[
            (row_words["x0"] >= left) &
            (row_words["x0"] < right)
        ].sort_values("x0")["text"].astype(str).tolist()

        value = "".join(tokens)
        result[field] = value

    return result

# Group words by their vertical row position.
body = page[(page["top"] >= 150) & (page["top"] <= 620)].copy()
body = body.sort_values(["top", "x0"]).reset_index(drop=True)

row_groups = []
for _, word in body.iterrows():
    y = float(word["top"])
    matches = [
        i for i, group in enumerate(row_groups)
        if abs(group["mean_y"] - y) <= 2.0
    ]

    if matches:
        i = min(matches, key=lambda j: abs(row_groups[j]["mean_y"] - y))
        row_groups[i]["words"].append(word)
        row_groups[i]["mean_y"] = sum(
            float(w["top"]) for w in row_groups[i]["words"]
        ) / len(row_groups[i]["words"])
    else:
        row_groups.append({"mean_y": y, "words": [word]})

records = []

for group in sorted(row_groups, key=lambda g: g["mean_y"]):
    row_words = pd.DataFrame(group["words"]).sort_values("x0")

    # The first column contains the ward code (e.g., 0001).
    code_tokens = row_words[
        (row_words["x0"] >= 20) & (row_words["x0"] < 55)
    ]["text"].astype(str).tolist()
    code = "".join(code_tokens)

    if not re.fullmatch(r"\d{4}", code):
        continue

    # Ward rows have a code and "Ward No." label.
    name_text = " ".join(
        row_words[
            (row_words["x0"] >= 55) & (row_words["x0"] < 150)
        ]["text"].astype(str).tolist()
    )

    if "Ward" not in name_text:
        continue

    record = {
        "ward_code": code,
        "ward_name": name_text,
        "pdf_page": 313,
        "row_y": round(group["mean_y"], 2),
    }
    record.update(parse_row(row_words))
    records.append(record)

wards = pd.DataFrame(records)

def to_num(series):
    return pd.to_numeric(
        series.astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

for field in columns:
    wards[field + "_numeric"] = to_num(wards[field])

print("Ward rows extracted:", len(wards))
display(wards.head(15))

# Kakori wards are listed before Lucknow on this page.
# Use the first 13 rows as a check for the known Kakori population total.
kakori = wards.head(13)
print("\nKakori ward count:", len(kakori))
print(
    "Kakori ward population sum:",
    kakori["population_total_numeric"].sum()
)

print("\nExtracted ward codes:")
print(wards["ward_code"].tolist())


Ward rows extracted: 44


,ward_code,ward_name,pdf_page,row_y,households,population_total,population_male,population_female,age_0_6_total,age_0_6_male,age_0_6_female,households_numeric,population_total_numeric,population_male_numeric,population_female_numeric,age_0_6_total_numeric,age_0_6_male_numeric,age_0_6_female_numeric
0,0001,Ward No. 1,313,169.37,302,"1,644",852,792,259,133,126,302,1644,852,792,259,133,126
1,0002,Ward No. 2,313,179.57,265,"1,508",799,709,185,92,93,265,1508,799,709,185,92,93
2,0003,Ward No. 3,313,189.77,262,"1,433",739,694,187,92,95,262,1433,739,694,187,92,95
3,0004,Ward No. 4,313,199.97,301,"1,882",991,891,300,156,144,301,1882,991,891,300,156,144
4,0005,Ward No. 5,313,210.17,231,"1,268",640,628,147,64,83,231,1268,640,628,147,64,83
5,0006,Ward No. 6,313,220.37,264,"1,451",783,668,180,107,73,264,1451,783,668,180,107,73
6,0007,Ward No. 7,313,230.57,230,"1,342",692,650,199,97,102,230,1342,692,650,199,97,102
7,0008,Ward No. 8,313,240.77,204,"1,084",593,491,98,53,45,204,1084,593,491,98,53,45
8,0009,Ward No. 9,313,250.97,276,"1,475",774,701,186,86,100,276,1475,774,701,186,86,100
9,0010,Ward No. 10,313,261.17,266,"1,517",801,716,230,127,103,266,1517,801,716,230,127,103



Kakori ward count: 13
Kakori ward population sum: 19403

Extracted ward codes:
['0001', '0002', '0003', '0004', '0005', '0006', '0007', '0008', '0009', '0010', '0011', '0012', '0013', '0001', '0002', '0003', '0004', '0005', '0006', '0007', '0008', '0009', '0010', '0011', '0012', '0013', '0014', '0015', '0016', '0017', '0018', '0019', '0020', '0021', '0022', '0023', '0024', '0025', '0026', '0027', '0028', '0029', '0030', '0031']


In [ ]:

import os
import re
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"

words = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "lucknow_urban_pca_words_313_318.csv"
    )
)

# Page 313 has the population and age 0–6 columns.
page = words[words["pdf_page"] == 313].copy()

columns = {
    "households": (200, 250),
    "population_total": (250, 295),
    "population_male": (295, 340),
    "population_female": (340, 390),
    "age_0_6_total": (390, 435),
    "age_0_6_male": (435, 480),
    "age_0_6_female": (480, 530),
}

def group_rows(df, tolerance=2.0):
    groups = []
    for _, word in df.sort_values(["top", "x0"]).iterrows():
        y = float(word["top"])
        matches = [
            i for i, g in enumerate(groups)
            if abs(g["mean_y"] - y) <= tolerance
        ]

        if matches:
            i = min(matches, key=lambda j: abs(groups[j]["mean_y"] - y))
            groups[i]["words"].append(word)
            groups[i]["mean_y"] = sum(
                float(w["top"]) for w in groups[i]["words"]
            ) / len(groups[i]["words"])
        else:
            groups.append({"mean_y": y, "words": [word]})
    return sorted(groups, key=lambda g: g["mean_y"])

def parse_numeric_fields(row_words):
    result = {}
    for field, (left, right) in columns.items():
        tokens = row_words[
            (row_words["x0"] >= left) &
            (row_words["x0"] < right)
        ].sort_values("x0")["text"].astype(str).tolist()
        result[field] = "".join(tokens)
    return result

records = []

body = page[(page["top"] >= 150) & (page["top"] <= 620)]
for group in group_rows(body):
    row = pd.DataFrame(group["words"]).sort_values("x0")

    code = "".join(
        row[(row["x0"] >= 20) & (row["x0"] < 55)]
        ["text"].astype(str).tolist()
    )

    if not re.fullmatch(r"\d{4}", code):
        continue

    name = " ".join(
        row[(row["x0"] >= 55) & (row["x0"] < 150)]
        ["text"].astype(str).tolist()
    )

    if "Ward" not in name:
        continue

    town_code = "800950" if group["mean_y"] < 300 else "800951"

    record = {
        "town_code": town_code,
        "ward_code": code,
        "ward_name": name,
        "row_y": round(group["mean_y"], 2),
    }
    record.update(parse_numeric_fields(row))
    records.append(record)

wards = pd.DataFrame(records)

# Convert comma-separated values to numeric values.
fields = list(columns)
for field in fields:
    wards[field] = pd.to_numeric(
        wards[field].str.replace(",", "", regex=False),
        errors="coerce"
    )

wards["geography_id"] = (
    wards["town_code"] + "_" + wards["ward_code"]
)

wards["population_sex_sum"] = (
    wards["population_male"] + wards["population_female"]
)
wards["age_0_6_sex_sum"] = (
    wards["age_0_6_male"] + wards["age_0_6_female"]
)

wards["population_check"] = (
    wards["population_sex_sum"] == wards["population_total"]
)
wards["age_0_6_check"] = (
    wards["age_0_6_sex_sum"] == wards["age_0_6_total"]
)

print("Total ward rows:", len(wards))
print("\nWard counts by town code:")
print(wards.groupby("town_code")["ward_code"].nunique())

print("\nPopulation validation:")
print(wards.groupby("town_code").agg(
    ward_count=("ward_code", "nunique"),
    population_sum=("population_total", "sum"),
    population_checks_pass=("population_check", "all"),
    age_0_6_checks_pass=("age_0_6_check", "all"),
))

print("\nRows with failed arithmetic checks:")
display(wards[
    (~wards["population_check"]) |
    (~wards["age_0_6_check"])
])

out_path = os.path.join(
    OUTPUT_DIR,
    "lucknow_urban_pca_page313_ward_population_validation.csv"
)
wards.to_csv(out_path, index=False)
print("\nSaved:", out_path)


Total ward rows: 44

Ward counts by town code:
town_code
800950    13
800951    31
Name: ward_code, dtype: int64

Population validation:
           ward_count  population_sum  population_checks_pass  \
town_code                                                       
800950             13           19403                    True   
800951             31          989490                    True   

           age_0_6_checks_pass  
town_code                       
800950                    True  
800951                    True  

Rows with failed arithmetic checks:


,town_code,ward_code,ward_name,row_y,households,population_total,population_male,population_female,age_0_6_total,age_0_6_male,age_0_6_female,geography_id,population_sex_sum,age_0_6_sex_sum,population_check,age_0_6_check



Saved: /content/gdrive/MyDrive/lucknow_census_processed/lucknow_urban_pca_page313_ward_population_validation.csv


In [ ]:

import os
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"

words = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "lucknow_urban_pca_words_313_318.csv"
    )
)

# Inspect the left side of each page, where town and ward identifiers appear.
left = words[
    (words["x0"] < 150) &
    (words["top"] >= 145) &
    (words["top"] <= 620)
].copy()

# Group tokens into approximate horizontal lines.
left["y_group"] = (left["top"] / 2.0).round().astype(int)

for (page, y_group), group in left.groupby(
    ["pdf_page", "y_group"], sort=True
):
    group = group.sort_values("x0")
    text = " ".join(group["text"].astype(str))
    # Show town labels and likely ward rows.
    if (
        "Kakori" in text or "Lucknow" in text
        or "Ward" in text
        or any(
            token.isdigit() and len(token) == 4
            for token in group["text"].astype(str)
        )
    ):
        print(
            f"page={page}, y≈{group['top'].mean():.1f}: {text}"
        )


page=313, y≈159.1: 800950 Kakori (NP)
page=313, y≈169.2: 0001 Ward No. 1
page=313, y≈179.4: 0002 Ward No. 2
page=313, y≈189.6: 0003 Ward No. 3
page=313, y≈199.8: 0004 Ward No. 4
page=313, y≈210.0: 0005 Ward No. 5
page=313, y≈220.2: 0006 Ward No. 6
page=313, y≈230.4: 0007 Ward No. 7
page=313, y≈240.6: 0008 Ward No. 8
page=313, y≈250.8: Ward No. 9
page=313, y≈251.0: 0009
page=313, y≈261.0: Ward No. 10
page=313, y≈261.2: 0010
page=313, y≈271.2: 0011 Ward No. 11
page=313, y≈281.4: 0012 Ward No. 12
page=313, y≈291.6: 0013 Ward No. 13
page=313, y≈301.9: 800951 Lucknow (M Corp.)
page=313, y≈312.0: 0001 Ward No. 1
page=313, y≈322.2: 0002 Ward No. 2
page=313, y≈332.4: 0003 Ward No. 3
page=313, y≈342.6: 0004 Ward No. 4
page=313, y≈352.8: Ward No. 5
page=313, y≈353.0: 0005
page=313, y≈363.0: Ward No. 6
page=313, y≈363.2: 0006
page=313, y≈373.2: 0007 Ward No. 7
page=313, y≈383.4: 0008 Ward No. 8
page=313, y≈393.6: 0009 Ward No. 9
page=313, y≈403.8: 0010 Ward No. 10
page=313, y≈414.0: 0011 Ward No.

In [ ]:

import os
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"

words = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "lucknow_urban_pca_words_313_318.csv"
    )
)

page = words[words["pdf_page"] == 313].copy()

print("Page 313 vertical range:")
print("Minimum top:", page["top"].min())
print("Maximum bottom:", page["bottom"].max())

print("\nLast 50 words on page 313:")
display(
    page.sort_values(["top", "x0"])[
        ["top", "x0", "text"]
    ].tail(50)
)

print("\nWords below y=600:")
display(
    page[page["top"] >= 600]
    .sort_values(["top", "x0"])[["top", "x0", "text"]]
)


Page 313 vertical range:
Minimum top: 67.64847899999995
Maximum bottom: 791.848379

Last 50 words on page 313:


,top,x0,text
1082,740.608479,228.846065,",136"
1083,740.608479,265.080000,2
1084,740.608479,269.526065,"4,961"
1085,740.608479,309.480000,1
1086,740.608479,313.926065,"2,999"
1087,740.608479,356.160000,1
1088,740.608479,360.606065,"1,962"
1089,740.608479,410.160000,2
1090,740.608479,414.606065,",468"
1091,740.608479,451.560000,1



Words below y=600:


,top,x0,text
844,607.768479,63.960000,Ward
845,607.768479,85.691850,No.
846,607.768479,100.453106,30
843,608.008479,24.720000,0030
847,608.008479,219.960000,1
...,...,...,...
1127,761.008479,451.560000,1
1128,761.008479,456.006065,",508"
1129,761.008479,492.960000,1
1130,761.008479,497.406065,",379"


In [ ]:

import os
import re
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"

words = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "lucknow_urban_pca_words_313_318.csv"
    )
)

page = words[words["pdf_page"] == 313].copy()

columns = {
    "households": (200, 250),
    "population_total": (250, 295),
    "population_male": (295, 340),
    "population_female": (340, 390),
    "age_0_6_total": (390, 435),
    "age_0_6_male": (435, 480),
    "age_0_6_female": (480, 530),
}

def group_rows(df, tolerance=2.0):
    groups = []
    for _, word in df.sort_values(["top", "x0"]).iterrows():
        y = float(word["top"])
        matches = [
            i for i, g in enumerate(groups)
            if abs(g["mean_y"] - y) <= tolerance
        ]

        if matches:
            i = min(matches, key=lambda j: abs(groups[j]["mean_y"] - y))
            groups[i]["words"].append(word)
            groups[i]["mean_y"] = sum(
                float(w["top"]) for w in groups[i]["words"]
            ) / len(groups[i]["words"])
        else:
            groups.append({"mean_y": y, "words": [word]})
    return sorted(groups, key=lambda g: g["mean_y"])

records = []

# Include the full table body, including the bottom rows.
body = page[(page["top"] >= 150) & (page["top"] <= 770)]

for group in group_rows(body):
    row = pd.DataFrame(group["words"]).sort_values("x0")

    code = "".join(
        row[(row["x0"] >= 20) & (row["x0"] < 55)]
        ["text"].astype(str).tolist()
    )

    if not re.fullmatch(r"\d{4}", code):
        continue

    name = " ".join(
        row[(row["x0"] >= 55) & (row["x0"] < 150)]
        ["text"].astype(str).tolist()
    )

    if "Ward" not in name:
        continue

    town_code = "800950" if group["mean_y"] < 300 else "800951"

    record = {
        "town_code": town_code,
        "ward_code": code,
        "ward_name": name,
        "row_y": round(group["mean_y"], 2),
    }

    for field, (left, right) in columns.items():
        tokens = row[
            (row["x0"] >= left) &
            (row["x0"] < right)
        ].sort_values("x0")["text"].astype(str).tolist()

        record[field] = "".join(tokens)

    records.append(record)

wards = pd.DataFrame(records)

# Convert numeric fields after reconstructing split tokens.
numeric_fields = list(columns)
for field in numeric_fields:
    wards[field] = pd.to_numeric(
        wards[field].str.replace(",", "", regex=False),
        errors="coerce"
    )

wards["geography_id"] = wards["town_code"] + "_" + wards["ward_code"]

print("Ward rows extracted:", len(wards))
print("\nWard count by town:")
print(wards.groupby("town_code")["ward_code"].nunique())

print("\nPopulation totals by town:")
print(wards.groupby("town_code")["population_total"].sum())

print("\nSex-total validation:")
wards["sex_sum"] = wards["population_male"] + wards["population_female"]
wards["sex_check"] = wards["sex_sum"] == wards["population_total"]
print(wards.groupby("town_code")["sex_check"].all())

print("\nAge 0–6 validation:")
wards["age_sum"] = wards["age_0_6_male"] + wards["age_0_6_female"]
wards["age_check"] = wards["age_sum"] == wards["age_0_6_total"]
print(wards.groupby("town_code")["age_check"].all())

out_path = os.path.join(
    OUTPUT_DIR,
    "lucknow_urban_pca_page313_all_wards.csv"
)
wards.to_csv(out_path, index=False)
print("\nSaved:", out_path)


Ward rows extracted: 58

Ward count by town:
town_code
800950    13
800951    45
Name: ward_code, dtype: int64

Population totals by town:
town_code
800950      19403
800951    1356403
Name: population_total, dtype: int64

Sex-total validation:
town_code
800950    True
800951    True
Name: sex_check, dtype: bool

Age 0–6 validation:
town_code
800950    True
800951    True
Name: age_check, dtype: bool

Saved: /content/gdrive/MyDrive/lucknow_census_processed/lucknow_urban_pca_page313_all_wards.csv


In [ ]:

!pip -q install pdfplumber

import os
import pandas as pd
import pdfplumber

RAW_DIR = "/content/gdrive/MyDrive/Lucknow_Census_Raw"
OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"

PDF_PATH = os.path.join(
    RAW_DIR,
    "DH_2011_0926_PART_B_DCHB_LUCKNOW.pdf"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

records = []

with pdfplumber.open(PDF_PATH) as pdf:
    print("Total PDF pages:", len(pdf.pages))

    # Extract printed PDF pages 319–324 (1-based page numbers).
    for page_num in range(319, 325):
        page = pdf.pages[page_num - 1]
        words = page.extract_words(
            x_tolerance=1,
            y_tolerance=3,
            keep_blank_chars=False
        )

        for word in words:
            records.append({
                "pdf_page": page_num,
                "x0": word["x0"],
                "x1": word["x1"],
                "top": word["top"],
                "bottom": word["bottom"],
                "text": word["text"]
            })

words_next = pd.DataFrame(records)

out_path = os.path.join(
    OUTPUT_DIR,
    "lucknow_urban_pca_words_319_324.csv"
)
words_next.to_csv(out_path, index=False)

print("Extracted word records:", len(words_next))
print("Saved:", out_path)
print("\nWord count per page:")
print(words_next.groupby("pdf_page").size())


Total PDF pages: 350
Extracted word records: 7940
Saved: /content/gdrive/MyDrive/lucknow_census_processed/lucknow_urban_pca_words_319_324.csv

Word count per page:
pdf_page
319    1131
320    1207
321    1360
322    1611
323    1276
324    1355
dtype: int64


In [ ]:

import os
import pandas as pd

INPUT_PATH = (
    "/content/gdrive/MyDrive/lucknow_census_processed/"
    "lucknow_urban_pca_words_319_324.csv"
)

words = pd.read_csv(INPUT_PATH)

left = words[
    (words["x0"] < 150) &
    (words["top"] >= 145) &
    (words["top"] <= 780)
].copy()

# Group words into approximate horizontal lines.
left["y_group"] = (left["top"] / 2.0).round().astype(int)

for (page, y_group), group in left.groupby(
    ["pdf_page", "y_group"], sort=True
):
    group = group.sort_values("x0")
    text = " ".join(group["text"].astype(str))

    if (
        "Kakori" in text
        or "Lucknow" in text
        or "Ward" in text
        or any(
            token.isdigit() and len(token) in (4, 6)
            for token in group["text"].astype(str)
        )
    ):
        print(
            f"page={page}, y≈{group['top'].mean():.1f}: {text}"
        )


page=319, y≈159.0: Ward No. 46
page=319, y≈159.2: 0046
page=319, y≈169.2: 0047 Ward No. 47
page=319, y≈179.4: 0048 Ward No. 48
page=319, y≈189.6: 0049 Ward No. 49
page=319, y≈199.8: 0050 Ward No. 50
page=319, y≈210.0: 0051 Ward No. 51
page=319, y≈220.2: 0052 Ward No. 52
page=319, y≈230.4: 0053 Ward No. 53
page=319, y≈240.6: 0054 Ward No. 54
page=319, y≈250.8: Ward No. 55
page=319, y≈251.0: 0055
page=319, y≈261.0: Ward No. 56
page=319, y≈261.2: 0056
page=319, y≈271.2: 0057 Ward No. 57
page=319, y≈281.4: 0058 Ward No. 58
page=319, y≈291.6: 0059 Ward No. 59
page=319, y≈301.8: 0060 Ward No. 60
page=319, y≈312.0: 0061 Ward No. 61
page=319, y≈322.2: 0062 Ward No. 62
page=319, y≈332.4: 0063 Ward No. 63
page=319, y≈342.6: 0064 Ward No. 64
page=319, y≈352.8: Ward No. 65
page=319, y≈353.0: 0065
page=319, y≈363.0: Ward No. 66
page=319, y≈363.2: 0066
page=319, y≈373.2: 0067 Ward No. 67
page=319, y≈383.4: 0068 Ward No. 68
page=319, y≈393.6: 0069 Ward No. 69
page=319, y≈403.8: 0070 Ward No. 70
page=

In [ ]:

import os
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"

words = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "lucknow_urban_pca_words_319_324.csv"
    )
)

page = words[words["pdf_page"] == 319].copy()

print("Page 319 maximum bottom:", page["bottom"].max())

print("\nLast 100 words on page 319:")
display(
    page.sort_values(["top", "x0"])[
        ["top", "x0", "text"]
    ].tail(100)
)

print("\nLabels and codes near the bottom:")
bottom = page[page["top"] >= 740].sort_values(["top", "x0"])
display(bottom[["top", "x0", "text"]])


Page 319 maximum bottom: 791.848379

Last 100 words on page 319:


,top,x0,text
1031,710.008479,318.366065,",852"
1032,710.008479,360.600000,8
1033,710.008479,365.046065,",501"
1034,710.008479,410.160000,1
1035,710.008479,414.606065,",671"
...,...,...,...
1126,761.008479,451.560000,2
1127,761.008479,456.006065,",817"
1128,761.008479,492.960000,2
1129,761.008479,497.406065,",543"



Labels and codes near the bottom:


,top,x0,text
1077,740.368479,63.960000,Ward
1078,740.368479,85.691850,No.
1079,740.368479,100.453106,103
1076,740.608479,24.720000,0103
1080,740.608479,224.400000,3
1081,740.608479,228.846065,",341"
1082,740.608479,265.080000,1
1083,740.608479,269.526065,"7,010"
1084,740.608479,313.920000,8
1085,740.608479,318.366065,",340"


In [ ]:
import pandas as pd

# Official Census 2011 PCA values for Lucknow Municipal Corporation
# Source: District Census Handbook, Lucknow, Part B

official = pd.DataFrame({
    "ward_no": [106, 107, 108, 109, 110],
    "official_population": [
        19082, 23720, 12195, 23179, 18404
    ]
})

print("Official population for wards 106–110:")
display(official)

print("Total population of these five wards:",
      official["official_population"].sum())

Official population for wards 106–110:


,ward_no,official_population
0,106,19082
1,107,23720
2,108,12195
3,109,23179
4,110,18404


Total population of these five wards: 96580


In [ ]:
import os
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Census 2011 ward populations for Lucknow Municipal Corporation
# Town code: 800951
populations = [
    43282, 41971, 17988, 22717, 21960, 38910, 18999, 25553, 50574, 38053,
    54513, 55227, 19668, 19946, 46057, 55566, 16072, 23292, 23588, 19054,
    68993, 27010, 19263, 35400, 35312, 19965, 18326, 25218, 22318, 53630,
    11065, 52328, 17224, 15832, 31756, 26795, 21207, 20544, 14829, 31412,
    34903, 16816, 24961, 29270, 29036, 24264, 32543, 21552, 21023, 18196,
    22192, 21938, 29006, 17086, 17087, 16353, 17016, 23299, 22074, 18365,
    17218, 24040, 30739, 17919, 15583, 32880, 20828, 14747, 24770, 34253,
    13367, 17657, 39612, 23256, 16314, 20055, 20361, 36173, 19524, 20501,
    26055, 37800, 16256, 26103, 26973, 24452, 22154, 26217, 22266, 24495,
    34352, 27897, 13780, 17606, 13468, 19274, 22774, 14342, 37468, 17353,
    19155, 14384, 17010, 18788, 41909, 19082, 23720, 12195, 23179, 18404
]

reference = pd.DataFrame({
    "town_code": ["800951"] * 110,
    "ward_no": list(range(1, 111)),
    "geography_id": [f"800951_{i:04d}" for i in range(1, 111)],
    "official_population_2011": populations
})

# Validation
assert len(reference) == 110
assert reference["ward_no"].nunique() == 110
assert reference["ward_no"].min() == 1
assert reference["ward_no"].max() == 110

official_total = reference["official_population_2011"].sum()
expected_total = 2_817_105

print("Number of wards:", len(reference))
print("Unique ward numbers:", reference["ward_no"].nunique())
print("Sum of ward populations:", f"{official_total:,}")
print("Official town population:", f"{expected_total:,}")
print("Population reconciliation:", official_total == expected_total)

assert official_total == expected_total

output_path = os.path.join(
    OUTPUT_DIR,
    "lucknow_urban_ward_population_reference_2011.csv"
)

reference.to_csv(output_path, index=False)

print("\nSaved reference file:")
print(output_path)

display(reference.tail(10))

Number of wards: 110
Unique ward numbers: 110
Sum of ward populations: 2,817,105
Official town population: 2,817,105
Population reconciliation: True

Saved reference file:
/content/gdrive/MyDrive/lucknow_census_processed/lucknow_urban_ward_population_reference_2011.csv


,town_code,ward_no,geography_id,official_population_2011
100,800951,101,800951_0101,19155
101,800951,102,800951_0102,14384
102,800951,103,800951_0103,17010
103,800951,104,800951_0104,18788
104,800951,105,800951_0105,41909
105,800951,106,800951_0106,19082
106,800951,107,800951_0107,23720
107,800951,108,800951_0108,12195
108,800951,109,800951_0109,23179
109,800951,110,800951_0110,18404


In [ ]:
import os
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"

extracted_path = os.path.join(
    OUTPUT_DIR,
    "lucknow_urban_pca_page313_all_wards.csv"
)

reference_path = os.path.join(
    OUTPUT_DIR,
    "lucknow_urban_ward_population_reference_2011.csv"
)

if not os.path.exists(extracted_path):
    raise FileNotFoundError(
        f"Could not find {extracted_path}. "
        "Check the filename in your output directory."
    )

if not os.path.exists(reference_path):
    raise FileNotFoundError(
        f"Could not find {reference_path}. "
        "Run the previous reference-file cell first."
    )

extracted = pd.read_csv(extracted_path)
reference = pd.read_csv(reference_path)

print("Extracted data shape:", extracted.shape)
print("\nExtracted columns:")
print(extracted.columns.tolist())

print("\nFirst 10 extracted rows:")
display(extracted.head(10))

print("\nLast 10 extracted rows:")
display(extracted.tail(10))

Extracted data shape: (58, 16)

Extracted columns:
['town_code', 'ward_code', 'ward_name', 'row_y', 'households', 'population_total', 'population_male', 'population_female', 'age_0_6_total', 'age_0_6_male', 'age_0_6_female', 'geography_id', 'sex_sum', 'sex_check', 'age_sum', 'age_check']

First 10 extracted rows:


,town_code,ward_code,ward_name,row_y,households,population_total,population_male,population_female,age_0_6_total,age_0_6_male,age_0_6_female,geography_id,sex_sum,sex_check,age_sum,age_check
0,800950,1,Ward No. 1,169.37,302,1644,852,792,259,133,126,800950_0001,1644,True,259,True
1,800950,2,Ward No. 2,179.57,265,1508,799,709,185,92,93,800950_0002,1508,True,185,True
2,800950,3,Ward No. 3,189.77,262,1433,739,694,187,92,95,800950_0003,1433,True,187,True
3,800950,4,Ward No. 4,199.97,301,1882,991,891,300,156,144,800950_0004,1882,True,300,True
4,800950,5,Ward No. 5,210.17,231,1268,640,628,147,64,83,800950_0005,1268,True,147,True
5,800950,6,Ward No. 6,220.37,264,1451,783,668,180,107,73,800950_0006,1451,True,180,True
6,800950,7,Ward No. 7,230.57,230,1342,692,650,199,97,102,800950_0007,1342,True,199,True
7,800950,8,Ward No. 8,240.77,204,1084,593,491,98,53,45,800950_0008,1084,True,98,True
8,800950,9,Ward No. 9,250.97,276,1475,774,701,186,86,100,800950_0009,1475,True,186,True
9,800950,10,Ward No. 10,261.17,266,1517,801,716,230,127,103,800950_0010,1517,True,230,True



Last 10 extracted rows:


,town_code,ward_code,ward_name,row_y,households,population_total,population_male,population_female,age_0_6_total,age_0_6_male,age_0_6_female,geography_id,sex_sum,sex_check,age_sum,age_check
48,800951,36,Ward No. 36,669.17,5525,26795,13829,12966,2384,1226,1158,800951_0036,26795,True,2384,True
49,800951,37,Ward No. 37,679.37,4508,21207,10665,10542,1731,919,812,800951_0037,21207,True,1731,True
50,800951,38,Ward No. 38,689.57,4463,20544,10345,10199,1739,860,879,800951_0038,20544,True,1739,True
51,800951,39,Ward No. 39,699.77,2740,14829,7643,7186,1278,649,629,800951_0039,14829,True,1278,True
52,800951,40,Ward No. 40,709.97,6609,31412,16431,14981,3118,1665,1453,800951_0040,31412,True,3118,True
53,800951,41,Ward No. 41,720.17,6772,34903,17885,17018,3598,1862,1736,800951_0041,34903,True,3598,True
54,800951,42,Ward No. 42,730.37,3514,16816,8838,7978,1494,776,718,800951_0042,16816,True,1494,True
55,800951,43,Ward No. 43,740.57,5136,24961,12999,11962,2468,1317,1151,800951_0043,24961,True,2468,True
56,800951,44,Ward No. 44,750.77,5622,29270,15189,14081,2661,1420,1241,800951_0044,29270,True,2661,True
57,800951,45,Ward No. 45,760.97,6026,29036,15068,13968,2887,1508,1379,800951_0045,29036,True,2887,True


In [ ]:
import os
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"

extracted_path = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_page313_all_wards.csv"
)
reference_path = os.path.join(
    OUTPUT_DIR, "lucknow_urban_ward_population_reference_2011.csv"
)

extracted = pd.read_csv(extracted_path)
reference = pd.read_csv(reference_path)

# Normalize identifiers before comparing
for df in [extracted, reference]:
    df["town_code"] = df["town_code"].astype(str).str.replace(r"\.0$", "", regex=True)
    df["ward_no"] = pd.to_numeric(
        df["ward_code"] if "ward_code" in df.columns else df["ward_no"],
        errors="coerce"
    ).astype("Int64")

# Keep Lucknow Municipal Corporation only
lucknow = extracted[extracted["town_code"] == "800951"].copy()
lucknow_ref = reference[reference["town_code"] == "800951"].copy()

print("Extracted Lucknow ward rows:", len(lucknow))
print("Unique extracted wards:", lucknow["ward_no"].nunique())
print("Reference ward rows:", len(lucknow_ref))
print("Unique reference wards:", lucknow_ref["ward_no"].nunique())

# Duplicate checks
print("\nDuplicate extracted wards:")
display(
    lucknow[lucknow.duplicated("ward_no", keep=False)]
    .sort_values("ward_no")
)

# Missing ward numbers in the extracted data
expected_wards = set(range(1, 111))
actual_wards = set(lucknow["ward_no"].dropna().astype(int))
missing = sorted(expected_wards - actual_wards)

print("\nWard numbers missing from the extracted CSV:")
print(missing)

# Compare populations only where both files contain the ward
comparison = lucknow.merge(
    lucknow_ref[["ward_no", "official_population_2011"]],
    on="ward_no",
    how="outer",
    indicator=True
)

comparison["population_difference"] = (
    comparison["population_total"]
    - comparison["official_population_2011"]
)

print("\nComparison status:")
display(comparison["_merge"].value_counts().rename_axis("status").to_frame("rows"))

print("\nPopulation mismatches (only overlapping wards):")
mismatches = comparison[
    (comparison["_merge"] == "both")
    & (comparison["population_difference"] != 0)
]
display(
    mismatches[
        ["ward_no", "population_total",
         "official_population_2011", "population_difference"]
    ].sort_values("ward_no")
)

# Totals
print("\nExtracted population total for Lucknow wards:",
      f"{lucknow['population_total'].sum():,}")
print("Reference population total:",
      f"{lucknow_ref['official_population_2011'].sum():,}")
print("Official town-level total for comparison: 2,817,105")

# Save comparison for inspection
comparison_path = os.path.join(
    OUTPUT_DIR, "lucknow_ward_population_comparison.csv"
)
comparison.to_csv(comparison_path, index=False)
print("\nSaved comparison to:", comparison_path)

Extracted Lucknow ward rows: 45
Unique extracted wards: 45
Reference ward rows: 110
Unique reference wards: 110

Duplicate extracted wards:


,town_code,ward_code,ward_name,row_y,households,population_total,population_male,population_female,age_0_6_total,age_0_6_male,age_0_6_female,geography_id,sex_sum,sex_check,age_sum,age_check,ward_no



Ward numbers missing from the extracted CSV:
[46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110]

Comparison status:


,rows
status,
right_only,65
both,45
left_only,0



Population mismatches (only overlapping wards):


,ward_no,population_total,official_population_2011,population_difference



Extracted population total for Lucknow wards: 1,356,403
Reference population total: 2,817,105
Official town-level total for comparison: 2,817,105

Saved comparison to: /content/gdrive/MyDrive/lucknow_census_processed/lucknow_ward_population_comparison.csv


In [ ]:
import os
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"

words_path = os.path.join(
    OUTPUT_DIR,
    "lucknow_urban_pca_words_319_324.csv"
)

words = pd.read_csv(words_path)

# Page 319 contains the population table for Lucknow wards 46 onward
page = words[words["pdf_page"] == 319].copy()

# Show the text in reading order, with coordinates
page = page.sort_values(["top", "x0"])

# Inspect the table's row labels and numeric columns
for y in sorted(page.loc[page["text"].astype(str).str.fullmatch(r"\d{1,3}")]["top"].unique()):
    row = page[(page["top"] - y).abs() < 1.0].sort_values("x0")
    text = " ".join(row["text"].astype(str))
    if "Ward" in text or any(
        f" {n} " in f" {text} " for n in range(46, 111)
    ):
        print(f"y={y:.2f}: {text}")

y=158.97: 0046 Ward No. 46 4 ,474 2 4,264 1 2,363 1 1,901 2 ,432 1 ,254 1 ,178
y=159.21: 0046 Ward No. 46 4 ,474 2 4,264 1 2,363 1 1,901 2 ,432 1 ,254 1 ,178
y=169.17: 0047 Ward No. 47 5 ,742 3 2,543 1 6,933 1 5,610 4 ,218 2 ,176 2 ,042
y=169.41: 0047 Ward No. 47 5 ,742 3 2,543 1 6,933 1 5,610 4 ,218 2 ,176 2 ,042
y=179.37: 0048 Ward No. 48 4 ,054 2 1,552 1 1,044 1 0,508 1 ,967 9 91 9 76
y=179.61: 0048 Ward No. 48 4 ,054 2 1,552 1 1,044 1 0,508 1 ,967 9 91 9 76
y=189.57: 0049 Ward No. 49 4 ,526 2 1,023 1 1,026 9 ,997 1 ,667 8 92 7 75
y=189.81: 0049 Ward No. 49 4 ,526 2 1,023 1 1,026 9 ,997 1 ,667 8 92 7 75
y=199.77: 0050 Ward No. 50 3 ,604 1 8,196 9 ,352 8 ,844 1 ,597 8 52 7 45
y=200.01: 0050 Ward No. 50 3 ,604 1 8,196 9 ,352 8 ,844 1 ,597 8 52 7 45
y=209.97: 0051 Ward No. 51 3 ,714 2 2,192 1 1,351 1 0,841 2 ,670 1 ,368 1 ,302
y=210.21: 0051 Ward No. 51 3 ,714 2 2,192 1 1,351 1 0,841 2 ,670 1 ,368 1 ,302
y=220.17: 0052 Ward No. 52 4 ,649 2 1,938 1 1,294 1 0,644 2 ,237 1 ,185 1 ,052
y=2

In [ ]:
import os
import re
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"

words_path = os.path.join(
    OUTPUT_DIR,
    "lucknow_urban_pca_words_319_324.csv"
)

words = pd.read_csv(words_path)
page = words[words["pdf_page"] == 319].copy()
page["text"] = page["text"].astype(str)

# Keep one copy of duplicated PDF text-layer tokens.
page["x_round"] = page["x0"].round(1)
page = page.drop_duplicates(
    subset=["top", "x_round", "text"]
)

# Find the row position for each ward code at the far left of the table.
ward_codes = page[
    (page["x0"] < 45) &
    page["text"].str.fullmatch(r"01\d{2}")
].copy()

ward_codes["ward_no"] = ward_codes["text"].astype(int)

# Collapse repeated text-layer occurrences for each ward.
ward_codes = (
    ward_codes.sort_values("top")
    .drop_duplicates("ward_no", keep="first")
)

# x-coordinate bands based on the page 319 table layout.
bands = {
    "households": (210, 250),
    "population_total": (250, 305),
    "population_male": (305, 355),
    "population_female": (355, 405),
    "age_0_6_total": (405, 450),
    "age_0_6_male": (450, 490),
    "age_0_6_female": (490, 530),
}

def reconstruct_number(row_words, xmin, xmax):
    tokens = row_words[
        (row_words["x0"] >= xmin) &
        (row_words["x0"] < xmax)
    ].sort_values("x0")

    # Remove duplicate tokens at effectively the same x position.
    tokens = tokens.drop_duplicates(
        subset=["x_round", "text"]
    )

    combined = "".join(tokens["text"].tolist())
    digits = re.sub(r"\D", "", combined)

    return int(digits) if digits else None

records = []

for _, ward in ward_codes.iterrows():
    ward_no = int(ward["ward_no"])
    y = float(ward["top"])

    # Include tokens on this row, including the nearby repeated text layer.
    row_words = page[
        (page["top"] - y).abs() <= 0.6
    ].copy()

    # Deduplicate tokens repeated by the PDF text layer.
    row_words["x_round"] = row_words["x0"].round(1)
    row_words = row_words.drop_duplicates(
        subset=["x_round", "text"]
    )

    record = {
        "town_code": "800951",
        "ward_code": ward_no,
        "ward_name": f"Ward No. {ward_no}",
        "row_y": y,
        "geography_id": f"800951_{ward_no:04d}",
    }

    for col, (xmin, xmax) in bands.items():
        record[col] = reconstruct_number(
            row_words, xmin, xmax
        )

    records.append(record)

new_wards = pd.DataFrame(records)

# Validation checks
new_wards["sex_sum"] = (
    new_wards["population_male"]
    + new_wards["population_female"]
)
new_wards["sex_check"] = (
    new_wards["sex_sum"] == new_wards["population_total"]
)

new_wards["age_sum"] = (
    new_wards["age_0_6_male"]
    + new_wards["age_0_6_female"]
)
new_wards["age_check"] = (
    new_wards["age_sum"] == new_wards["age_0_6_total"]
)

output_path = os.path.join(
    OUTPUT_DIR,
    "lucknow_urban_pca_page319_wards46_105.csv"
)
new_wards.to_csv(output_path, index=False)

print("Extracted ward rows:", len(new_wards))
print("Ward numbers:", new_wards["ward_code"].min(),
      "to", new_wards["ward_code"].max())
print("Duplicate ward numbers:", new_wards["ward_code"].duplicated().sum())
print("Missing values by column:")
display(new_wards.isna().sum().to_frame("missing_values"))

print("\nMale/female population check:")
print(new_wards["sex_check"].value_counts(dropna=False))

print("\nAge 0–6 check:")
print(new_wards["age_check"].value_counts(dropna=False))

print("\nFirst five extracted wards:")
display(new_wards.head())

print("\nLast five extracted wards:")
display(new_wards.tail())

print("\nSaved to:", output_path)

Extracted ward rows: 6
Ward numbers: 100 to 105
Duplicate ward numbers: 0
Missing values by column:


,missing_values
town_code,0
ward_code,0
ward_name,0
row_y,0
geography_id,0
households,0
population_total,0
population_male,0
population_female,0
age_0_6_total,0



Male/female population check:
sex_check
True    6
Name: count, dtype: int64

Age 0–6 check:
age_check
True    6
Name: count, dtype: int64

First five extracted wards:


,town_code,ward_code,ward_name,row_y,geography_id,households,population_total,population_male,population_female,age_0_6_total,age_0_6_male,age_0_6_female,sex_sum,sex_check,age_sum,age_check
0,800951,100,Ward No. 100,710.008479,800951_0100,3078,17353,8852,8501,1671,832,839,17353,True,1671,True
1,800951,101,Ward No. 101,720.208479,800951_0101,3577,19155,9648,9507,2150,1133,1017,19155,True,2150,True
2,800951,102,Ward No. 102,730.408479,800951_0102,2944,14384,7318,7066,1241,669,572,14384,True,1241,True
3,800951,103,Ward No. 103,740.608479,800951_0103,3341,17010,8340,8670,1384,757,627,17010,True,1384,True
4,800951,104,Ward No. 104,750.808479,800951_0104,3462,18788,9538,9250,1819,925,894,18788,True,1819,True



Last five extracted wards:


,town_code,ward_code,ward_name,row_y,geography_id,households,population_total,population_male,population_female,age_0_6_total,age_0_6_male,age_0_6_female,sex_sum,sex_check,age_sum,age_check
1,800951,101,Ward No. 101,720.208479,800951_0101,3577,19155,9648,9507,2150,1133,1017,19155,True,2150,True
2,800951,102,Ward No. 102,730.408479,800951_0102,2944,14384,7318,7066,1241,669,572,14384,True,1241,True
3,800951,103,Ward No. 103,740.608479,800951_0103,3341,17010,8340,8670,1384,757,627,17010,True,1384,True
4,800951,104,Ward No. 104,750.808479,800951_0104,3462,18788,9538,9250,1819,925,894,18788,True,1819,True
5,800951,105,Ward No. 105,761.008479,800951_0105,6757,41909,21550,20359,5360,2817,2543,41909,True,5360,True



Saved to: /content/gdrive/MyDrive/lucknow_census_processed/lucknow_urban_pca_page319_wards46_105.csv


In [ ]:
import os
import re
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"
words_path = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_words_319_324.csv"
)

words = pd.read_csv(words_path)
page = words[words["pdf_page"] == 319].copy()
page["text"] = page["text"].astype(str).str.strip()

# Ward codes are 0046–0105, not just 0100–0105
ward_codes = page[
    (page["x0"] < 45) &
    page["text"].str.fullmatch(r"00\d{2}|010\d|0110")
].copy()

ward_codes["ward_no"] = pd.to_numeric(
    ward_codes["text"], errors="coerce"
)

# Convert codes such as 0046 to ward number 46
ward_codes["ward_no"] = ward_codes["ward_no"].astype(int)

# Remove duplicate text-layer entries
ward_codes = (
    ward_codes.sort_values("top")
    .drop_duplicates("ward_no", keep="first")
)

bands = {
    "households": (210, 250),
    "population_total": (250, 305),
    "population_male": (305, 355),
    "population_female": (355, 405),
    "age_0_6_total": (405, 450),
    "age_0_6_male": (450, 490),
    "age_0_6_female": (490, 530),
}

def reconstruct_number(row_words, xmin, xmax):
    tokens = row_words[
        (row_words["x0"] >= xmin) &
        (row_words["x0"] < xmax)
    ].sort_values("x0")

    tokens = tokens.drop_duplicates(["x0", "text"])
    combined = "".join(tokens["text"].tolist())
    digits = re.sub(r"\D", "", combined)
    return int(digits) if digits else None

records = []

for _, ward in ward_codes.iterrows():
    ward_no = int(ward["ward_no"])
    y = float(ward["top"])

    row_words = page[
        (page["top"] - y).abs() <= 0.6
    ].copy()

    records_row = {
        "town_code": "800951",
        "ward_code": ward_no,
        "ward_name": f"Ward No. {ward_no}",
        "row_y": y,
        "geography_id": f"800951_{ward_no:04d}",
    }

    for col, (xmin, xmax) in bands.items():
        records_row[col] = reconstruct_number(
            row_words, xmin, xmax
        )

    records.append(records_row)

new_wards = pd.DataFrame(records)
new_wards = new_wards.sort_values("ward_code")

new_wards["sex_sum"] = (
    new_wards["population_male"]
    + new_wards["population_female"]
)
new_wards["sex_check"] = (
    new_wards["sex_sum"] == new_wards["population_total"]
)
new_wards["age_sum"] = (
    new_wards["age_0_6_male"]
    + new_wards["age_0_6_female"]
)
new_wards["age_check"] = (
    new_wards["age_sum"] == new_wards["age_0_6_total"]
)

output_path = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_page319_wards46_105.csv"
)
new_wards.to_csv(output_path, index=False)

print("Rows extracted:", len(new_wards))
print("Ward range:", new_wards["ward_code"].min(),
      "to", new_wards["ward_code"].max())
print("Missing ward numbers:",
      sorted(set(range(46, 106)) - set(new_wards["ward_code"])))
print("Duplicate wards:", new_wards["ward_code"].duplicated().sum())
print("Missing values:")
display(new_wards.isna().sum().to_frame("missing_values"))
print("Sex checks:")
print(new_wards["sex_check"].value_counts(dropna=False))
print("Age checks:")
print(new_wards["age_check"].value_counts(dropna=False))

display(new_wards.head())
display(new_wards.tail())

Rows extracted: 60
Ward range: 46 to 105
Missing ward numbers: []
Duplicate wards: 0
Missing values:


,missing_values
town_code,0
ward_code,0
ward_name,0
row_y,0
geography_id,0
households,0
population_total,0
population_male,0
population_female,0
age_0_6_total,0


Sex checks:
sex_check
True    60
Name: count, dtype: int64
Age checks:
age_check
True    60
Name: count, dtype: int64


,town_code,ward_code,ward_name,row_y,geography_id,households,population_total,population_male,population_female,age_0_6_total,age_0_6_male,age_0_6_female,sex_sum,sex_check,age_sum,age_check
0,800951,46,Ward No. 46,159.208479,800951_0046,4474,24264,12363,11901,2432,1254,1178,24264,True,2432,True
1,800951,47,Ward No. 47,169.408479,800951_0047,5742,32543,16933,15610,4218,2176,2042,32543,True,4218,True
2,800951,48,Ward No. 48,179.608479,800951_0048,4054,21552,11044,10508,1967,991,976,21552,True,1967,True
3,800951,49,Ward No. 49,189.808479,800951_0049,4526,21023,11026,9997,1667,892,775,21023,True,1667,True
4,800951,50,Ward No. 50,200.008479,800951_0050,3604,18196,9352,8844,1597,852,745,18196,True,1597,True


,town_code,ward_code,ward_name,row_y,geography_id,households,population_total,population_male,population_female,age_0_6_total,age_0_6_male,age_0_6_female,sex_sum,sex_check,age_sum,age_check
55,800951,101,Ward No. 101,720.208479,800951_0101,3577,19155,9648,9507,2150,1133,1017,19155,True,2150,True
56,800951,102,Ward No. 102,730.408479,800951_0102,2944,14384,7318,7066,1241,669,572,14384,True,1241,True
57,800951,103,Ward No. 103,740.608479,800951_0103,3341,17010,8340,8670,1384,757,627,17010,True,1384,True
58,800951,104,Ward No. 104,750.808479,800951_0104,3462,18788,9538,9250,1819,925,894,18788,True,1819,True
59,800951,105,Ward No. 105,761.008479,800951_0105,6757,41909,21550,20359,5360,2817,2543,41909,True,5360,True


In [ ]:
import os
import re
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/lucknow_census_processed"
words_path = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_words_319_324.csv"
)

words = pd.read_csv(words_path)
page = words[words["pdf_page"] == 319].copy()
page["text"] = page["text"].astype(str)

records = []

# Page 319 has a row every ~10.2 coordinate units.
for ward_no in range(46, 106):
    label = page[
        page["text"].str.fullmatch(str(ward_no))
        & (page["x0"].between(95, 115))
    ]

    if label.empty:
        print(f"WARNING: Ward {ward_no} label not found")
        continue

    y = float(label.iloc[0]["top"])

    # Collect all tokens on the same physical row.
    row = page[(page["top"] - y).abs() < 0.7].sort_values("x0")

    # Merge the fragments of each numeric field in its x-coordinate band.
    bands = {
        "households": (210, 250),
        "population_total": (250, 305),
        "population_male": (305, 355),
        "population_female": (355, 405),
        "age_0_6_total": (405, 450),
        "age_0_6_male": (450, 490),
        "age_0_6_female": (490, 530),
    }

    rec = {
        "town_code": "800951",
        "ward_code": ward_no,
        "ward_name": f"Ward No. {ward_no}",
        "row_y": y,
        "geography_id": f"800951_{ward_no:04d}",
    }

    for col, (xmin, xmax) in bands.items():
        tokens = row[row["x0"].between(xmin, xmax, inclusive="left")]
        digits = re.sub(r"\D", "", "".join(tokens["text"].tolist()))
        rec[col] = int(digits) if digits else None

    records.append(rec)

new_wards = pd.DataFrame(records)

new_wards["sex_sum"] = (
    new_wards["population_male"] + new_wards["population_female"]
)
new_wards["sex_check"] = (
    new_wards["sex_sum"] == new_wards["population_total"]
)
new_wards["age_sum"] = (
    new_wards["age_0_6_male"] + new_wards["age_0_6_female"]
)
new_wards["age_check"] = (
    new_wards["age_sum"] == new_wards["age_0_6_total"]
)

print("Rows extracted:", len(new_wards))
print("Unique ward numbers:", new_wards["ward_code"].nunique())
print("Missing ward numbers:",
      sorted(set(range(46, 106)) - set(new_wards["ward_code"])))

print("\nMissing values:")
display(new_wards.isna().sum().to_frame("missing_values"))

print("\nPopulation consistency checks:")
print("Sex check:", new_wards["sex_check"].value_counts(dropna=False).to_dict())
print("Age check:", new_wards["age_check"].value_counts(dropna=False).to_dict())

display(new_wards.head(5))
display(new_wards.tail(5))

output_path = os.path.join(
    OUTPUT_DIR, "lucknow_urban_pca_page319_wards46_105.csv"
)
new_wards.to_csv(output_path, index=False)
print("\nSaved:", output_path)

Rows extracted: 60
Unique ward numbers: 60
Missing ward numbers: []

Missing values:


,missing_values
town_code,0
ward_code,0
ward_name,0
row_y,0
geography_id,0
households,0
population_total,0
population_male,0
population_female,0
age_0_6_total,0



Population consistency checks:
Sex check: {True: 60}
Age check: {True: 60}


,town_code,ward_code,ward_name,row_y,geography_id,households,population_total,population_male,population_female,age_0_6_total,age_0_6_male,age_0_6_female,sex_sum,sex_check,age_sum,age_check
0,800951,46,Ward No. 46,158.968479,800951_0046,4474,24264,12363,11901,2432,1254,1178,24264,True,2432,True
1,800951,47,Ward No. 47,169.168479,800951_0047,5742,32543,16933,15610,4218,2176,2042,32543,True,4218,True
2,800951,48,Ward No. 48,179.368479,800951_0048,4054,21552,11044,10508,1967,991,976,21552,True,1967,True
3,800951,49,Ward No. 49,189.568479,800951_0049,4526,21023,11026,9997,1667,892,775,21023,True,1667,True
4,800951,50,Ward No. 50,199.768479,800951_0050,3604,18196,9352,8844,1597,852,745,18196,True,1597,True


,town_code,ward_code,ward_name,row_y,geography_id,households,population_total,population_male,population_female,age_0_6_total,age_0_6_male,age_0_6_female,sex_sum,sex_check,age_sum,age_check
55,800951,101,Ward No. 101,719.968479,800951_0101,3577,19155,9648,9507,2150,1133,1017,19155,True,2150,True
56,800951,102,Ward No. 102,730.168479,800951_0102,2944,14384,7318,7066,1241,669,572,14384,True,1241,True
57,800951,103,Ward No. 103,740.368479,800951_0103,3341,17010,8340,8670,1384,757,627,17010,True,1384,True
58,800951,104,Ward No. 104,750.568479,800951_0104,3462,18788,9538,9250,1819,925,894,18788,True,1819,True
59,800951,105,Ward No. 105,760.768479,800951_0105,6757,41909,21550,20359,5360,2817,2543,41909,True,5360,True



Saved: /content/gdrive/MyDrive/lucknow_census_processed/lucknow_urban_pca_page319_wards46_105.csv


In [ ]:
import pdfplumber

pdf_path = "/content/gdrive/MyDrive/Lucknow_Census_Raw/DH_2011_0926_PART_B_DCHB_LUCKNOW.pdf"

with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[324]  # PDF page 325; Python uses zero-based indexing
    words = page.extract_words()

    for word in words:
        if word["x0"] < 140 and word["top"] < 250:
            print(f'y={word["top"]:.1f}, x={word["x0"]:.1f}: {word["text"]}')

y=108.1, x=26.8: Location
y=119.6, x=34.3: code
y=131.1, x=29.0: number
y=131.1, x=69.2: Name
y=131.1, x=92.8: of
y=131.1, x=102.5: Town/
y=131.1, x=128.1: Ward
y=142.9, x=40.8: 1
y=142.9, x=106.7: 2
y=159.2, x=24.7: 0106
y=159.0, x=64.0: Ward
y=159.0, x=85.7: No.
y=159.0, x=100.5: 106
y=169.4, x=24.7: 0107
y=169.2, x=64.0: Ward
y=169.2, x=85.7: No.
y=169.2, x=100.5: 107
y=179.6, x=24.7: 0108
y=179.4, x=64.0: Ward
y=179.4, x=85.7: No.
y=179.4, x=100.5: 108
y=189.8, x=24.7: 0109
y=189.6, x=64.0: Ward
y=189.6, x=85.7: No.
y=189.6, x=100.5: 109
y=200.0, x=24.7: 0110
y=199.8, x=64.0: Ward
y=199.8, x=85.7: No.
y=199.8, x=100.5: 110
y=210.2, x=24.7: 800952
y=210.1, x=64.0: Lucknow
y=210.1, x=101.1: (CB)
y=220.4, x=24.7: 0001
y=220.2, x=64.0: Ward
y=220.2, x=85.7: No.
y=220.2, x=100.5: 1
y=230.6, x=24.7: 0002
y=230.4, x=64.0: Ward
y=230.4, x=85.7: No.
y=230.4, x=100.5: 2
y=240.8, x=24.7: 0003
y=240.6, x=64.0: Ward
y=240.6, x=85.7: No.
y=240.6, x=100.5: 3


In [ ]:
import pandas as pd
import pdfplumber
import re

pdf_path = "/content/gdrive/MyDrive/Lucknow_Census_Raw/DH_2011_0926_PART_B_DCHB_LUCKNOW.pdf"

with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[324]  # Page 325
    words = pd.DataFrame(page.extract_words())

# Convert relevant fields to numeric
words["x0"] = pd.to_numeric(words["x0"])
words["top"] = pd.to_numeric(words["top"])

# Extract wards 106–110 using the ward-number labels
results = []

bands = {
    "households": (210, 250),
    "population_total": (250, 305),
    "population_male": (305, 355),
    "population_female": (355, 405),
    "age_0_6_total": (405, 450),
    "age_0_6_male": (450, 490),
    "age_0_6_female": (490, 530),
}

for ward_no in range(106, 111):
    label = words[
        (words["text"] == str(ward_no)) &
        words["x0"].between(95, 115)
    ]

    if label.empty:
        print(f"Ward {ward_no}: label not found")
        continue

    y = label.iloc[0]["top"]

    row = words[(words["top"] - y).abs() < 0.7].copy()

    # Remove duplicate PDF text layers
    row["x_round"] = row["x0"].round(1)
    row = row.drop_duplicates(subset=["x_round", "text"])

    record = {
        "town_code": 800951,
        "ward_code": ward_no,
        "ward_name": f"Ward No. {ward_no}",
        "geography_id": f"800951_{ward_no:04d}",
    }

    for field, (xmin, xmax) in bands.items():
        tokens = row[
            row["x0"].between(xmin, xmax, inclusive="left")
        ].sort_values("x0")["text"]

        digits = "".join(re.sub(r"\D", "", str(t)) for t in tokens)

        record[field] = int(digits) if digits else None

    results.append(record)

extra_wards = pd.DataFrame(results)

# Validate extracted rows
extra_wards["sex_check"] = (
    extra_wards["population_male"] +
    extra_wards["population_female"] ==
    extra_wards["population_total"]
)

extra_wards["age_check"] = (
    extra_wards["age_0_6_male"] +
    extra_wards["age_0_6_female"] ==
    extra_wards["age_0_6_total"]
)

display(extra_wards)

,town_code,ward_code,ward_name,geography_id,households,population_total,population_male,population_female,age_0_6_total,age_0_6_male,age_0_6_female,sex_check,age_check
0,800951,106,Ward No. 106,800951_0106,3218,19082,9829,9253,1922,956,966,True,True
1,800951,107,Ward No. 107,800951_0107,4208,23720,12239,11481,2731,1374,1357,True,True
2,800951,108,Ward No. 108,800951_0108,2079,12195,6143,6052,1377,702,675,True,True
3,800951,109,Ward No. 109,800951_0109,4004,23179,11945,11234,2675,1408,1267,True,True
4,800951,110,Ward No. 110,800951_0110,3325,18404,9296,9108,1869,942,927,True,True


In [ ]:
import pandas as pd
import os

output_dir = "/content/gdrive/MyDrive/lucknow_census_processed"

# Load wards 1–45
wards_1_45 = pd.read_csv(
    os.path.join(output_dir, "lucknow_urban_pca_page313_all_wards.csv")
)

# Keep only Lucknow Municipal Corporation (town code 800951)
wards_1_45 = wards_1_45[
    wards_1_45["town_code"].astype(str) == "800951"
].copy()

# Load wards 46–105
wards_46_105 = pd.read_csv(
    os.path.join(output_dir, "lucknow_urban_pca_page319_wards46_105.csv")
)

# Use extra_wards created from page 325 for wards 106–110
wards_106_110 = extra_wards.copy()

# Combine
all_wards = pd.concat(
    [wards_1_45, wards_46_105, wards_106_110],
    ignore_index=True
)

# Sort and validate
all_wards["ward_code"] = pd.to_numeric(all_wards["ward_code"])
all_wards = all_wards.sort_values("ward_code").reset_index(drop=True)

print("Total rows:", len(all_wards))
print("Unique wards:", all_wards["ward_code"].nunique())
print("Ward range:", all_wards["ward_code"].min(), "to", all_wards["ward_code"].max())
print("Missing ward numbers:", sorted(set(range(1, 111)) - set(all_wards["ward_code"])))

print("\nPopulation totals:")
print("Households:", all_wards["households"].sum())
print("Population:", all_wards["population_total"].sum())
print("Male population:", all_wards["population_male"].sum())
print("Female population:", all_wards["population_female"].sum())
print("Children aged 0–6:", all_wards["age_0_6_total"].sum())

print("\nValidation:")
print("Duplicate ward codes:", all_wards["ward_code"].duplicated().sum())
print("Sex checks passed:", (
    all_wards["population_male"] + all_wards["population_female"]
    == all_wards["population_total"]
).all())
print("Age checks passed:", (
    all_wards["age_0_6_male"] + all_wards["age_0_6_female"]
    == all_wards["age_0_6_total"]
).all())

# Save the combined file
master_path = os.path.join(
    output_dir, "lucknow_municipal_corporation_ward_pca_2011.csv"
)
all_wards.to_csv(master_path, index=False)

print("\nSaved to:", master_path)

Total rows: 110
Unique wards: 110
Ward range: 1 to 110
Missing ward numbers: []

Population totals:
Households: 538149
Population: 2817105
Male population: 1460970
Female population: 1356135
Children aged 0–6: 293697

Validation:
Duplicate ward codes: 0
Sex checks passed: True
Age checks passed: True

Saved to: /content/gdrive/MyDrive/lucknow_census_processed/lucknow_municipal_corporation_ward_pca_2011.csv
